In [ ]:
import os
import time
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import requests

# ==============================================================================
# CONFIGURATION
# ==============================================================================
SYMBOLS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XAUUSDT", "XAGUSDT"]
START_DATE = "2019-01-01"
END_DATE = None  # Defaults to current time
INTERVAL = "1h"
LIMIT = 1250

# Directory Setup
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FAPI_URL = "https://fapi.binance.com/fapi/v1/klines"

# ==============================================================================
# STAGE 1: INGESTION & RAW STORAGE
# ==============================================================================
def to_ms(date_str: str) -> int:
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

def fetch_and_save_raw(symbol: str, start_ms: int, end_ms: int) -> Path:
    """Fetches raw 1-hour OHLCV candles and saves directly to symbol CSV."""
    all_rows = []
    cursor = start_ms
    session = requests.Session()

    print(f"[Fetch] Downloading raw data for {symbol}...")
    while cursor < end_ms:
        params = {
            "symbol": symbol,
            "interval": INTERVAL,
            "startTime": cursor,
            "endTime": end_ms,
            "limit": LIMIT,
        }
        try:
            resp = session.get(FAPI_URL, params=params, timeout=20)
            resp.raise_for_status()
            batch = resp.json()
            if not batch:
                break
            all_rows.extend(batch)
            
            last_open_time = batch[-1][0]
            next_cursor = last_open_time + 1
            if next_cursor <= cursor:
                break
            cursor = next_cursor
            time.sleep(0.05)
        except Exception as e:
            print(f"Error fetching {symbol}: {e}")
            break

    columns = [
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "n_trades",
        "taker_buy_base", "taker_buy_quote", "ignore",
    ]
    df = pd.DataFrame(all_rows, columns=columns) if all_rows else pd.DataFrame(columns=columns)
    
    # Cast numerical columns
    num_cols = ["open", "high", "low", "close", "volume", "quote_volume", "n_trades"]
    df[num_cols] = df[num_cols].astype(float)
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df["symbol"] = symbol

    raw_path = RAW_DIR / f"{symbol}_1h_raw.csv"
    df.to_csv(raw_path, index=False)
    print(f" Saved raw dataset: {raw_path} ({len(df)} rows)")
    return raw_path

# ==============================================================================
# STAGE 2: LEAK-FREE FEATURE ENGINEERING
# ==============================================================================
def calculate_hurst(series: np.ndarray) -> float:
    """Calculates rolling Hurst Exponent on trailing close prices."""
    x = series[~np.isnan(series)]
    n = len(x)
    if n < 20:
        return np.nan
    lags = np.unique(np.geomspace(2, n // 2, num=10).astype(int))
    lags = lags[lags >= 2]
    if len(lags) < 3:
        return np.nan
    tau = [np.std(x[lag:] - x[:-lag]) for lag in lags]
    tau = np.array(tau)
    valid = ~np.isnan(tau) & (tau > 0)
    if valid.sum() < 3:
        return np.nan
    poly = np.polyfit(np.log(lags[valid]), np.log(tau[valid]), 1)
    return poly[0] * 2.0

def engineer_features(raw_path: Path) -> Path:
    """Reads raw CSV and constructs 12 technical features without lookahead bias."""
    df = pd.read_csv(raw_path)
    if df.empty:
        return None

    df["open_time"] = pd.to_datetime(df["open_time"])
    df = df.sort_values("open_time").drop_duplicates("open_time").reset_index(drop=True)

    # 1. Log Return
    df["log_return"] = np.log(df["close"]).diff()

    # 2. Realized Volatility (20-period trailing)
    df["realized_vol_20"] = df["log_return"].rolling(20, min_periods=20).std()

    # 3. ATR & 4. ATR Percentage (14-period Wilder Smoothed)
    high_low = df["high"] - df["low"]
    high_pc = (df["high"] - df["close"].shift(1)).abs()
    low_pc = (df["low"] - df["close"].shift(1)).abs()
    tr = pd.concat([high_low, high_pc, low_pc], axis=1).max(axis=1)
    
    df["atr_14"] = tr.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    df["atr_pct"] = df["atr_14"] / df["close"]

    # 5. ADX (14-period)
    up_move = df["high"].diff()
    down_move = -df["low"].diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    
    plus_di = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1/14, adjust=False, min_periods=14).mean() / df["atr_14"]
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/14, adjust=False, min_periods=14).mean() / df["atr_14"]
    dx = (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan) * 100
    df["adx_14"] = dx.ewm(alpha=1/14, adjust=False, min_periods=14).mean()

    # 6. Hurst Exponent (100-period trailing log returns)
    df["hurst_100"] = df["log_return"].rolling(100, min_periods=100).apply(calculate_hurst, raw=True)

    # 7-9. Momentum (5h, 20h, 60h)
    df["momentum_5h"] = np.log(df["close"]) - np.log(df["close"].shift(5))
    df["momentum_20h"] = np.log(df["close"]) - np.log(df["close"].shift(20))
    df["momentum_60h"] = np.log(df["close"]) - np.log(df["close"].shift(60))

    # 10. Volume Z-Score (20-period trailing)
    vol_mean = df["volume"].rolling(20, min_periods=20).mean()
    vol_std = df["volume"].rolling(20, min_periods=20).std().replace(0, np.nan)
    df["volume_zscore_20"] = (df["volume"] - vol_mean) / vol_std

    # 11. High-Low Range %
    df["hl_range_pct"] = (df["high"] - df["low"]) / df["close"]

    # 12. Candle Body %
    df["candle_body_pct"] = (df["close"] - df["open"]).abs() / (df["high"] - df["low"]).replace(0, np.nan)

    # Clean initial NaN warm-up window
    df_clean = df.dropna().reset_index(drop=True)

    # Save symbol-specific feature dataset
    symbol = df["symbol"].iloc[0]
    proc_path = PROCESSED_DIR / f"{symbol}_1h_features.csv"
    df_clean.to_csv(proc_path, index=False)
    print(f" Saved feature-engineered dataset: {proc_path} ({len(df_clean)} rows)")
    return proc_path

# ==============================================================================
# MAIN PIPELINE EXECUTION
# ==============================================================================
if __name__ == "__main__":
    start_ms = to_ms(START_DATE)
    end_ms = to_ms(END_DATE) if END_DATE else int(time.time() * 1000)

    for symbol in SYMBOLS:
        # Step 1: Fetch and store raw data directly per symbol
        raw_csv_path = fetch_and_save_raw(symbol, start_ms, end_ms)
        
        # Step 2: Build causal features and export individual feature datasets
        engineer_features(raw_csv_path)

    print("\n Pipeline completed successfully!")

import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM
from scipy.optimize import linear_sum_assignment

# ==============================================================================
# CONFIGURATION & PATH SETUP
# ==============================================================================
SYMBOLS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XAUUSDT", "XAGUSDT"]

# Local folder setup
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DATA_DIR / "regimes"

for d in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HMM_FEATURES = [
    "log_return",
    "log_realized_vol_20",
    "adx_14",
    "log_atr_pct",
    "hurst_100",
    "momentum_5",
    "momentum_20",
    "momentum_60",
    "volume_zscore_20"
]

EVENT_FEATURE = "event_flag"

# HMM Configuration
HMM_CANDIDATE_STATES = [2, 3, 4, 5]
HMM_FINAL_N_STATES = 4
HMM_COVARIANCE_TYPE = "diag"
HMM_N_ITER = 800
HMM_TOL = 1e-4
HMM_N_RESTARTS = 5
RANDOM_SEED = 42

# Walk-Forward Setup (1-hour bars)
WF_TRAIN_BARS = 24 * 180
WF_TEST_BARS = 24 * 30
WF_STEP_BARS = 24 * 30

# ==============================================================================
# STAGE 2: LEAK-FREE FEATURE ENGINEERING
# ==============================================================================
def calculate_hurst(series: np.ndarray) -> float:
    x = series[~np.isnan(series)]
    n = len(x)
    if n < 20:
        return np.nan
    lags = np.unique(np.geomspace(2, n // 2, num=10).astype(int))
    lags = lags[lags >= 2]
    if len(lags) < 3:
        return np.nan
    tau = [np.std(x[lag:] - x[:-lag]) for lag in lags]
    tau = np.array(tau)
    valid = ~np.isnan(tau) & (tau > 0)
    if valid.sum() < 3:
        return np.nan
    poly = np.polyfit(np.log(lags[valid]), np.log(tau[valid]), 1)
    return poly[0] * 2.0

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("open_time").drop_duplicates("open_time").reset_index(drop=True)

    # 1. Log Return
    df["log_return"] = np.log(df["close"]).diff()

    # 2. Log Realized Volatility (20-period trailing)
    realized_vol = df["log_return"].rolling(20, min_periods=20).std()
    df["log_realized_vol_20"] = np.log(realized_vol.replace(0, np.nan))

    # 3. ATR & 4. Log ATR Percentage (14-period Wilder Smoothed)
    high_low = df["high"] - df["low"]
    high_pc = (df["high"] - df["close"].shift(1)).abs()
    low_pc = (df["low"] - df["close"].shift(1)).abs()
    tr = pd.concat([high_low, high_pc, low_pc], axis=1).max(axis=1)
    
    atr_14 = tr.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    atr_pct = atr_14 / df["close"]
    df["log_atr_pct"] = np.log(atr_pct.replace(0, np.nan))

    # 5. ADX (14-period)
    up_move = df["high"].diff()
    down_move = -df["low"].diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    
    plus_di = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1/14, adjust=False, min_periods=14).mean() / atr_14
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/14, adjust=False, min_periods=14).mean() / atr_14
    dx = (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan) * 100
    df["adx_14"] = dx.ewm(alpha=1/14, adjust=False, min_periods=14).mean()

    # 6. Hurst Exponent (100-period trailing log returns)
    df["hurst_100"] = df["log_return"].rolling(100, min_periods=100).apply(calculate_hurst, raw=True)

    # 7-9. Momentum (5h, 20h, 60h)
    df["momentum_5"] = np.log(df["close"]) - np.log(df["close"].shift(5))
    df["momentum_20"] = np.log(df["close"]) - np.log(df["close"].shift(20))
    df["momentum_60"] = np.log(df["close"]) - np.log(df["close"].shift(60))

    # 10. Volume Z-Score (20-period trailing)
    vol_mean = df["volume"].rolling(20, min_periods=20).mean()
    vol_std = df["volume"].rolling(20, min_periods=20).std().replace(0, np.nan)
    df["volume_zscore_20"] = (df["volume"] - vol_mean) / vol_std

    return df.dropna().reset_index(drop=True)





[Fetch] Downloading raw data for BTCUSDT...
 Saved raw dataset: ..\data\raw\BTCUSDT_1h_raw.csv (61585 rows)
 Saved feature-engineered dataset: ..\data\processed\BTCUSDT_1h_features.csv (61485 rows)
[Fetch] Downloading raw data for ETHUSDT...
 Saved raw dataset: ..\data\raw\ETHUSDT_1h_raw.csv (59675 rows)
 Saved feature-engineered dataset: ..\data\processed\ETHUSDT_1h_features.csv (59575 rows)
[Fetch] Downloading raw data for BNBUSDT...
 Saved raw dataset: ..\data\raw\BNBUSDT_1h_raw.csv (57874 rows)
 Saved feature-engineered dataset: ..\data\processed\BNBUSDT_1h_features.csv (57774 rows)
[Fetch] Downloading raw data for XAUUSDT...
 Saved raw dataset: ..\data\raw\XAUUSDT_1h_raw.csv (6730 rows)
 Saved feature-engineered dataset: ..\data\processed\XAUUSDT_1h_features.csv (6629 rows)
[Fetch] Downloading raw data for XAGUSDT...
 Saved raw dataset: ..\data\raw\XAGUSDT_1h_raw.csv (6080 rows)
 Saved feature-engineered dataset: ..\data\processed\XAGUSDT_1h_features.csv (5980 rows)

 Pipeline com

In [2]:
# =============================================================================
# STAGE 5
# Standardization + Walk-Forward HMM Training
# Leakage-Safe Regime Detection
#
# Regimes:
#   1 = Trending
#   2 = Mean Reversion
#   3 = High Volatility
#   4 = Event
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM


# =============================================================================
# CONFIGURATION
# =============================================================================

SYMBOLS = [
    "BTCUSDT",
    "ETHUSDT",
    "BNBUSDT",
    "XAUUSDT",
    "XAGUSDT"
]

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DATA_DIR / "regimes"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# HMM FEATURES
# -----------------------------------------------------------------------------
# These names MUST match the columns actually produced by your Stage 2
# feature-engineering code.

HMM_FEATURES = [
    "log_return",
    "realized_vol_20",
    "adx_14",
    "atr_pct",
    "hurst_100",
    "momentum_5h",
    "momentum_20h",
    "momentum_60h",
    "volume_zscore_20",
]

# Compatibility for Stage 2 feature files
FEATURE_ALIASES = {
    "log_realized_vol_20": "realized_vol_20",
    "log_atr_pct": "atr_pct",
    "momentum_5": "momentum_5h",
    "momentum_20": "momentum_20h",
    "momentum_60": "momentum_60h",
}


# -----------------------------------------------------------------------------
# EVENT FEATURE
# -----------------------------------------------------------------------------
#
# IMPORTANT:
#
# If event_flag does not exist in your processed dataset, the code below
# creates it as 0.
#
# That means the HMM CANNOT genuinely learn an event regime from event_flag
# unless you supply event observations.
#
# You should eventually populate EVENT_DATES below.
# -----------------------------------------------------------------------------

EVENT_FEATURE = "event_flag"


# Example:
#
# EVENT_DATES = {
#     "BTCUSDT": [],
#     "ETHUSDT": [],
#     "BNBUSDT": [],
#     "XAUUSDT": [],
#     "XAGUSDT": []
# }
#
# You can later add timestamps corresponding to major events.

EVENT_DATES = {
    symbol: []
    for symbol in SYMBOLS
}


# -----------------------------------------------------------------------------
# HMM MODEL SELECTION
# -----------------------------------------------------------------------------

HMM_CANDIDATE_STATES = [2, 3, 4, 5]

# Your requested architecture is four regimes.
HMM_FINAL_N_STATES = 4

HMM_COVARIANCE_TYPE = "diag"

HMM_N_ITER = 800
HMM_TOL = 1e-4

# Multiple random restarts
HMM_N_RESTARTS = 5

RANDOM_SEED = 42


# -----------------------------------------------------------------------------
# WALK-FORWARD SETTINGS
# -----------------------------------------------------------------------------

# 1-hour data
WF_TRAIN_BARS = 24 * 180     # 180 days
WF_TEST_BARS = 24 * 30       # 30 days
WF_STEP_BARS = 24 * 30       # move forward 30 days


# =============================================================================
# REGIME DEFINITIONS
# =============================================================================

REGIME_LABELS = {
    1: "Trending",
    2: "Mean Reversion",
    3: "High Volatility",
    4: "Event",
}


# =============================================================================
# EVENT FLAG
# =============================================================================

def add_event_flag(df: pd.DataFrame, symbol: str) -> pd.DataFrame:

    df = df.copy()

    df[EVENT_FEATURE] = 0

    if symbol not in EVENT_DATES:
        return df

    if not EVENT_DATES[symbol]:
        return df

    event_dates = pd.to_datetime(
        EVENT_DATES[symbol],
        utc=True
    )

    event_days = set(event_dates.normalize())

    normalized_dates = df["open_time"].dt.normalize()

    df.loc[
        normalized_dates.isin(event_days),
        EVENT_FEATURE
    ] = 1

    return df


# =============================================================================
# LOAD DATA
# =============================================================================

def load_symbol_data(symbol: str) -> pd.DataFrame:

    path = PROCESSED_DIR / f"{symbol}_1h_features.csv"

    if not path.exists():
        raise FileNotFoundError(
            f"Feature file not found:\n{path}"
        )

    df = pd.read_csv(path)

    df["open_time"] = pd.to_datetime(df["open_time"], utc=True)

    df = (
        df
        .sort_values("open_time")
        .drop_duplicates("open_time")
        .reset_index(drop=True)
    )

    # Normalize Stage 2 feature names to the names expected by Stage 5
    df = df.rename(columns=FEATURE_ALIASES)

    df = add_event_flag(df, symbol)

    # Check required features
    missing = [
        col
        for col in HMM_FEATURES
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{symbol}: Missing HMM features: {missing}"
        )

    return df


# =============================================================================
# HMM PARAMETER COUNT
# =============================================================================

def count_hmm_parameters(
    n_states: int,
    n_features: int
) -> int:

    # Initial state probabilities
    start_params = n_states - 1

    # Transition matrix
    transition_params = n_states * (n_states - 1)

    # Mean parameters
    mean_params = n_states * n_features

    # Diagonal covariance parameters
    covariance_params = n_states * n_features

    return (
        start_params
        + transition_params
        + mean_params
        + covariance_params
    )


# =============================================================================
# BIC / AIC
# =============================================================================

def calculate_information_criteria(
    log_likelihood: float,
    n_samples: int,
    n_states: int,
    n_features: int
):

    n_params = count_hmm_parameters(
        n_states,
        n_features
    )

    aic = (
        -2 * log_likelihood
        + 2 * n_params
    )

    bic = (
        -2 * log_likelihood
        + n_params * np.log(n_samples)
    )

    return aic, bic, n_params


# =============================================================================
# MULTIPLE HMM RESTARTS
# =============================================================================

def fit_best_hmm(
    X,
    n_states: int,
    seed: int = RANDOM_SEED
):

    best_model = None
    best_score = -np.inf

    for restart in range(HMM_N_RESTARTS):

        model_seed = seed + restart

        model = GaussianHMM(
            n_components=n_states,
            covariance_type=HMM_COVARIANCE_TYPE,
            n_iter=HMM_N_ITER,
            tol=HMM_TOL,
            random_state=model_seed,
            init_params="stmc",
            params="stmc"
        )

        try:

            model.fit(X)

            train_score = model.score(X)

            if train_score > best_score:

                best_score = train_score
                best_model = model

        except Exception as e:

            print(
                f"      Restart {restart + 1} failed: {e}"
            )

    if best_model is None:
        raise RuntimeError(
            f"Unable to fit HMM with {n_states} states."
        )

    return best_model, best_score


# =============================================================================
# MODEL SELECTION
# =============================================================================

def select_hmm_state_count(
    X_train,
    symbol: str
):

    results = []

    print("\n--------------------------------------------")
    print(f"MODEL SELECTION — {symbol}")
    print("--------------------------------------------")

    for n_states in HMM_CANDIDATE_STATES:

        print(
            f"Testing {n_states}-state HMM..."
        )

        model, log_likelihood = fit_best_hmm(
            X_train,
            n_states=n_states
        )

        aic, bic, n_params = (
            calculate_information_criteria(
                log_likelihood=log_likelihood,
                n_samples=len(X_train),
                n_states=n_states,
                n_features=X_train.shape[1]
            )
        )

        results.append({

            "symbol": symbol,

            "n_states": n_states,

            "log_likelihood": log_likelihood,

            "n_parameters": n_params,

            "AIC": aic,

            "BIC": bic,

        })

        print(
            f"   states={n_states} | "
            f"loglik={log_likelihood:.2f} | "
            f"AIC={aic:.2f} | "
            f"BIC={bic:.2f}"
        )

    result_df = pd.DataFrame(results)

    selected_state_count = int(
        result_df.loc[
            result_df["BIC"].idxmin(),
            "n_states"
        ]
    )

    print(
        f"\nBIC-selected state count: "
        f"{selected_state_count}"
    )

    return result_df, selected_state_count


# =============================================================================
# CAUSAL FILTERED PROBABILITIES
# =============================================================================
#
# IMPORTANT:
#
# hmmlearn.predict_proba(X_test)
#
# gives smoothed posterior probabilities.
# A smoothed posterior for time t can depend on observations AFTER t.
#
# That is not acceptable for a forecasting pipeline.
#
# Therefore we calculate forward probabilities:
#
# P(state_t | observations_1 ... observations_t)
#
# which only uses information available up to t.
# =============================================================================

def emission_log_probability(
    model,
    X
):

    means = model.means_

    covars = model.covars_

    n_samples = X.shape[0]
    n_states = model.n_components

    log_prob = np.zeros(
        (n_samples, n_states)
    )

    for state in range(n_states):

        diff = X - means[state]

        variance = covars[state]

        variance = np.maximum(
            variance,
            1e-12
        )

        log_det = np.sum(
            np.log(variance)
        )

        quadratic = np.sum(
            (diff ** 2) / variance,
            axis=1
        )

        log_prob[:, state] = (
            -0.5
            * (
                X.shape[1] * np.log(2 * np.pi)
                + log_det
                + quadratic
            )
        )

    return log_prob


def causal_filter_probabilities(
    model,
    X
):

    """
    Computes:

        P(S_t | X_1,...,X_t)

    rather than:

        P(S_t | X_1,...,X_T)

    Therefore future test observations cannot affect the
    regime probability at time t.
    """

    n_samples = X.shape[0]

    n_states = model.n_components

    log_emissions = emission_log_probability(
        model,
        X
    )

    log_start = np.log(
        np.maximum(
            model.startprob_,
            1e-300
        )
    )

    log_trans = np.log(
        np.maximum(
            model.transmat_,
            1e-300
        )
    )

    filtered = np.zeros(
        (n_samples, n_states)
    )

    # -------------------------------------------------------------------------
    # First observation
    # -------------------------------------------------------------------------

    log_alpha = (
        log_start
        + log_emissions[0]
    )

    max_log = np.max(log_alpha)

    alpha = np.exp(
        log_alpha - max_log
    )

    alpha /= alpha.sum()

    filtered[0] = alpha

    # -------------------------------------------------------------------------
    # Remaining observations
    # -------------------------------------------------------------------------

    for t in range(1, n_samples):

        new_log_alpha = np.empty(
            n_states
        )

        for j in range(n_states):

            transition_terms = (
                log_alpha
                + log_trans[:, j]
            )

            max_val = np.max(
                transition_terms
            )

            log_sum = (
                max_val
                + np.log(
                    np.sum(
                        np.exp(
                            transition_terms
                            - max_val
                        )
                    )
                )
            )

            new_log_alpha[j] = (
                log_emissions[t, j]
                + log_sum
            )

        max_log = np.max(
            new_log_alpha
        )

        alpha = np.exp(
            new_log_alpha
            - max_log
        )

        alpha /= alpha.sum()

        filtered[t] = alpha

        log_alpha = new_log_alpha

    return filtered


# =============================================================================
# STATE INTERPRETATION
# =============================================================================

def interpret_states(
    model,
    scaler,
    train_df: pd.DataFrame
):

    """
    Assign the four latent HMM states to:

        1 = Trending
        2 = Mean Reversion
        3 = High Volatility
        4 = Event

    using training-fold feature characteristics.

    IMPORTANT:
    HMM states are latent. Their numerical IDs have no economic meaning.
    """

    feature_means_scaled = model.means_

    # Convert HMM means back to original feature scale
    feature_means = scaler.inverse_transform(
        feature_means_scaled
    )

    means_df = pd.DataFrame(
        feature_means,
        columns=HMM_FEATURES
    )

    means_df["latent_state"] = np.arange(
        model.n_components
    )

    # Add event mean if available
    if EVENT_FEATURE in train_df.columns:

        # Calculate event exposure by latent state
        X_train = scaler.transform(
            train_df[HMM_FEATURES].values
        )

        train_probs = causal_filter_probabilities(
            model,
            X_train
        )

        train_states = np.argmax(
            train_probs,
            axis=1
        )

        event_exposure = {}

        for state in range(model.n_components):

            mask = (
                train_states == state
            )

            if mask.sum() > 0:

                event_exposure[state] = (
                    train_df.loc[
                        train_df.index[mask],
                        EVENT_FEATURE
                    ].mean()
                )

            else:

                event_exposure[state] = 0.0

        means_df["event_exposure"] = (
            means_df["latent_state"]
            .map(event_exposure)
            .fillna(0)
        )

    else:

        means_df["event_exposure"] = 0.0


    # -------------------------------------------------------------------------
    # Normalize ranking metrics
    # -------------------------------------------------------------------------

    def safe_rank(series):

        if series.nunique() <= 1:
            return pd.Series(
                0.5,
                index=series.index
            )

        return series.rank(
            pct=True
        )


    # High volatility score
    vol_score = (
        safe_rank(
            means_df["realized_vol_20"]
        )
        +
        safe_rank(
            means_df["atr_pct"]
        )
    ) / 2


    # Trend score
    #
    # High ADX = stronger trend
    # High absolute momentum = directional movement
    # Hurst > 0.5 tends to support persistence
    #

    momentum_strength = (
        means_df[
            [
                "momentum_5h",
                "momentum_20h",
                "momentum_60h"
            ]
        ]
        .abs()
        .mean(axis=1)
    )

    trend_score = (
        safe_rank(
            means_df["adx_14"]
        )
        +
        safe_rank(
            momentum_strength
        )
        +
        safe_rank(
            means_df["hurst_100"]
        )
    ) / 3


    # Mean-reversion score
    #
    # Lower Hurst + weaker ADX + weaker momentum
    #

    mean_reversion_score = (
        (1 - safe_rank(
            means_df["hurst_100"]
        ))
        +
        (1 - safe_rank(
            means_df["adx_14"]
        ))
        +
        (1 - safe_rank(
            momentum_strength
        ))
    ) / 3


    # Event score
    #
    # Event states should exhibit:
    #
    #   high event exposure
    #   high volatility
    #

    event_score = (
        safe_rank(
            means_df["event_exposure"]
        )
        * 0.7
        +
        vol_score * 0.3
    )


    means_df["trend_score"] = trend_score
    means_df["mean_reversion_score"] = (
        mean_reversion_score
    )
    means_df["volatility_score"] = vol_score
    means_df["event_score"] = event_score


    # -------------------------------------------------------------------------
    # Assign states
    # -------------------------------------------------------------------------

    assigned = {}

    available_states = set(
        means_df["latent_state"]
    )

    # Event first
    event_state = (
        means_df
        .loc[
            means_df["latent_state"].isin(
                available_states
            ),
            "event_score"
        ]
        .idxmax()
    )

    event_latent = int(
        means_df.loc[
            event_state,
            "latent_state"
        ]
    )

    assigned[event_latent] = 4

    available_states.remove(
        event_latent
    )


    # High volatility second
    vol_candidates = means_df[
        means_df["latent_state"].isin(
            available_states
        )
    ]

    vol_latent = int(
        vol_candidates.loc[
            vol_candidates["volatility_score"].idxmax(),
            "latent_state"
        ]
    )

    assigned[vol_latent] = 3

    available_states.remove(
        vol_latent
    )


    # Trending third
    trend_candidates = means_df[
        means_df["latent_state"].isin(
            available_states
        )
    ]

    trend_latent = int(
        trend_candidates.loc[
            trend_candidates["trend_score"].idxmax(),
            "latent_state"
        ]
    )

    assigned[trend_latent] = 1

    available_states.remove(
        trend_latent
    )


    # Remaining state = mean reversion
    for latent_state in available_states:

        assigned[int(latent_state)] = 2


    means_df["regime"] = (
        means_df["latent_state"]
        .map(assigned)
    )

    means_df["regime_label"] = (
        means_df["regime"]
        .map(REGIME_LABELS)
    )

    return means_df, assigned


# =============================================================================
# PROCESS ONE SYMBOL
# =============================================================================

def process_symbol(symbol: str):

    print("\n")
    print("=" * 80)
    print(f"STAGE 5 — {symbol}")
    print("=" * 80)

    df = load_symbol_data(symbol)

    print(
        f"Loaded {len(df):,} rows."
    )

    # -------------------------------------------------------------------------
    # First fold
    # -------------------------------------------------------------------------

    first_train_end = WF_TRAIN_BARS

    if len(df) <= first_train_end:

        raise ValueError(
            f"{symbol}: insufficient observations "
            f"for first training window."
        )

    first_train = df.iloc[
        :first_train_end
    ].copy()

    X_first = first_train[
        HMM_FEATURES
    ].values

    # -------------------------------------------------------------------------
    # FIRST-FOLD SCALER
    # -------------------------------------------------------------------------

    first_scaler = StandardScaler()

    X_first_scaled = (
        first_scaler.fit_transform(
            X_first
        )
    )

    # -------------------------------------------------------------------------
    # MODEL SELECTION
    # -------------------------------------------------------------------------

    selection_df, bic_selected_states = (
        select_hmm_state_count(
            X_first_scaled,
            symbol
        )
    )

    selection_path = (
        OUTPUT_DIR
        / f"{symbol}_05_hmm_model_selection.csv"
    )

    selection_df.to_csv(
        selection_path,
        index=False
    )

    print(
        f"\nSaved model selection:\n"
        f"{selection_path}"
    )

    # -------------------------------------------------------------------------
    # FORCE FOUR STATES
    # -------------------------------------------------------------------------
    #
    # Your architecture explicitly requires:
    #
    # 1 Trending
    # 2 Mean Reversion
    # 3 High Volatility
    # 4 Event
    #
    # Therefore the final model uses 4 states.
    #

    final_n_states = HMM_FINAL_N_STATES

    print(
        f"\nFinal HMM state count: "
        f"{final_n_states}"
    )

    if bic_selected_states != final_n_states:

        print(
            f"WARNING: BIC selected "
            f"{bic_selected_states} states, "
            f"but architecture requires "
            f"{final_n_states}."
        )


    # -------------------------------------------------------------------------
    # WALK-FORWARD PROCESSING
    # -------------------------------------------------------------------------

    all_state_results = []

    fold_number = 0

    train_start = 0
    train_end = WF_TRAIN_BARS

    while (
        train_end < len(df)
    ):

        test_start = train_end

        test_end = min(
            test_start + WF_TEST_BARS,
            len(df)
        )

        if test_start >= test_end:
            break

        print(
            f"\nFold {fold_number}"
        )

        print(
            f"Train: "
            f"{df.iloc[train_start]['open_time']} "
            f"→ "
            f"{df.iloc[train_end - 1]['open_time']}"
        )

        print(
            f"Test : "
            f"{df.iloc[test_start]['open_time']} "
            f"→ "
            f"{df.iloc[test_end - 1]['open_time']}"
        )

        train_df = df.iloc[
            train_start:train_end
        ].copy()

        test_df = df.iloc[
            test_start:test_end
        ].copy()


        # =====================================================================
        # FIT SCALER ONLY ON TRAINING DATA
        # =====================================================================

        scaler = StandardScaler()

        X_train = scaler.fit_transform(
            train_df[
                HMM_FEATURES
            ].values
        )

        X_test = scaler.transform(
            test_df[
                HMM_FEATURES
            ].values
        )


        # =====================================================================
        # FIT HMM ONLY ON TRAINING DATA
        # =====================================================================

        model, train_loglik = fit_best_hmm(
            X_train,
            n_states=final_n_states
        )


        # =====================================================================
        # INTERPRET LATENT STATES
        # =====================================================================

        interpretation_df, state_mapping = (
            interpret_states(
                model=model,
                scaler=scaler,
                train_df=train_df
            )
        )


        # =====================================================================
        # CAUSAL TEST PROBABILITIES
        # =====================================================================

        test_probabilities = (
            causal_filter_probabilities(
                model,
                X_test
            )
        )


        # =====================================================================
        # ASSIGNED LATENT STATE
        # =====================================================================

        latent_states = np.argmax(
            test_probabilities,
            axis=1
        )


        # =====================================================================
        # MAP LATENT STATE → ECONOMIC REGIME
        # =====================================================================

        economic_states = np.array([
            state_mapping[int(state)]
            for state in latent_states
        ])


        # =====================================================================
        # SAVE TEST RESULTS
        # =====================================================================

        fold_result = test_df[
            [
                "open_time",
                "symbol"
            ]
        ].copy()

        fold_result["fold"] = fold_number

        # Raw latent state
        fold_result["latent_state"] = (
            latent_states
        )

        # Required regime
        fold_result["hmm_state"] = (
            economic_states
        )

        fold_result["regime_label"] = (
            fold_result["hmm_state"]
            .map(REGIME_LABELS)
        )


        # ---------------------------------------------------------------------
        # State probabilities
        # ---------------------------------------------------------------------

        for latent_state in range(
            final_n_states
        ):

            economic_state = state_mapping[
                latent_state
            ]

            fold_result[
                f"state_prob_{economic_state}"
            ] = test_probabilities[
                :,
                latent_state
            ]


        # ---------------------------------------------------------------------
        # Add original HMM features
        # ---------------------------------------------------------------------

        for feature in HMM_FEATURES:

            fold_result[feature] = (
                test_df[feature]
                .values
            )


        # ---------------------------------------------------------------------
        # Event flag
        # ---------------------------------------------------------------------

        fold_result[EVENT_FEATURE] = (
            test_df[EVENT_FEATURE]
            .values
        )


        all_state_results.append(
            fold_result
        )


        # =====================================================================
        # SAVE INTERPRETATION FOR THIS FOLD
        # =====================================================================

        interpretation_df[
            "symbol"
        ] = symbol

        interpretation_df[
            "fold"
        ] = fold_number

        interpretation_df[
            "train_start"
        ] = train_df[
            "open_time"
        ].iloc[0]

        interpretation_df[
            "train_end"
        ] = train_df[
            "open_time"
        ].iloc[-1]


        interpretation_path = (
            OUTPUT_DIR
            / f"{symbol}_05_hmm_state_interpretation_fold_{fold_number}.csv"
        )

        interpretation_df.to_csv(
            interpretation_path,
            index=False
        )


        print(
            "Regime distribution:"
        )

        print(
            pd.Series(
                economic_states
            )
            .value_counts()
            .sort_index()
            .rename(
                index=REGIME_LABELS
            )
        )


        # =====================================================================
        # MOVE WALK-FORWARD WINDOW
        # =====================================================================

        fold_number += 1

        train_start += WF_STEP_BARS
        train_end += WF_STEP_BARS


    # =============================================================================
    # COMBINE ALL OUT-OF-SAMPLE TEST FOLDS
    # =============================================================================

    if not all_state_results:

        raise RuntimeError(
            f"{symbol}: no walk-forward folds produced."
        )


    states_df = pd.concat(
        all_state_results,
        ignore_index=True
    )

    states_df = (
        states_df
        .sort_values("open_time")
        .reset_index(drop=True)
    )


    # =============================================================================
    # FINAL STATE OUTPUT
    # =============================================================================

    states_path = (
        OUTPUT_DIR
        / f"{symbol}_05_hmm_states.csv"
    )

    states_df.to_csv(
        states_path,
        index=False
    )


    # =============================================================================
    # COMBINED INTERPRETATION
    # =============================================================================

    # For the final interpretation file, calculate summary statistics
    # using the OOS assigned regimes.

    interpretation_summary = []

    for regime in [
        1, 2, 3, 4
    ]:

        regime_data = states_df[
            states_df["hmm_state"] == regime
        ]

        if len(regime_data) == 0:
            continue

        row = {
            "symbol": symbol,
            "regime": regime,
            "regime_label": REGIME_LABELS[regime],
            "observations": len(regime_data),
            "mean_log_return": regime_data[
                "log_return"
            ].mean(),
            "mean_realized_vol_20": regime_data[
                "realized_vol_20"
            ].mean(),
            "mean_adx_14": regime_data[
                "adx_14"
            ].mean(),
            "mean_atr_pct": regime_data[
                "atr_pct"
            ].mean(),
            "mean_hurst_100": regime_data[
                "hurst_100"
            ].mean(),
            "mean_momentum_5h": regime_data[
                "momentum_5h"
            ].mean(),
            "mean_momentum_20h": regime_data[
                "momentum_20h"
            ].mean(),
            "mean_momentum_60h": regime_data[
                "momentum_60h"
            ].mean(),
            "mean_volume_zscore_20": regime_data[
                "volume_zscore_20"
            ].mean(),
            "event_frequency": regime_data[
                EVENT_FEATURE
            ].mean(),
        }

        interpretation_summary.append(
            row
        )


    interpretation_summary_df = pd.DataFrame(
        interpretation_summary
    )


    interpretation_path = (
        OUTPUT_DIR
        / f"{symbol}_05_hmm_state_interpretation.csv"
    )

    interpretation_summary_df.to_csv(
        interpretation_path,
        index=False
    )


    print(
        f"\nSaved HMM states:\n"
        f"{states_path}"
    )

    print(
        f"\nSaved state interpretation:\n"
        f"{interpretation_path}"
    )

    print(
        f"\nCompleted {symbol}."
    )

    return (
        states_df,
        selection_df,
        interpretation_summary_df
    )


# =============================================================================
# RUN ALL SYMBOLS
# =============================================================================

if __name__ == "__main__":

    for symbol in SYMBOLS:

        try:

            feature_path = PROCESSED_DIR / f"{symbol}_1h_features.csv"

            if not feature_path.exists():
                print(f"\n[INFO] Missing feature file for {symbol}; generating it now...")
                start_ms = to_ms(START_DATE)
                end_ms = int(time.time() * 1000) if END_DATE is None else to_ms(END_DATE)

                raw_path = fetch_and_save_raw(symbol, start_ms, end_ms)
                if raw_path is None or not raw_path.exists():
                    raise FileNotFoundError(f"Raw data file could not be created for {symbol}.")

                raw_df = pd.read_csv(raw_path)
                feature_df = engineer_features(raw_df)  # DataFrame-based version from Stage 5
                feature_path = PROCESSED_DIR / f"{symbol}_1h_features.csv"
                feature_df.to_csv(feature_path, index=False)
                print(f"[INFO] Generated feature file for {symbol}: {feature_path}")
                if feature_path is None or not feature_path.exists():
                    raise FileNotFoundError(f"Feature file could not be created for {symbol}.")

            # Compatibility fix for hmmlearn: some versions expose diagonal covariances as
            # (n_states, n_features), while others may return full covariances.
            # This prevents the broadcast error: (4320, 9) vs (9, 9).
            def emission_log_probability(model, X):
                X = np.asarray(X, dtype=float)
                means = np.asarray(model.means_, dtype=float)
                covars = np.asarray(model.covars_, dtype=float)

                n_samples, n_features = X.shape
                n_states = model.n_components
                log_prob = np.zeros((n_samples, n_states), dtype=float)

                # Diagonal covariance: covars shape -> (n_states, n_features)
                if covars.ndim == 2:
                    variances = np.maximum(covars, 1e-12)
                    log_det = np.sum(np.log(variances), axis=1)

                    for state in range(n_states):
                        diff = X - means[state]
                        quad = np.sum((diff ** 2) / variances[state], axis=1)
                        log_prob[:, state] = -0.5 * (
                            n_features * np.log(2 * np.pi)
                            + log_det[state]
                            + quad
                        )
                    return log_prob

                # Full covariance: covars shape -> (n_states, n_features, n_features)
                for state in range(n_states):
                    diff = X - means[state]
                    cov = covars[state].copy()
                    cov = cov + np.eye(n_features) * 1e-6
                    sign, logdet = np.linalg.slogdet(cov)
                    if sign <= 0:
                        cov = cov + np.eye(n_features) * 1e-6
                        sign, logdet = np.linalg.slogdet(cov)

                    inv_cov = np.linalg.inv(cov)
                    quad = np.einsum("...i,ij,...j->...", diff, inv_cov, diff)
                    log_prob[:, state] = -0.5 * (
                        n_features * np.log(2 * np.pi)
                        + logdet
                        + quad
                    )

                return log_prob

            process_symbol(symbol)

        except Exception as e:

            print(
                f"\nERROR processing {symbol}:"
            )

            print(e)

            raise


    print("\n")
    print("=" * 80)
    print("STAGE 5 COMPLETED SUCCESSFULLY")
    print("=" * 80)



STAGE 5 — BTCUSDT
Loaded 58,597 rows.

--------------------------------------------
MODEL SELECTION — BTCUSDT
--------------------------------------------
Testing 2-state HMM...
   states=2 | loglik=-43774.16 | AIC=87626.32 | BIC=87874.79
Testing 3-state HMM...
   states=3 | loglik=-39746.44 | AIC=79616.89 | BIC=80011.89
Testing 4-state HMM...
   states=4 | loglik=-37614.16 | AIC=75402.33 | BIC=75956.60
Testing 5-state HMM...
   states=5 | loglik=-36193.01 | AIC=72614.02 | BIC=73340.31

BIC-selected state count: 5

Saved model selection:
c:\Users\dell\Downloads\data\regimes\BTCUSDT_05_hmm_model_selection.csv

Final HMM state count: 4

Fold 0
Train: 2020-01-05 04:00:00+00:00 → 2020-07-03 03:00:00+00:00
Test : 2020-07-03 04:00:00+00:00 → 2020-08-02 03:00:00+00:00
Regime distribution:
Trending           551
Mean Reversion      73
High Volatility     76
Event               20
Name: count, dtype: int64

Fold 1
Train: 2020-02-04 04:00:00+00:00 → 2020-08-02 03:00:00+00:00
Test : 2020-08-02 

In [3]:
# =============================================================================
# STAGE 6
# Future-Regime Target Construction
# =============================================================================
#
# Purpose:
# For every time t, construct:
#
# future_regime[t] =
#     MODE(
#         hmm_state[t+1],
#         hmm_state[t+2],
#         hmm_state[t+3],
#         hmm_state[t+4]
#     )
#
# IMPORTANT:
# 1. Uses ONLY Stage-5 OOS HMM states.
# 2. Future states are calculated WITHIN THE SAME STAGE-5 FOLD.
# 3. Last 4 observations of every fold are dropped.
# 4. Features at time t are untouched.
# 5. The future label is allowed to look forward.
#
# Output:
# data/regimes/{SYMBOL}_06_ml_dataset.csv
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd


# =============================================================================
# CONFIGURATION
# =============================================================================

SYMBOLS = [
    "BTCUSDT",
    "ETHUSDT",
    "BNBUSDT",
    "XAUUSDT",
    "XAGUSDT"
]

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
REGIME_DIR = DATA_DIR / "regimes"

REGIME_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# TARGET HORIZON
# =============================================================================

TARGET_HORIZON_BARS = 4


# =============================================================================
# REGIME DEFINITIONS
# =============================================================================

REGIME_LABELS = {
    1: "Trending",
    2: "Mean Reversion",
    3: "High Volatility",
    4: "Event"
}


# =============================================================================
# FEATURES THAT STAGE 7 IS ALLOWED TO USE
# =============================================================================
#
# These are exactly the HMM features used in Stage 5.
#
# Stage 5's existing implementation uses:
# log_return, realized_vol_20, adx_14, atr_pct, hurst_100,
# momentum_5h, momentum_20h, momentum_60h, volume_zscore_20
#
# Event flag is additionally included.
# =============================================================================

HMM_FEATURES = [
    "log_return",
    "realized_vol_20",
    "adx_14",
    "atr_pct",
    "hurst_100",
    "momentum_5h",
    "momentum_20h",
    "momentum_60h",
    "volume_zscore_20"
]

EVENT_FEATURE = "event_flag"


# =============================================================================
# LOAD STAGE 5 OOS STATES
# =============================================================================

def load_stage5_states(symbol):

    path = REGIME_DIR / f"{symbol}_05_hmm_states.csv"

    if not path.exists():
        raise FileNotFoundError(
            f"\nStage 5 state file not found for {symbol}:\n{path}"
        )

    df = pd.read_csv(path)

    if "open_time" not in df.columns:
        raise ValueError(
            f"{symbol}: 'open_time' column missing."
        )

    if "fold" not in df.columns:
        raise ValueError(
            f"{symbol}: 'fold' column missing."
        )

    if "hmm_state" not in df.columns:
        raise ValueError(
            f"{symbol}: 'hmm_state' column missing."
        )

    # -------------------------------------------------------------------------
    # Datetime
    # -------------------------------------------------------------------------

    df["open_time"] = pd.to_datetime(
        df["open_time"],
        utc=True,
        errors="coerce"
    )

    if df["open_time"].isna().any():
        raise ValueError(
            f"{symbol}: invalid timestamps detected."
        )

    # -------------------------------------------------------------------------
    # Sort chronologically
    # -------------------------------------------------------------------------

    df = (
        df
        .sort_values(["fold", "open_time"])
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # Check duplicate timestamps within fold
    # -------------------------------------------------------------------------

    duplicates = df.duplicated(
        subset=["fold", "open_time"]
    ).sum()

    if duplicates > 0:
        raise ValueError(
            f"{symbol}: found {duplicates} duplicate "
            f"(fold, open_time) observations."
        )

    # -------------------------------------------------------------------------
    # Validate HMM states
    # -------------------------------------------------------------------------

    valid_states = {1, 2, 3, 4}

    bad_states = set(
        df["hmm_state"]
        .dropna()
        .astype(int)
        .unique()
    ) - valid_states

    if bad_states:
        raise ValueError(
            f"{symbol}: invalid HMM states found: {bad_states}"
        )

    return df


# =============================================================================
# CHECK REQUIRED FEATURES
# =============================================================================

def validate_features(df, symbol):

    required_features = (
        [
            "open_time",
            "symbol",
            "fold",
            "hmm_state"
        ]
        + HMM_FEATURES
        + [EVENT_FEATURE]
        + [
            "state_prob_1",
            "state_prob_2",
            "state_prob_3",
            "state_prob_4"
        ]
    )

    missing = [
        col
        for col in required_features
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"\n{symbol}: Stage 5 file is missing required "
            f"columns:\n{missing}"
        )


# =============================================================================
# MODE FUNCTION
# =============================================================================

def deterministic_mode(values):

    """
    Return the most frequent regime.

    If there is a tie:
    choose the regime that appears earliest in the
    t+1 -> t+4 sequence.

    Example:

    [1, 2, 1, 2]

    both 1 and 2 occur twice.

    Return 1 because it appears first.
    """

    values = [
        int(v)
        for v in values
        if pd.notna(v)
    ]

    if not values:
        return np.nan

    counts = {}

    for value in values:
        counts[value] = counts.get(value, 0) + 1

    max_count = max(counts.values())

    # Earliest occurrence wins in a tie
    for value in values:
        if counts[value] == max_count:
            return value

    return np.nan


# =============================================================================
# CONSTRUCT FUTURE TARGET
# =============================================================================

def construct_future_target(df, symbol):

    df = df.copy()

    # -------------------------------------------------------------------------
    # IMPORTANT:
    #
    # Shift INSIDE each fold.
    #
    # This prevents:
    #
    # Fold 0 final row
    #       ↓
    # using
    # Fold 1 state
    #
    # which would be wrong because Fold 1 was generated by another HMM fit.
    # -------------------------------------------------------------------------

    grouped_state = df.groupby("fold")["hmm_state"]

    for h in range(1, TARGET_HORIZON_BARS + 1):

        df[f"future_state_t_plus_{h}"] = (
            grouped_state.shift(-h)
        )

    # -------------------------------------------------------------------------
    # MODE ACROSS t+1 ... t+4
    # -------------------------------------------------------------------------

    future_columns = [
        f"future_state_t_plus_{h}"
        for h in range(1, TARGET_HORIZON_BARS + 1)
    ]

    df["future_regime"] = df.apply(
        lambda row: deterministic_mode(
            row[future_columns].values
        ),
        axis=1
    )

    # -------------------------------------------------------------------------
    # Number of bars ahead
    # -------------------------------------------------------------------------

    df["bars_to_target"] = TARGET_HORIZON_BARS

    # -------------------------------------------------------------------------
    # Drop rows without four future observations.
    #
    # These are normally the final 4 rows of every Stage-5 fold.
    # -------------------------------------------------------------------------

    before = len(df)

    df = df.dropna(
        subset=future_columns
    ).copy()

    dropped = before - len(df)

    print(
        f"{symbol}: dropped {dropped:,} rows "
        f"without complete t+1...t+4 target."
    )

    # -------------------------------------------------------------------------
    # Convert target to integer
    # -------------------------------------------------------------------------

    df["future_regime"] = (
        df["future_regime"]
        .astype(int)
    )

    # -------------------------------------------------------------------------
    # Human-readable target label
    # Useful for checking, but NOT used by LightGBM.
    # -------------------------------------------------------------------------

    df["future_regime_label"] = (
        df["future_regime"]
        .map(REGIME_LABELS)
    )

    return df


# =============================================================================
# VALIDATE TARGET
# =============================================================================

def validate_target(df, symbol):

    # Target must only contain 1-4
    valid = {1, 2, 3, 4}

    actual = set(
        df["future_regime"]
        .dropna()
        .astype(int)
        .unique()
    )

    invalid = actual - valid

    if invalid:
        raise ValueError(
            f"{symbol}: invalid future regimes: {invalid}"
        )

    # Check probability columns
    prob_cols = [
        "state_prob_1",
        "state_prob_2",
        "state_prob_3",
        "state_prob_4"
    ]

    prob_sum = df[prob_cols].sum(axis=1)

    max_error = np.abs(
        prob_sum - 1.0
    ).max()

    if max_error > 1e-4:

        print(
            f"WARNING {symbol}: state probabilities "
            f"do not sum exactly to 1."
        )

        print(
            f"Maximum error = {max_error:.8f}"
        )

    # Check future target distribution

    print("\nFuture regime distribution:")

    distribution = (
        df["future_regime"]
        .value_counts()
        .sort_index()
        .rename(index=REGIME_LABELS)
    )

    print(distribution)


# =============================================================================
# PROCESS ONE SYMBOL
# =============================================================================

def process_symbol(symbol):

    print("\n")
    print("=" * 80)
    print(f"STAGE 6 — {symbol}")
    print("=" * 80)

    # -------------------------------------------------------------------------
    # Load Stage 5
    # -------------------------------------------------------------------------

    df = load_stage5_states(symbol)

    print(
        f"Loaded Stage 5 OOS rows: {len(df):,}"
    )

    # -------------------------------------------------------------------------
    # Validate
    # -------------------------------------------------------------------------

    validate_features(
        df,
        symbol
    )

    # -------------------------------------------------------------------------
    # Print fold information
    # -------------------------------------------------------------------------

    print(
        f"Number of Stage 5 folds: "
        f"{df['fold'].nunique()}"
    )

    print("\nRows per fold:")

    print(
        df.groupby("fold")
        .size()
        .to_string()
    )

    # -------------------------------------------------------------------------
    # Construct target
    # -------------------------------------------------------------------------

    df = construct_future_target(
        df,
        symbol
    )

    # -------------------------------------------------------------------------
    # Validate target
    # -------------------------------------------------------------------------

    validate_target(
        df,
        symbol
    )

    # -------------------------------------------------------------------------
    # Remove temporary future-state columns from ML dataset?
    #
    # We KEEP them because they are useful for auditing Stage 6.
    # Stage 7 explicitly excludes them from its feature list.
    # -------------------------------------------------------------------------

    # -------------------------------------------------------------------------
    # Sort
    # -------------------------------------------------------------------------

    df = (
        df
        .sort_values(["fold", "open_time"])
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # Save
    # -------------------------------------------------------------------------

    output_path = (
        REGIME_DIR
        / f"{symbol}_06_ml_dataset.csv"
    )

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"\nSaved Stage 6 dataset:"
        f"\n{output_path}"
    )

    print(
        f"Final rows: {len(df):,}"
    )

    print(
        "\nTarget example:"
    )

    display_columns = [
        "open_time",
        "fold",
        "hmm_state",
        "future_state_t_plus_1",
        "future_state_t_plus_2",
        "future_state_t_plus_3",
        "future_state_t_plus_4",
        "future_regime",
        "future_regime_label"
    ]

    print(
        df[display_columns]
        .head(10)
        .to_string(index=False)
    )

    return df


# =============================================================================
# RUN ALL SYMBOLS
# =============================================================================

if __name__ == "__main__":

    for symbol in SYMBOLS:

        try:

            process_symbol(symbol)

        except Exception as e:

            print(
                f"\nERROR processing {symbol}:"
            )

            print(e)

            raise

    print("\n")
    print("=" * 80)
    print("STAGE 6 COMPLETED SUCCESSFULLY")
    print("=" * 80)



STAGE 6 — BTCUSDT
Loaded Stage 5 OOS rows: 54,277
Number of Stage 5 folds: 76

Rows per fold:
fold
0     720
1     720
2     720
3     720
4     720
5     720
6     720
7     720
8     720
9     720
10    720
11    720
12    720
13    720
14    720
15    720
16    720
17    720
18    720
19    720
20    720
21    720
22    720
23    720
24    720
25    720
26    720
27    720
28    720
29    720
30    720
31    720
32    720
33    720
34    720
35    720
36    720
37    720
38    720
39    720
40    720
41    720
42    720
43    720
44    720
45    720
46    720
47    720
48    720
49    720
50    720
51    720
52    720
53    720
54    720
55    720
56    720
57    720
58    720
59    720
60    720
61    720
62    720
63    720
64    720
65    720
66    720
67    720
68    720
69    720
70    720
71    720
72    720
73    720
74    720
75    277
BTCUSDT: dropped 304 rows without complete t+1...t+4 target.

Future regime distribution:
future_regime
Trending           15275
Mean Rever

In [7]:
# =============================================================================
# STAGE 7
# UPCOMING REGIME CLASSIFIER — LIGHTGBM
#
# Objective:
# Predict the DOMINANT HMM REGIME over the next 4 hourly bars.
#
# Target:
# future_regime = MODE(
#       hmm_state[t+1],
#       hmm_state[t+2],
#       hmm_state[t+3],
#       hmm_state[t+4]
# )
#
# Regimes:
#   1 = Trending
#   2 = Mean Reversion
#   3 = High Volatility
#   4 = Event
#
# Leakage controls:
#   1. Same Stage-5 / Stage-6 chronological folds
#   2. No random train/test split
#   3. Last 4 training observations are purged
#   4. Features are ONLY information available at t
#   5. Test block is strictly out-of-sample
#   6. LightGBM is fitted separately inside every fold
# =============================================================================


import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    classification_report,
    confusion_matrix
)


# =============================================================================
# CONFIGURATION
# =============================================================================

SYMBOLS = [
    "BTCUSDT",
    "ETHUSDT",
    "BNBUSDT",
    "XAUUSDT",
    "XAGUSDT"
]

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
REGIME_DIR = DATA_DIR / "regimes"

REGIME_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# TARGET HORIZON
# =============================================================================

HORIZON = 4


# =============================================================================
# REQUIRED REGIME DEFINITIONS
# =============================================================================

REGIME_LABELS = {
    1: "Trending",
    2: "Mean Reversion",
    3: "High Volatility",
    4: "Event"
}

REGIME_NAMES = [
    "Trending",
    "Mean Reversion",
    "High Volatility",
    "Event"
]


# =============================================================================
# FEATURES
# =============================================================================
#
# These must be information available AT TIME t.
#
# They come from Stage 5 / Stage 6.
# =============================================================================

FEATURES = [

    # Current HMM regime
    "hmm_state",

    # HMM posterior probabilities
    "state_prob_1",
    "state_prob_2",
    "state_prob_3",
    "state_prob_4",

    # HMM feature vector
    "log_return",
    "realized_vol_20",
    "adx_14",
    "atr_pct",
    "hurst_100",
    "momentum_5h",
    "momentum_20h",
    "momentum_60h",
    "volume_zscore_20",

    # Event information
    "event_flag"
]


# =============================================================================
# WALK-FORWARD SETTINGS
# =============================================================================
#
# IMPORTANT:
# Stage 5 used:
#
# 180-day training
# 30-day testing
# 30-day step
#
# Stage 7 follows the SAME chronology.
# =============================================================================

WF_TRAIN_BARS = 24 * 180
WF_TEST_BARS = 24 * 30
WF_STEP_BARS = 24 * 30


# =============================================================================
# LIGHTGBM SETTINGS
# =============================================================================

LGBM_PARAMS = {

    "objective": "multiclass",

    "num_class": 4,

    "n_estimators": 1000,

    "learning_rate": 0.03,

    "num_leaves": 31,

    "max_depth": -1,

    "min_child_samples": 50,

    "subsample": 0.8,

    "colsample_bytree": 0.8,

    "reg_alpha": 0.1,

    "reg_lambda": 1.0,

    "random_state": 42,

    "n_jobs": -1,

    "verbosity": -1
}


# =============================================================================
# LOAD STAGE 6
# =============================================================================

def load_stage6(symbol):

    path = (
        REGIME_DIR
        / f"{symbol}_06_ml_dataset.csv"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"\nStage 6 file not found:\n{path}"
        )

    df = pd.read_csv(path)

    print(
        f"Loaded Stage 6 dataset for "
        f"{symbol}: {len(df):,} rows"
    )

    # -------------------------------------------------------------------------
    # Datetime
    # -------------------------------------------------------------------------

    df["open_time"] = pd.to_datetime(
        df["open_time"],
        utc=True
    )

    # -------------------------------------------------------------------------
    # Sort chronologically
    # -------------------------------------------------------------------------

    df = (
        df
        .sort_values("open_time")
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # Remove duplicate timestamps
    # -------------------------------------------------------------------------

    df = (
        df
        .drop_duplicates(
            subset=["open_time"],
            keep="first"
        )
        .reset_index(drop=True)
    )

    return df


# =============================================================================
# VALIDATE STAGE 6
# =============================================================================

def validate_stage6(df, symbol):

    required_columns = [

        "open_time",
        "symbol",
        "fold",

        "hmm_state",

        "state_prob_1",
        "state_prob_2",
        "state_prob_3",
        "state_prob_4",

        "future_state_t_plus_1",
        "future_state_t_plus_2",
        "future_state_t_plus_3",
        "future_state_t_plus_4",

        "future_regime",
        "future_regime_label"
    ]

    missing = [
        c
        for c in required_columns
        if c not in df.columns
    ]

    if missing:

        raise ValueError(
            f"{symbol}: Stage 6 is missing columns:\n"
            f"{missing}"
        )

    # -------------------------------------------------------------------------
    # Validate regime values
    # -------------------------------------------------------------------------

    valid_regimes = {1, 2, 3, 4}

    invalid = set(
        pd.to_numeric(
            df["future_regime"],
            errors="coerce"
        )
        .dropna()
        .astype(int)
        .unique()
    ) - valid_regimes

    if invalid:

        raise ValueError(
            f"{symbol}: invalid future_regime values: "
            f"{invalid}"
        )

    print(
        f"{symbol}: Stage 6 validation passed."
    )


# =============================================================================
# PREPARE FEATURES
# =============================================================================

def prepare_features(df, symbol):

    # -------------------------------------------------------------------------
    # Make sure required feature columns exist
    # -------------------------------------------------------------------------

    missing = [
        c
        for c in FEATURES
        if c not in df.columns
    ]

    if missing:

        raise ValueError(
            f"{symbol}: missing ML features:\n"
            f"{missing}"
        )

    # -------------------------------------------------------------------------
    # Numeric conversion
    # -------------------------------------------------------------------------

    X = df[FEATURES].copy()

    for col in FEATURES:

        X[col] = pd.to_numeric(
            X[col],
            errors="coerce"
        )

    # -------------------------------------------------------------------------
    # Infinite values
    # -------------------------------------------------------------------------

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # -------------------------------------------------------------------------
    # We DO NOT forward-fill.
    #
    # Why?
    # Forward filling a feature could carry an old value into a time
    # where it was not actually observed.
    # -------------------------------------------------------------------------

    return X


# =============================================================================
# PREPARE TARGET
# =============================================================================

def prepare_target(df):

    y = pd.to_numeric(
        df["future_regime"],
        errors="coerce"
    )

    return y


# =============================================================================
# CREATE WALK-FORWARD FOLDS
# =============================================================================

def create_walk_forward_folds(df):

    folds = []

    train_start = 0
    train_end = WF_TRAIN_BARS

    fold_number = 0

    while train_end < len(df):

        test_start = train_end

        test_end = min(
            test_start + WF_TEST_BARS,
            len(df)
        )

        if test_start >= test_end:
            break

        folds.append({
            "fold": fold_number,
            "train_start": train_start,
            "train_end": train_end,
            "test_start": test_start,
            "test_end": test_end
        })

        train_start += WF_STEP_BARS
        train_end += WF_STEP_BARS

        fold_number += 1

    return folds


# =============================================================================
# TRAIN ONE LIGHTGBM MODEL
# =============================================================================

def train_model(
    X_train,
    y_train,
    X_valid,
    y_valid
):

    model = LGBMClassifier(
        **LGBM_PARAMS
    )

    model.fit(

        X_train,
        y_train,

        eval_set=[
            (X_valid, y_valid)
        ],

        eval_metric="multi_logloss",

        callbacks=[
            early_stopping(
                100,
                verbose=False
            )
        ]
    )

    return model


# =============================================================================
# PROCESS ONE SYMBOL
# =============================================================================

def process_symbol(symbol):

    print("\n")
    print("=" * 80)
    print(
        f"STAGE 7 — {symbol}"
    )
    print("=" * 80)

    # =========================================================================
    # LOAD DATA
    # =========================================================================

    df = load_stage6(symbol)

    validate_stage6(
        df,
        symbol
    )

    # =========================================================================
    # FEATURES
    # =========================================================================

    X = prepare_features(
        df,
        symbol
    )

    # =========================================================================
    # TARGET
    # =========================================================================

    y = prepare_target(df)

    # -------------------------------------------------------------------------
    # LightGBM uses classes 0,1,2,3 internally.
    #
    # Our economic regimes are:
    #
    # 1 Trending
    # 2 Mean Reversion
    # 3 High Volatility
    # 4 Event
    #
    # Therefore:
    #
    # economic regime 1 -> ML class 0
    # economic regime 2 -> ML class 1
    # economic regime 3 -> ML class 2
    # economic regime 4 -> ML class 3
    # -------------------------------------------------------------------------

    y_ml = y - 1

    # =========================================================================
    # WALK-FORWARD FOLDS
    # =========================================================================

    folds = create_walk_forward_folds(
        df
    )

    print(
        f"\nNumber of Stage 7 folds: "
        f"{len(folds)}"
    )

    # =========================================================================
    # OUTPUT CONTAINERS
    # =========================================================================

    all_predictions = []

    fold_metrics = []

    # =========================================================================
    # WALK FORWARD
    # =========================================================================

    for fold_info in folds:

        fold = fold_info["fold"]

        train_start = fold_info["train_start"]
        train_end = fold_info["train_end"]

        test_start = fold_info["test_start"]
        test_end = fold_info["test_end"]

        print("\n" + "-" * 70)

        print(
            f"Fold {fold}"
        )

        print(
            f"Original train: "
            f"{df.iloc[train_start]['open_time']} "
            f"→ "
            f"{df.iloc[train_end - 1]['open_time']}"
        )

        print(
            f"Test: "
            f"{df.iloc[test_start]['open_time']} "
            f"→ "
            f"{df.iloc[test_end - 1]['open_time']}"
        )

        # =====================================================================
        # PURGE
        # =====================================================================
        #
        # The target at t contains:
        #
        # t+1
        # t+2
        # t+3
        # t+4
        #
        # Therefore the final 4 training rows can contain target information
        # overlapping the beginning of the test period.
        #
        # Remove them.
        # =====================================================================

        purged_train_end = (
            train_end - HORIZON
        )

        if purged_train_end <= train_start:

            print(
                "Skipping fold: "
                "not enough training data after purge."
            )

            continue

        # =====================================================================
        # TRAIN DATA
        # =====================================================================

        X_train = X.iloc[
            train_start:purged_train_end
        ].copy()

        y_train = y_ml.iloc[
            train_start:purged_train_end
        ].copy()

        # =====================================================================
        # TEST DATA
        # =====================================================================

        X_test = X.iloc[
            test_start:test_end
        ].copy()

        y_test = y_ml.iloc[
            test_start:test_end
        ].copy()

        # =====================================================================
        # REMOVE ROWS WITH MISSING FEATURES
        # =====================================================================

        train_valid_mask = (
            X_train.notna().all(axis=1)
            &
            y_train.notna()
        )

        test_valid_mask = (
            X_test.notna().all(axis=1)
            &
            y_test.notna()
        )

        X_train = X_train.loc[
            train_valid_mask
        ]

        y_train = y_train.loc[
            train_valid_mask
        ]

        X_test = X_test.loc[
            test_valid_mask
        ]

        y_test = y_test.loc[
            test_valid_mask
        ]

        # =====================================================================
        # CHECK CLASS AVAILABILITY
        # =====================================================================

        train_classes = sorted(
            y_train.astype(int).unique()
        )

        print(
            f"Training rows after purge: "
            f"{len(X_train):,}"
        )

        print(
            f"Training classes: "
            f"{[c + 1 for c in train_classes]}"
        )

        if len(train_classes) < 2:

            print(
                "Skipping fold: "
                "training contains fewer than "
                "2 classes."
            )

            continue

        # =====================================================================
        # TRAIN LIGHTGBM
        # =====================================================================

        model = train_model(
            X_train,
            y_train.astype(int),
            X_test,
            y_test.astype(int)
        )

        # =====================================================================
        # PREDICTION
        # =====================================================================

        probabilities = model.predict_proba(
            X_test
        )

        # ---------------------------------------------------------------------
        # LightGBM may return fewer than 4 probability columns if a fold
        # contains fewer classes.
        #
        # Rebuild a 4-column probability matrix.
        # ---------------------------------------------------------------------

        full_probabilities = np.zeros(
            (len(X_test), 4),
            dtype=float
        )

        for j, cls in enumerate(
            model.classes_
        ):

            full_probabilities[
                :,
                int(cls)
            ] = probabilities[:, j]

        # ---------------------------------------------------------------------
        # Normalize
        # ---------------------------------------------------------------------

        row_sums = (
            full_probabilities.sum(axis=1)
        )

        valid_sum = row_sums > 0

        full_probabilities[
            valid_sum
        ] = (
            full_probabilities[valid_sum]
            /
            row_sums[valid_sum, None]
        )

        # =====================================================================
        # PREDICTED CLASS
        # =====================================================================

        predicted_ml_class = np.argmax(
            full_probabilities,
            axis=1
        )

        predicted_regime = (
            predicted_ml_class + 1
        )

        actual_regime = (
            y_test.astype(int).values + 1
        )

        # =====================================================================
        # METRICS
        # =====================================================================

        accuracy = accuracy_score(
            actual_regime,
            predicted_regime
        )

        precision = precision_score(
            actual_regime,
            predicted_regime,
            labels=[1, 2, 3, 4],
            average="macro",
            zero_division=0
        )

        recall = recall_score(
            actual_regime,
            predicted_regime,
            labels=[1, 2, 3, 4],
            average="macro",
            zero_division=0
        )

        f1 = f1_score(
            actual_regime,
            predicted_regime,
            labels=[1, 2, 3, 4],
            average="macro",
            zero_division=0
        )

        # ---------------------------------------------------------------------
        # Log loss
        #
        # Use all four classes.
        # ---------------------------------------------------------------------

        logloss = log_loss(
            actual_regime,
            full_probabilities,
            labels=[1, 2, 3, 4]
        )

        # =====================================================================
        # SAVE FOLD METRICS
        # =====================================================================

        fold_metrics.append({

            "symbol": symbol,

            "fold": fold,

            "train_start":
                df.iloc[
                    train_start
                ]["open_time"],

            "original_train_end":
                df.iloc[
                    train_end - 1
                ]["open_time"],

            "purged_train_end":
                df.iloc[
                    purged_train_end - 1
                ]["open_time"],

            "test_start":
                df.iloc[
                    test_start
                ]["open_time"],

            "test_end":
                df.iloc[
                    test_end - 1
                ]["open_time"],

            "train_rows":
                len(X_train),

            "test_rows":
                len(X_test),

            "accuracy":
                accuracy,

            "precision_macro":
                precision,

            "recall_macro":
                recall,

            "f1_macro":
                f1,

            "multiclass_log_loss":
                logloss
        })

        # =====================================================================
        # SAVE OOS PREDICTIONS
        # =====================================================================

        prediction_df = pd.DataFrame({

            "open_time":
                df.loc[
                    X_test.index,
                    "open_time"
                ].values,

            "symbol":
                symbol,

            "fold":
                fold,

            "actual_future_regime":
                actual_regime,

            "actual_future_regime_label":
                [
                    REGIME_LABELS[x]
                    for x in actual_regime
                ],

            "predicted_regime":
                predicted_regime,

            "predicted_regime_label":
                [
                    REGIME_LABELS[x]
                    for x in predicted_regime
                ],

            "prob_trending":
                full_probabilities[:, 0],

            "prob_mean_reversion":
                full_probabilities[:, 1],

            "prob_high_volatility":
                full_probabilities[:, 2],

            "prob_event":
                full_probabilities[:, 3],

            "prediction_confidence":
                np.max(
                    full_probabilities,
                    axis=1
                )
        })

        all_predictions.append(
            prediction_df
        )

        # =====================================================================
        # PRINT RESULTS
        # =====================================================================

        print(
            f"Accuracy          : {accuracy:.4f}"
        )

        print(
            f"Precision (macro) : {precision:.4f}"
        )

        print(
            f"Recall (macro)    : {recall:.4f}"
        )

        print(
            f"F1 (macro)        : {f1:.4f}"
        )

        print(
            f"Log-loss          : {logloss:.4f}"
        )

    # =========================================================================
    # COMBINE PREDICTIONS
    # =========================================================================

    if not all_predictions:

        raise RuntimeError(
            f"{symbol}: no Stage 7 predictions generated."
        )

    predictions_df = pd.concat(
        all_predictions,
        ignore_index=True
    )

    predictions_df = (
        predictions_df
        .sort_values("open_time")
        .reset_index(drop=True)
    )

    # =========================================================================
    # COMBINE METRICS
    # =========================================================================

    metrics_df = pd.DataFrame(
        fold_metrics
    )

    # =========================================================================
    # SAVE PREDICTIONS
    # =========================================================================

    prediction_path = (
        REGIME_DIR
        / f"{symbol}_07_predictions.csv"
    )

    predictions_df.to_csv(
        prediction_path,
        index=False
    )

    # =========================================================================
    # SAVE FOLD METRICS
    # =========================================================================

    metrics_path = (
        REGIME_DIR
        / f"{symbol}_07_classification_report.csv"
    )

    metrics_df.to_csv(
        metrics_path,
        index=False
    )

    # =========================================================================
    # OVERALL OOS PERFORMANCE
    # =========================================================================

    actual_all = (
        predictions_df[
            "actual_future_regime"
        ]
        .astype(int)
        .values
    )

    predicted_all = (
        predictions_df[
            "predicted_regime"
        ]
        .astype(int)
        .values
    )

    probabilities_all = (
        predictions_df[
            [
                "prob_trending",
                "prob_mean_reversion",
                "prob_high_volatility",
                "prob_event"
            ]
        ]
        .values
    )

    overall_accuracy = accuracy_score(
        actual_all,
        predicted_all
    )

    overall_precision = precision_score(
        actual_all,
        predicted_all,
        labels=[1, 2, 3, 4],
        average="macro",
        zero_division=0
    )

    overall_recall = recall_score(
        actual_all,
        predicted_all,
        labels=[1, 2, 3, 4],
        average="macro",
        zero_division=0
    )

    overall_f1 = f1_score(
        actual_all,
        predicted_all,
        labels=[1, 2, 3, 4],
        average="macro",
        zero_division=0
    )

    overall_logloss = log_loss(
        actual_all,
        probabilities_all,
        labels=[1, 2, 3, 4]
    )

    # =========================================================================
    # CONFUSION MATRIX
    # =========================================================================

    cm = confusion_matrix(
        actual_all,
        predicted_all,
        labels=[1, 2, 3, 4]
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            "Actual Trending",
            "Actual Mean Reversion",
            "Actual High Volatility",
            "Actual Event"
        ],
        columns=[
            "Pred Trending",
            "Pred Mean Reversion",
            "Pred High Volatility",
            "Pred Event"
        ]
    )

    confusion_path = (
        REGIME_DIR
        / f"{symbol}_07_confusion_matrix.csv"
    )

    cm_df.to_csv(
        confusion_path
    )

    # =========================================================================
    # OVERALL REPORT
    # =========================================================================

    overall_report = pd.DataFrame({

        "symbol": [symbol],

        "oos_rows":
            [len(predictions_df)],

        "accuracy":
            [overall_accuracy],

        "precision_macro":
            [overall_precision],

        "recall_macro":
            [overall_recall],

        "f1_macro":
            [overall_f1],

        "multiclass_log_loss":
            [overall_logloss]
    })

    overall_path = (
        REGIME_DIR
        / f"{symbol}_07_overall_metrics.csv"
    )

    overall_report.to_csv(
        overall_path,
        index=False
    )

    # =========================================================================
    # PRINT FINAL RESULTS
    # =========================================================================

    print("\n")
    print("=" * 80)
    print(
        f"FINAL STAGE 7 RESULTS — {symbol}"
    )
    print("=" * 80)

    print(
        f"OOS observations     : "
        f"{len(predictions_df):,}"
    )

    print(
        f"Accuracy              : "
        f"{overall_accuracy:.4f}"
    )

    print(
        f"Precision (macro)     : "
        f"{overall_precision:.4f}"
    )

    print(
        f"Recall (macro)        : "
        f"{overall_recall:.4f}"
    )

    print(
        f"F1 (macro)            : "
        f"{overall_f1:.4f}"
    )

    print(
        f"Multiclass log-loss   : "
        f"{overall_logloss:.4f}"
    )

    print("\nConfusion Matrix:")

    print(
        cm_df
    )

    print(
        f"\nSaved predictions:\n"
        f"{prediction_path}"
    )

    print(
        f"\nSaved fold metrics:\n"
        f"{metrics_path}"
    )

    print(
        f"\nSaved overall metrics:\n"
        f"{overall_path}"
    )

    print(
        f"\nSaved confusion matrix:\n"
        f"{confusion_path}"
    )

    return (
        predictions_df,
        metrics_df,
        overall_report,
        cm_df
    )


# =============================================================================
# RUN ALL SYMBOLS
# =============================================================================

if __name__ == "__main__":

    print("\n")
    print("=" * 80)
    print("STAGE 7 — LIGHTGBM UPCOMING REGIME CLASSIFIER")
    print("=" * 80)

    all_results = {}

    for symbol in SYMBOLS:

        try:

            try:
                result = process_symbol(symbol)
                all_results[symbol] = result
            except RuntimeError as e:
                if "no Stage 7 predictions generated" in str(e):
                    print(f"\nWARNING: {symbol} produced no valid Stage 7 predictions; skipping this symbol.")
                    print(e)
                    all_results[symbol] = None
                    continue
                raise
            except Exception as e:
                print(f"\nERROR processing {symbol}")
                print(e)
                raise

            all_results[symbol] = result

        except Exception as e:

            print("\n")
            print("=" * 80)

            print(
                f"ERROR processing {symbol}"
            )

            print(e)

            print("=" * 80)

            raise

    print("\n")
    print("=" * 80)
    print("STAGE 7 COMPLETED SUCCESSFULLY")
    print("=" * 80)



STAGE 7 — LIGHTGBM UPCOMING REGIME CLASSIFIER


STAGE 7 — BTCUSDT
Loaded Stage 6 dataset for BTCUSDT: 53,973 rows
BTCUSDT: Stage 6 validation passed.

Number of Stage 7 folds: 69

----------------------------------------------------------------------
Fold 0
Original train: 2020-07-03 04:00:00+00:00 → 2020-12-31 03:00:00+00:00
Test: 2020-12-31 04:00:00+00:00 → 2021-01-30 07:00:00+00:00
Training rows after purge: 4,316
Training classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Accuracy          : 0.9181
Precision (macro) : 0.6741
Recall (macro)    : 0.6976
F1 (macro)        : 0.6845
Log-loss          : 0.2557

----------------------------------------------------------------------
Fold 1
Original train: 2020-08-02 08:00:00+00:00 → 2021-01-30 07:00:00+00:00
Test: 2021-01-30 08:00:00+00:00 → 2021-03-01 11:00:00+00:00
Training rows after purge: 4,316
Training classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Accuracy          : 0.9542
Precision (macro) : 0.4804
Reca

In [ ]:
# ============================================================
# STAGE 7 - LATEST NEXT-4-BAR REGIME PREDICTIONS
# ============================================================
#
# Meaning:
# At the latest available prediction time t,
# predict the DOMINANT regime across:
#
#       t+1, t+2, t+3, t+4
#
# This gives ONE dominant regime for the next four hourly bars.
#
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# CONFIGURATION
# ============================================================

REGIME_DIR = Path(r"C:\Users\dell\Downloads\data\regimes")

# Automatically find all Stage 7 prediction files
prediction_files = sorted(REGIME_DIR.glob("*07_predictions.csv"))

# These commodities do not have enough futures data
EXCLUDE_SYMBOLS = {
    "XAUUSDT",
    "XAGUSDT"
}


# ============================================================
# REGIME LABEL MAPPING
# ============================================================

REGIME_NAMES = {
    1: "Trending",
    2: "Mean Reversion",
    3: "High Volatility",
    4: "Event"
}


# ============================================================
# HELPER FUNCTION
# ============================================================

def find_column(df, possible_names):

    """
    Find a column from several possible naming conventions.
    """

    lower_map = {
        str(col).strip().lower(): col
        for col in df.columns
    }

    for name in possible_names:

        key = name.strip().lower()

        if key in lower_map:
            return lower_map[key]

    return None


# ============================================================
# PROCESS EACH SYMBOL
# ============================================================

results = []

print("\n")
print("=" * 90)
print("LATEST NEXT-4-BAR DOMINANT REGIME PREDICTIONS")
print("=" * 90)

for file_path in prediction_files:

    # --------------------------------------------------------
    # Get symbol from filename
    # --------------------------------------------------------

    symbol = file_path.name.replace(
        " 07_predictions.csv", ""
    ).strip()

    # Skip gold/silver
    if symbol in EXCLUDE_SYMBOLS:
        continue

    print(f"\n[LOAD] {symbol}")

    try:

        df = pd.read_csv(file_path)

        if df.empty:
            print(f"[WARNING] Empty prediction file for {symbol}")
            continue

        # ----------------------------------------------------
        # Find timestamp column
        # ----------------------------------------------------

        time_col = find_column(
            df,
            [
                "open_time",
                "timestamp",
                "datetime",
                "date",
                "time"
            ]
        )

        # ----------------------------------------------------
        # Find predicted regime column
        # ----------------------------------------------------

        pred_col = find_column(
            df,
            [
                "predicted_regime",
                "prediction",
                "pred_regime",
                "y_pred"
            ]
        )

        # ----------------------------------------------------
        # Find probability/confidence column
        # ----------------------------------------------------

        confidence_col = find_column(
            df,
            [
                "confidence",
                "max_probability",
                "prediction_probability",
                "predicted_probability"
            ]
        )

        # ----------------------------------------------------
        # Check prediction column
        # ----------------------------------------------------

        if pred_col is None:

            print(
                f"[ERROR] Could not find predicted regime "
                f"column for {symbol}"
            )

            print("Available columns:")
            print(df.columns.tolist())

            continue

        # ----------------------------------------------------
        # Sort chronologically
        # ----------------------------------------------------

        if time_col is not None:

            df[time_col] = pd.to_datetime(
                df[time_col],
                errors="coerce"
            )

            df = df.sort_values(time_col)

        # ----------------------------------------------------
        # Remove rows with no prediction
        # ----------------------------------------------------

        df = df.dropna(
            subset=[pred_col]
        )

        if df.empty:

            print(
                f"[WARNING] No valid predictions for {symbol}"
            )

            continue

        # ----------------------------------------------------
        # Take the LATEST prediction
        # ----------------------------------------------------

        latest = df.iloc[-1]

        prediction = latest[pred_col]

        # ----------------------------------------------------
        # Convert numeric regime to readable name
        # ----------------------------------------------------

        if isinstance(prediction, str):

            prediction_text = prediction

            # Sometimes prediction is saved as "1.0"
            try:

                numeric_prediction = int(
                    float(prediction)
                )

                if numeric_prediction in REGIME_NAMES:

                    prediction_text = REGIME_NAMES[
                        numeric_prediction
                    ]

            except:

                pass

        else:

            try:

                numeric_prediction = int(
                    prediction
                )

                prediction_text = REGIME_NAMES.get(
                    numeric_prediction,
                    str(prediction)
                )

            except:

                prediction_text = str(prediction)

        # ----------------------------------------------------
        # Get confidence
        # ----------------------------------------------------

        confidence = np.nan

        if confidence_col is not None:

            try:

                confidence = float(
                    latest[confidence_col]
                )

            except:

                confidence = np.nan

        # ----------------------------------------------------
        # Prediction time
        # ----------------------------------------------------

        if time_col is not None:

            prediction_time = latest[time_col]

        else:

            prediction_time = "Unknown"

        # ----------------------------------------------------
        # Store result
        # ----------------------------------------------------

        results.append({

            "symbol": symbol,

            "prediction_time": prediction_time,

            "forecast_horizon": "t+1 to t+4",

            "predicted_dominant_regime":
                prediction_text,

            "confidence":
                confidence

        })

        print(
            f"  Prediction time : {prediction_time}"
        )

        print(
            f"  Next 4 bars     : t+1, t+2, t+3, t+4"
        )

        print(
            f"  Dominant regime : {prediction_text}"
        )

        if not pd.isna(confidence):

            print(
                f"  Confidence      : "
                f"{confidence:.4f}"
            )

    except Exception as e:

        print(
            f"[ERROR] Could not process "
            f"{symbol}: {e}"
        )


# ============================================================
# CREATE FINAL TABLE
# ============================================================

results_df = pd.DataFrame(results)


# ============================================================
# DISPLAY FINAL RESULT
# ============================================================

print("\n")
print("=" * 90)
print("FINAL NEXT-4-BAR REGIME FORECAST")
print("=" * 90)

if not results_df.empty:

    display(
        results_df
    )

else:

    print(
        "No valid Stage 7 predictions found."
    )


# ============================================================
# SAVE RESULT
# ============================================================

output_file = (
    REGIME_DIR /
    "ALL_SYMBOLS_NEXT_4_BAR_REGIME_FORECAST.csv"
)

results_df.to_csv(
    output_file,
    index=False
)

print("\nSaved final forecast:")
print(output_file)



LATEST NEXT-4-BAR DOMINANT REGIME PREDICTIONS

[LOAD] BNBUSDT_07_predictions.csv
  Prediction time : 2026-09-11 13:00:00
  Next 4 bars     : t+1, t+2, t+3, t+4
  Dominant regime : Event

[LOAD] BTCUSDT_07_predictions.csv
  Prediction time : 2026-09-11 12:00:00
  Next 4 bars     : t+1, t+2, t+3, t+4
  Dominant regime : High Volatility

[LOAD] ETHUSDT_07_predictions.csv
  Prediction time : 2026-09-11 13:00:00
  Next 4 bars     : t+1, t+2, t+3, t+4
  Dominant regime : Event


FINAL NEXT-4-BAR REGIME FORECAST


,symbol,prediction_time,forecast_horizon,predicted_dominant_regime,confidence
0,BNBUSDT_07_predictions.csv,2026-09-11 13:00:00,t+1 to t+4,Event,NaN
1,BTCUSDT_07_predictions.csv,2026-09-11 12:00:00,t+1 to t+4,High Volatility,NaN
2,ETHUSDT_07_predictions.csv,2026-09-11 13:00:00,t+1 to t+4,Event,NaN



Saved final forecast:
C:\Users\dell\Downloads\data\regimes\ALL_SYMBOLS_NEXT_4_BAR_REGIME_FORECAST.csv


In [5]:
# =============================================================================
# PRINT LATEST NEXT-4-BAR REGIME FORECAST PER SYMBOL
# =============================================================================
#
# Reads each symbol's Stage 7 prediction CSV, takes the most recent row
# (latest open_time == latest available prediction), and prints:
#
#   SYMBOL
#
#   Next 4-bar dominant regime:
#   <regime>
#
#   Probabilities:
#   Trending          87.0%
#   Mean Reversion     5.2%
#   High Volatility    6.1%
#   Event              1.7%
#
# Symbols with missing/empty prediction files (e.g. XAUUSDT, XAGUSDT due to
# insufficient data) are skipped automatically.
# =============================================================================

from pathlib import Path
import pandas as pd

# -----------------------------------------------------------------------------
# CONFIG — reuse the same paths/names as your Stage 7 cell
# -----------------------------------------------------------------------------

SYMBOLS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XAUUSDT", "XAGUSDT"]

PREDICTIONS_DIR = Path(r"C:\Users\dell\Downloads\data\predictions")

# Regime display order (matches REGIME_NAMES in Stage 7)
REGIME_NAMES = ["Trending", "Mean Reversion", "High Volatility", "Event"]

# Try these filename patterns in order (adjust/add if yours differs)
FILENAME_PATTERNS = [
    "{symbol}_07_lightgbm_predictions.csv",
    "{symbol}_07_predictions.csv",
]


def find_prediction_file(symbol: str) -> Path | None:
    for pattern in FILENAME_PATTERNS:
        candidate = PREDICTIONS_DIR / pattern.format(symbol=symbol)
        if candidate.exists():
            return candidate
    return None


def print_latest_forecast(symbol: str):
    path = find_prediction_file(symbol)

    if path is None:
        print(f"[SKIP] {symbol}: no prediction file found.\n")
        return

    df = pd.read_csv(path)

    if df.empty:
        print(f"[SKIP] {symbol}: prediction file is empty.\n")
        return

    # Make sure we grab the truly latest prediction
    if "open_time" in df.columns:
        df["open_time"] = pd.to_datetime(df["open_time"], utc=True, errors="coerce")
        df = df.sort_values("open_time")

    latest = df.iloc[-1]

    dominant_regime = latest.get("predicted_regime", "N/A")

    prob_cols = [f"prob_{name}" for name in REGIME_NAMES]
    missing_probs = [c for c in prob_cols if c not in df.columns]

    print(symbol)
    print()
    print("Next 4-bar dominant regime:")
    print(dominant_regime)
    print()
    print("Probabilities:")

    if missing_probs:
        print(f"  (missing probability columns: {missing_probs})")
    else:
        for name in REGIME_NAMES:
            pct = latest[f"prob_{name}"] * 100
            # left-justify label to 18 chars, right-justify percentage to 6 chars
            print(f"{name:<18}{pct:>5.1f}%")

    print()
    print("-" * 40)
    print()


if __name__ == "__main__":
    for symbol in SYMBOLS:
        print_latest_forecast(symbol)

BTCUSDT

Next 4-bar dominant regime:
N/A

Probabilities:
Trending           25.7%
Mean Reversion     18.4%
High Volatility    38.8%
Event              17.1%

----------------------------------------

ETHUSDT

Next 4-bar dominant regime:
N/A

Probabilities:
Trending           17.8%
Mean Reversion      0.8%
High Volatility    40.0%
Event              41.5%

----------------------------------------

BNBUSDT

Next 4-bar dominant regime:
N/A

Probabilities:
Trending           26.1%
Mean Reversion     14.1%
High Volatility    38.5%
Event              21.3%

----------------------------------------

XAUUSDT

Next 4-bar dominant regime:
N/A

Probabilities:
Trending            0.8%
Mean Reversion      0.6%
High Volatility    33.5%
Event              65.1%

----------------------------------------

XAGUSDT

Next 4-bar dominant regime:
N/A

Probabilities:
Trending           33.4%
Mean Reversion      0.2%
High Volatility    66.4%
Event               0.0%

----------------------------------------



In [13]:
from pathlib import Path
import pandas as pd

PREDICTIONS_DIR = Path(r"C:\Users\dell\Downloads\data\predictions")

for pattern in ["{symbol}_07_lightgbm_predictions.csv", "{symbol}_07_predictions.csv"]:
    path = PREDICTIONS_DIR / pattern.format(symbol="BTCUSDT")
    if path.exists():
        print(f"Found: {path}\n")
        df = pd.read_csv(path, nrows=3)
        print("Columns (one per line):")
        for col in df.columns:
            print(f"  - {col!r}")
        print()
        print("First row as dict:")
        print(df.iloc[0].to_dict())
        break
else:
    print("No BTCUSDT predictions file found under either filename pattern.")

Found: C:\Users\dell\Downloads\data\predictions\BTCUSDT_07_lightgbm_predictions.csv

Columns (one per line):
  - 'open_time'
  - 'symbol'
  - 'fold'
  - 'regime_label'
  - 'future_regime'
  - 'pred_future_regime_code'
  - 'pred_future_regime'
  - 'prob_Trending'
  - 'prob_Mean Reversion'
  - 'prob_High Volatility'
  - 'prob_Event'

First row as dict:
{'open_time': '2020-08-02 04:00:00+00:00', 'symbol': 'BTCUSDT', 'fold': 1, 'regime_label': 'Trending', 'future_regime': 'Event', 'pred_future_regime_code': 2, 'pred_future_regime': 'High Volatility', 'prob_Trending': 0.0184311882105862, 'prob_Mean Reversion': 0.0494556123712452, 'prob_High Volatility': 0.9018269096316562, 'prob_Event': 0.0302862897865121}


In [6]:
# =============================================================================
# STAGE 8 — WALK-FORWARD BACKTEST
#
# Turns the existing Stage 7 walk-forward OOS regime predictions into a
# trading equity curve and computes:
#
#   - Sharpe ratio
#   - Sortino ratio
#   - Max drawdown
#   - Win rate
#   - Total trades executed
#   - Net OOS return   (after fees + slippage)
#   - Gross return     (before fees + slippage)
#   - Buy & hold net return (benchmark, no fees)
#
# No separate walk-forward loop is needed here: Stage 5 (HMM) and Stage 7
# (LightGBM) already produce leakage-free, purged, walk-forward
# out-of-sample predictions. This script just evaluates them as a
# continuous OOS trading simulation.
#
# -----------------------------------------------------------------------
# TRADING RULE (edit REGIME_POSITION_MAP to change this)
# -----------------------------------------------------------------------
# A signal generated at time t (using only information available at t)
# is used to hold a position from t -> t+1 (i.e. it captures the next
# single realized bar return, not the full 4-bar horizon). This avoids
# overlapping-trade accounting complexity while still respecting the
# "no lookahead" requirement of your pipeline.
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

# =============================================================================
# CONFIGURATION
# =============================================================================

SYMBOLS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XAUUSDT", "XAGUSDT"]

REGIMES_DIR = Path(r"C:\Users\dell\Downloads\data\regimes")
PREDICTIONS_DIR = Path(r"C:\Users\dell\Downloads\data\predictions")
BACKTEST_DIR = Path(r"C:\Users\dell\Downloads\data\backtest")
BACKTEST_DIR.mkdir(parents=True, exist_ok=True)

# Costs (round-trip cost = 2x this, since it's applied on every position change)
FEE_RATE = 0.0004        # 0.04%
SLIPPAGE_RATE = 0.0002   # 0.02%
COST_RATE = FEE_RATE + SLIPPAGE_RATE   # 0.06% per unit of position change

# Bars per year for annualizing Sharpe/Sortino (1-hour bars, 24/7 crypto market)
BARS_PER_YEAR = 24 * 365

# Position taken for each predicted regime.
# Default: long only in Trending, flat everywhere else.
# Example with shorts: {"Trending": 1, "Mean Reversion": 0, "High Volatility": -1, "Event": 0}
REGIME_POSITION_MAP = {
    "Trending": 1,
    "Mean Reversion": 0,
    "High Volatility": 0,
    "Event": 0,
}

# Prediction filename patterns to try, in order
PREDICTION_FILENAME_PATTERNS = [
    "{symbol}_07_lightgbm_predictions.csv",
    "{symbol}_07_predictions.csv",
]

STAGE6_FILENAME = "{symbol}_06_ml_dataset.csv"

# -----------------------------------------------------------------------
# COLUMN NAME AUTO-DETECTION
# -----------------------------------------------------------------------
# Different versions of your Stage 7 script have saved predictions under
# slightly different column names. List every variant you've used here;
# the loader picks whichever one is actually present in the file.
# -----------------------------------------------------------------------

TIMESTAMP_COLUMN_CANDIDATES = [
    "open_time",
    "prediction_time",
    "timestamp",
    "time",
]

REGIME_COLUMN_CANDIDATES = [
    "pred_future_regime",
    "predicted_regime",
    "predicted_dominant_regime",
    "regime_prediction",
    "dominant_regime",
]

CONFIDENCE_COLUMN_CANDIDATES = [
    "prediction_confidence",
    "confidence",
]

# Optional: if present, this is the model's per-row max class probability
# and can be used as a confidence score even when no explicit confidence
# column exists (built from prob_<RegimeName> columns).
PROB_COLUMN_PREFIX = "prob_"


def resolve_column(df: pd.DataFrame, candidates: list, purpose: str, symbol: str) -> str:
    for name in candidates:
        if name in df.columns:
            return name
    columns_list = "\n  - " + "\n  - ".join(repr(c) for c in df.columns)
    raise KeyError(
        f"{symbol}: could not find a {purpose} column.\n"
        f"Tried: {candidates}\n"
        f"Actual columns in file:{columns_list}"
    )


# =============================================================================
# LOADERS
# =============================================================================

def find_predictions_file(symbol: str):
    for pattern in PREDICTION_FILENAME_PATTERNS:
        candidate = PREDICTIONS_DIR / pattern.format(symbol=symbol)
        if candidate.exists():
            return candidate
    return None


def load_predictions(symbol: str) -> pd.DataFrame:
    path = find_predictions_file(symbol)
    if path is None:
        raise FileNotFoundError(f"{symbol}: no Stage 7 predictions file found in {PREDICTIONS_DIR}")

    df = pd.read_csv(path)

    # Auto-detect the timestamp and predicted-regime columns, whatever
    # they're actually called in this file, and standardize their names
    # so the rest of the script doesn't need to know which variant it got.
    ts_col = resolve_column(df, TIMESTAMP_COLUMN_CANDIDATES, "timestamp", symbol)
    regime_col = resolve_column(df, REGIME_COLUMN_CANDIDATES, "predicted regime", symbol)

    df = df.rename(columns={ts_col: "open_time", regime_col: "predicted_regime"})

    # Confidence column is optional — only rename it if present.
    for cand in CONFIDENCE_COLUMN_CANDIDATES:
        if cand in df.columns and cand != "prediction_confidence":
            df = df.rename(columns={cand: "prediction_confidence"})
            break

    # If there's still no confidence column, derive it as the max class
    # probability across the prob_<RegimeName> columns (e.g. prob_Trending,
    # prob_Mean Reversion, prob_High Volatility, prob_Event).
    if "prediction_confidence" not in df.columns:
        prob_cols = [c for c in df.columns if c.startswith(PROB_COLUMN_PREFIX)]
        if prob_cols:
            df["prediction_confidence"] = df[prob_cols].max(axis=1)

    df["open_time"] = pd.to_datetime(df["open_time"], utc=True, errors="coerce")
    df = df.dropna(subset=["open_time"])
    df = df.sort_values("open_time").reset_index(drop=True)
    return df


def load_forward_returns(symbol: str) -> pd.DataFrame:
    """
    Loads Stage 6 dataset (has log_return for every bar, not just prediction
    rows) and builds the forward 1-bar simple return for each timestamp:

        fwd_return[t] = simple return realized from t -> t+1

    Using the full Stage 6 series (rather than the predictions subset) avoids
    gaps at fold boundaries breaking the shift(-1) calculation.
    """
    path = PREDICTIONS_DIR / STAGE6_FILENAME.format(symbol=symbol)
    if not path.exists():
        raise FileNotFoundError(f"{symbol}: Stage 6 dataset not found at {path}")

    df = pd.read_csv(path)

    if "log_return" not in df.columns:
        raise ValueError(f"{symbol}: 'log_return' column missing from Stage 6 dataset.")

    df["open_time"] = pd.to_datetime(df["open_time"], utc=True, errors="coerce")
    df = df.dropna(subset=["open_time"])
    df = df.sort_values("open_time").drop_duplicates("open_time").reset_index(drop=True)

    # log_return at row t is the return realized ENDING at t (i.e. from t-1 to t).
    # The forward return we need for a position opened at t is the return from
    # t -> t+1, which is exactly the log_return value stored at row t+1.
    df["fwd_return"] = np.expm1(df["log_return"].shift(-1))

    return df[["open_time", "fwd_return"]]


# =============================================================================
# METRICS
# =============================================================================

def compute_max_drawdown(equity_curve: pd.Series) -> float:
    running_max = equity_curve.cummax()
    drawdown = equity_curve / running_max - 1.0
    return drawdown.min()


def compute_sharpe(returns: pd.Series, bars_per_year: int) -> float:
    if returns.std(ddof=0) == 0 or returns.empty:
        return np.nan
    return (returns.mean() / returns.std(ddof=0)) * np.sqrt(bars_per_year)


def compute_sortino(returns: pd.Series, bars_per_year: int) -> float:
    downside = returns.clip(upper=0.0)
    downside_dev = np.sqrt((downside ** 2).mean())
    if downside_dev == 0 or returns.empty:
        return np.nan
    return (returns.mean() / downside_dev) * np.sqrt(bars_per_year)


def identify_trades(position: pd.Series, net_return: pd.Series) -> pd.DataFrame:
    """
    Groups contiguous nonzero-position bars into individual trades and
    computes the compounded net return of each trade.
    """
    trade_id = (position != position.shift(1)).cumsum()

    df = pd.DataFrame({
        "position": position.values,
        "net_return": net_return.values,
        "trade_id": trade_id.values,
    })

    active_trades = df[df["position"] != 0]

    trade_records = []
    for tid, group in active_trades.groupby("trade_id"):
        compounded_return = np.prod(1 + group["net_return"]) - 1
        trade_records.append({
            "trade_id": tid,
            "n_bars": len(group),
            "net_return": compounded_return,
        })

    return pd.DataFrame(trade_records)


# =============================================================================
# BACKTEST FOR ONE SYMBOL
# =============================================================================

def backtest_symbol(symbol: str):
    predictions = load_predictions(symbol)
    fwd_returns = load_forward_returns(symbol)

    merged = predictions.merge(fwd_returns, on="open_time", how="inner")
    merged = merged.dropna(subset=["fwd_return"])
    merged = merged.sort_values("open_time").reset_index(drop=True)

    if merged.empty:
        raise RuntimeError(f"{symbol}: no overlapping rows between predictions and Stage 6 returns.")

    # -------------------------------------------------------------------
    # POSITION FROM PREDICTED REGIME
    # -------------------------------------------------------------------
    merged["position"] = merged["predicted_regime"].map(REGIME_POSITION_MAP).fillna(0)

    # -------------------------------------------------------------------
    # GROSS RETURN (before costs)
    # -------------------------------------------------------------------
    merged["gross_return"] = merged["position"] * merged["fwd_return"]

    # -------------------------------------------------------------------
    # TRANSACTION COSTS
    # Charged whenever the position changes size (entry, exit, or flip).
    # -------------------------------------------------------------------
    position_change = merged["position"].diff().abs().fillna(merged["position"].abs())
    merged["cost"] = position_change * COST_RATE

    # -------------------------------------------------------------------
    # NET RETURN (after costs)
    # -------------------------------------------------------------------
    merged["net_return"] = merged["gross_return"] - merged["cost"]

    # -------------------------------------------------------------------
    # EQUITY CURVES
    # -------------------------------------------------------------------
    merged["gross_equity"] = (1 + merged["gross_return"]).cumprod()
    merged["net_equity"] = (1 + merged["net_return"]).cumprod()
    merged["buy_hold_equity"] = (1 + merged["fwd_return"]).cumprod()

    gross_return_total = merged["gross_equity"].iloc[-1] - 1
    net_return_total = merged["net_equity"].iloc[-1] - 1
    buy_hold_return_total = merged["buy_hold_equity"].iloc[-1] - 1

    # -------------------------------------------------------------------
    # RISK METRICS (computed on net returns)
    # -------------------------------------------------------------------
    sharpe = compute_sharpe(merged["net_return"], BARS_PER_YEAR)
    sortino = compute_sortino(merged["net_return"], BARS_PER_YEAR)
    max_dd = compute_max_drawdown(merged["net_equity"])

    # -------------------------------------------------------------------
    # TRADE-LEVEL STATS
    # -------------------------------------------------------------------
    trades_df = identify_trades(merged["position"], merged["net_return"])

    total_trades = len(trades_df)
    if total_trades > 0:
        win_rate = (trades_df["net_return"] > 0).mean()
    else:
        win_rate = np.nan

    result = {
        "symbol": symbol,
        "bars_tested": len(merged),
        "total_trades": total_trades,
        "win_rate": win_rate,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino,
        "max_drawdown": max_dd,
        "gross_return": gross_return_total,
        "net_return": net_return_total,
        "buy_hold_return": buy_hold_return_total,
    }

    return result, merged


# =============================================================================
# PRINT SUMMARY
# =============================================================================

def print_backtest_summary(result: dict):
    print("=" * 70)
    print(f"WALK-FORWARD BACKTEST — {result['symbol']}")
    print("=" * 70)
    print(f"Bars tested (OOS)         : {result['bars_tested']:,}")
    print(f"Total trades executed     : {result['total_trades']:,}")
    if not np.isnan(result['win_rate']):
        print(f"Win rate                  : {result['win_rate'] * 100:.2f}%")
    else:
        print(f"Win rate                  : N/A (no trades)")
    print(f"Sharpe ratio (annualized) : {result['sharpe_ratio']:.3f}")
    print(f"Sortino ratio (annualized): {result['sortino_ratio']:.3f}")
    print(f"Max drawdown              : {result['max_drawdown'] * 100:.2f}%")
    print(f"Gross return (no costs)   : {result['gross_return'] * 100:.2f}%")
    print(f"Net OOS return (w/ costs) : {result['net_return'] * 100:.2f}%")
    print(f"Buy & hold net return     : {result['buy_hold_return'] * 100:.2f}%")
    print()


# =============================================================================
# RUN ALL SYMBOLS
# =============================================================================

if __name__ == "__main__":

    all_results = []

    for symbol in SYMBOLS:
        try:
            result, merged = backtest_symbol(symbol)
            print_backtest_summary(result)
            all_results.append(result)

            # Save per-symbol bar-level backtest detail
            detail_path = BACKTEST_DIR / f"{symbol}_08_backtest_detail.csv"
            merged.to_csv(detail_path, index=False)

        except FileNotFoundError as e:
            print(f"[SKIP] {symbol}: {e}\n")
            continue
        except KeyError as e:
            print(f"[ERROR] {symbol}: column lookup failed — {e}\n")
            continue
        except Exception as e:
            print(f"[ERROR] {symbol}: {type(e).__name__}: {e}\n")
            continue

    if all_results:
        summary_df = pd.DataFrame(all_results)
        summary_path = BACKTEST_DIR / "STAGE_8_ALL_SYMBOLS_BACKTEST_SUMMARY.csv"
        summary_df.to_csv(summary_path, index=False)

        print("=" * 70)
        print("BACKTEST SUMMARY — ALL SYMBOLS")
        print("=" * 70)
        print(summary_df.to_string(index=False))
        print(f"\nSaved combined summary to:\n{summary_path}")
    else:
        print("No symbols were successfully backtested.")

WALK-FORWARD BACKTEST — BTCUSDT
Bars tested (OOS)         : 53,552
Total trades executed     : 2,850
Win rate                  : 46.98%
Sharpe ratio (annualized) : -1.657
Sortino ratio (annualized): -2.308
Max drawdown              : -96.64%
Gross return (no costs)   : 34.28%
Net OOS return (w/ costs) : -95.61%
Buy & hold net return     : 597.30%

WALK-FORWARD BACKTEST — ETHUSDT
Bars tested (OOS)         : 53,553
Total trades executed     : 2,495
Win rate                  : 44.21%
Sharpe ratio (annualized) : -1.058
Sortino ratio (annualized): -1.499
Max drawdown              : -96.09%
Gross return (no costs)   : 46.25%
Net OOS return (w/ costs) : -92.68%
Buy & hold net return     : 603.50%

WALK-FORWARD BACKTEST — BNBUSDT
Bars tested (OOS)         : 52,585
Total trades executed     : 2,404
Win rate                  : 49.25%
Sharpe ratio (annualized) : -0.987
Sortino ratio (annualized): -1.336
Max drawdown              : -94.67%
Gross return (no costs)   : 75.82%
Net OOS return (w/ cost

In [ ]:
"""
===============================================================================
STAGE 8
Walk-Forward Trading Strategy + Risk Management + PnL
===============================================================================

INPUTS
------
1. Stage 7 LightGBM predictions:

   data/predictions/{SYMBOL}_07_lightgbm_predictions.csv

2. Stage 2/feature-engineered OHLCV files:

   data/processed/{SYMBOL}_1h_features.csv


IMPORTANT LEAKAGE RULE
----------------------
At hour t:

    Stage 7 prediction at t
              |
              v
       strategy signal
              |
              v
       execute at t+1 OPEN

Therefore information from candle t is never used to obtain a fill
before candle t closes.

The strategy does NOT use:
    future_regime
    realized future returns
    future prices
    future highs/lows
    future labels

Those columns may exist in Stage 7 for evaluation, but they are NEVER
used to generate trades.

===============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

SYMBOLS = [
    "BTCUSDT",
    "ETHUSDT",
    "BNBUSDT",
    "XAUUSDT",
    "XAGUSDT",
]


# -------------------------------------------------------------------------
# DATA LOCATION
# -------------------------------------------------------------------------

BASE_DIR = Path(r"C:\Users\dell\Downloads\data")

PREDICTION_DIR = BASE_DIR / "predictions"
FEATURE_DIR = BASE_DIR / "processed"

OUTPUT_DIR = BASE_DIR / "backtest" / "stage8"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. STRATEGY PARAMETERS
# =============================================================================

# -------------------------------------------------------------------------
# INITIAL CAPITAL
# -------------------------------------------------------------------------

INITIAL_EQUITY = 100_000.0


# -------------------------------------------------------------------------
# RISK MANAGEMENT
# -------------------------------------------------------------------------

RISK_PER_TRADE = 0.005       # 0.5% of current equity

STOP_ATR_MULTIPLIER = 1.5

TAKE_PROFIT_ATR_MULTIPLIER = 3.0


# -------------------------------------------------------------------------
# MAXIMUM NOTIONAL
# -------------------------------------------------------------------------
#
# Prevents ATR-based position sizing from producing an excessively large
# position when ATR becomes very small.
#
# 1.0 means maximum notional = 100% of equity.
#
# This is a risk cap, not leverage.
# -------------------------------------------------------------------------

MAX_NOTIONAL_MULTIPLE = 1.0


# -------------------------------------------------------------------------
# PREDICTION CONFIDENCE
# -------------------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.60


# -------------------------------------------------------------------------
# TREND FILTER
# -------------------------------------------------------------------------

ADX_THRESHOLD = 25.0


# -------------------------------------------------------------------------
# HOLDING PERIOD
# -------------------------------------------------------------------------

FORECAST_HORIZON_BARS = 4


# -------------------------------------------------------------------------
# TRANSACTION COSTS
# -------------------------------------------------------------------------

FEE_RATE = 0.0004          # 0.04%

SLIPPAGE_RATE = 0.0002     # 0.02%

# If you later want a harsher stop-execution assumption,
# change this separately.
#
# For now we use the same 0.02%.
SLIPPAGE_STOP = SLIPPAGE_RATE


ENTRY_COST_RATE = (
    FEE_RATE +
    SLIPPAGE_RATE
)


# =============================================================================
# 3. REGIME DEFINITIONS
# =============================================================================

REGIME_LABELS = {
    1: "Trending",
    2: "Mean Reversion",
    3: "High Volatility",
    4: "Event",
}


# =============================================================================
# 4. COLUMN COMPATIBILITY
# =============================================================================

"""
Your actual feature files contain names such as:

    momentum_5
    momentum_20
    momentum_60
    log_realized_vol_20
    log_atr_pct

while the earlier Stage 5 code normalized them internally to:

    momentum_5h
    momentum_20h
    momentum_60h
    realized_vol_20
    atr_pct

Stage 8 therefore handles both versions.
"""


FEATURE_ALIASES = {
    "momentum_5": "momentum_5h",
    "momentum_20": "momentum_20h",
    "momentum_60": "momentum_60h",

    "log_realized_vol_20": "realized_vol_20",
    "log_atr_pct": "atr_pct",
}


# =============================================================================
# 5. LOAD STAGE 7
# =============================================================================

def load_stage7(symbol):

    path = (
        PREDICTION_DIR /
        f"{symbol}_07_lightgbm_predictions.csv"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"\nStage 7 prediction file not found:\n{path}"
        )

    df = pd.read_csv(path)

    if "open_time" not in df.columns:

        raise ValueError(
            f"{symbol}: Stage 7 file does not contain open_time."
        )

    df["open_time"] = pd.to_datetime(
        df["open_time"],
        utc=True
    )

    df = (
        df
        .sort_values("open_time")
        .drop_duplicates("open_time")
        .reset_index(drop=True)
    )

    return df


# =============================================================================
# 6. LOAD FEATURES
# =============================================================================

def load_features(symbol):

    path = (
        FEATURE_DIR /
        f"{symbol}_1h_features.csv"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"\nFeature file not found:\n{path}"
        )

    df = pd.read_csv(path)

    if "open_time" not in df.columns:

        raise ValueError(
            f"{symbol}: Feature file does not contain open_time."
        )

    df["open_time"] = pd.to_datetime(
        df["open_time"],
        utc=True
    )

    df = (
        df
        .sort_values("open_time")
        .drop_duplicates("open_time")
        .reset_index(drop=True)
    )

    # Normalize feature names
    df = df.rename(
        columns=FEATURE_ALIASES
    )

    return df


# =============================================================================
# 7. CHECK STAGE 7 COLUMNS
# =============================================================================

def validate_stage7_columns(df, symbol):

    required = [
        "open_time",
        "pred_future_regime_code",
        "pred_future_regime",

        "prob_Trending",
        "prob_Mean Reversion",
        "prob_High Volatility",
        "prob_Event",
    ]

    missing = [
        col
        for col in required
        if col not in df.columns
    ]

    if missing:

        raise ValueError(
            f"{symbol}: Missing Stage 7 columns:\n{missing}"
        )


# =============================================================================
# 8. CALCULATE ATR14
# =============================================================================

def calculate_atr(df, period=14):

    high = df["high"]
    low = df["low"]
    close = df["close"]

    previous_close = close.shift(1)

    tr1 = high - low

    tr2 = (
        high -
        previous_close
    ).abs()

    tr3 = (
        low -
        previous_close
    ).abs()

    true_range = pd.concat(
        [
            tr1,
            tr2,
            tr3
        ],
        axis=1
    ).max(axis=1)

    atr = (
        true_range
        .rolling(
            period,
            min_periods=period
        )
        .mean()
    )

    return atr


# =============================================================================
# 9. PREPARE DATA
# =============================================================================

def prepare_data(symbol):

    print("\n" + "=" * 90)
    print(f"PREPARING {symbol}")
    print("=" * 90)

    stage7 = load_stage7(symbol)

    features = load_features(symbol)

    validate_stage7_columns(
        stage7,
        symbol
    )

    print(
        f"Stage 7 rows : {len(stage7):,}"
    )

    print(
        f"Feature rows : {len(features):,}"
    )

    # ---------------------------------------------------------------------
    # MERGE ONLY INFORMATION AVAILABLE AT SAME TIMESTAMP
    # ---------------------------------------------------------------------

    df = pd.merge(
        stage7,
        features,
        on="open_time",
        how="inner",
        suffixes=(
            "",
            "_feature"
        )
    )

    # ---------------------------------------------------------------------
    # Sort
    # ---------------------------------------------------------------------

    df = (
        df
        .sort_values("open_time")
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------------------
    # OHLC VALIDATION
    # ---------------------------------------------------------------------

    required_ohlc = [
        "open",
        "high",
        "low",
        "close",
    ]

    missing_ohlc = [
        col
        for col in required_ohlc
        if col not in df.columns
    ]

    if missing_ohlc:

        raise ValueError(
            f"{symbol}: Missing OHLC columns: "
            f"{missing_ohlc}"
        )

    # ---------------------------------------------------------------------
    # ATR14
    # ---------------------------------------------------------------------

    df["atr_14"] = calculate_atr(
        df,
        period=14
    )

    # ---------------------------------------------------------------------
    # EMA20
    #
    # EMA uses only current and historical close.
    # No future data is used.
    # ---------------------------------------------------------------------

    df["ema_20"] = (
        df["close"]
        .ewm(
            span=20,
            adjust=False,
            min_periods=20
        )
        .mean()
    )

    # ---------------------------------------------------------------------
    # PRICE STRETCH
    # ---------------------------------------------------------------------

    df["ema_distance"] = (
        df["close"] -
        df["ema_20"]
    )

    # ---------------------------------------------------------------------
    # DERIVE CONFIDENCE FROM THE PROBABILITY OF THE PREDICTED REGIME
    #
    # We DO NOT depend on a prediction_confidence column because your
    # actual Stage 7 output contains the four probability columns.
    # ---------------------------------------------------------------------

    probability_columns = {
        "Trending":
            "prob_Trending",

        "Mean Reversion":
            "prob_Mean Reversion",

        "High Volatility":
            "prob_High Volatility",

        "Event":
            "prob_Event",
    }

    df["prediction_confidence"] = np.nan

    for regime, probability_col in probability_columns.items():

        mask = (
            df["pred_future_regime"]
            .astype(str)
            .eq(regime)
        )

        df.loc[
            mask,
            "prediction_confidence"
        ] = pd.to_numeric(
            df.loc[
                mask,
                probability_col
            ],
            errors="coerce"
        )

    # ---------------------------------------------------------------------
    # FALLBACK:
    # If pred_future_regime_code exists but label is missing, map it.
    # ---------------------------------------------------------------------

    code_to_label = {
        1: "Trending",
        2: "Mean Reversion",
        3: "High Volatility",
        4: "Event",
    }

    missing_label = (
        df["pred_future_regime"]
        .isna()
    )

    if missing_label.any():

        df.loc[
            missing_label,
            "pred_future_regime"
        ] = (
            pd.to_numeric(
                df.loc[
                    missing_label,
                    "pred_future_regime_code"
                ],
                errors="coerce"
            )
            .map(code_to_label)
        )

    # ---------------------------------------------------------------------
    # RE-CALCULATE CONFIDENCE AFTER LABEL FALLBACK
    # ---------------------------------------------------------------------

    for regime, probability_col in probability_columns.items():

        mask = (
            df["pred_future_regime"]
            .astype(str)
            .eq(regime)
            &
            df["prediction_confidence"].isna()
        )

        df.loc[
            mask,
            "prediction_confidence"
        ] = pd.to_numeric(
            df.loc[
                mask,
                probability_col
            ],
            errors="coerce"
        )

    return df


# =============================================================================
# 10. DIRECTIONAL STRATEGY
# =============================================================================

def generate_signal(row):

    regime = row["pred_future_regime"]

    confidence = row["prediction_confidence"]

    # ---------------------------------------------------------------------
    # Confidence gate
    # ---------------------------------------------------------------------

    if pd.isna(confidence):

        return 0

    if confidence < CONFIDENCE_THRESHOLD:

        return 0

    # ---------------------------------------------------------------------
    # Required features
    # ---------------------------------------------------------------------

    momentum = row.get(
        "momentum_20h",
        np.nan
    )

    adx = row.get(
        "adx_14",
        np.nan
    )

    close = row["close"]

    ema20 = row["ema_20"]

    atr = row["atr_14"]

    if any(
        pd.isna(x)
        for x in [
            momentum,
            adx,
            close,
            ema20,
            atr
        ]
    ):

        return 0

    if atr <= 0:

        return 0

    # =====================================================================
    # TRENDING
    # =====================================================================

    if regime == "Trending":

        # LONG
        if (
            momentum > 0
            and
            adx >= ADX_THRESHOLD
        ):

            return 1

        # SHORT
        if (
            momentum < 0
            and
            adx >= ADX_THRESHOLD
        ):

            return -1

        return 0

    # =====================================================================
    # MEAN REVERSION
    # =====================================================================

    elif regime == "Mean Reversion":

        # Price significantly above EMA20
        # => fade upward move
        if close > (
            ema20 + atr
        ):

            return -1

        # Price significantly below EMA20
        # => fade downward move
        if close < (
            ema20 - atr
        ):

            return 1

        return 0

    # =====================================================================
    # HIGH VOLATILITY
    # =====================================================================

    elif regime == "High Volatility":

        # We do NOT automatically trade high volatility.
        #
        # The regime itself does not provide a direction.
        # Risk/reward can deteriorate because of large intrabar movement.

        return 0

    # =====================================================================
    # EVENT
    # =====================================================================

    elif regime == "Event":

        # No new positions during Event regime.
        return 0

    return 0


# =============================================================================
# 11. EXECUTION COSTS
# =============================================================================

def calculate_trade_pnl(
    direction,
    entry_price,
    exit_price,
    quantity,
    stopped_out
):

    # ---------------------------------------------------------------------
    # Gross return
    # ---------------------------------------------------------------------

    gross_return = (
        direction *
        (
            exit_price /
            entry_price
            - 1.0
        )
    )

    # ---------------------------------------------------------------------
    # Entry costs
    # ---------------------------------------------------------------------

    entry_cost = (
        FEE_RATE +
        SLIPPAGE_RATE
    )

    # ---------------------------------------------------------------------
    # Exit costs
    # ---------------------------------------------------------------------

    if stopped_out:

        exit_cost = (
            FEE_RATE +
            SLIPPAGE_STOP
        )

    else:

        exit_cost = (
            FEE_RATE +
            SLIPPAGE_RATE
        )

    # ---------------------------------------------------------------------
    # Net return
    # ---------------------------------------------------------------------

    net_return = (
        gross_return
        -
        entry_cost
        -
        exit_cost
    )

    # ---------------------------------------------------------------------
    # Notional
    # ---------------------------------------------------------------------

    entry_notional = (
        abs(quantity) *
        entry_price
    )

    # ---------------------------------------------------------------------
    # Dollar PnL
    # ---------------------------------------------------------------------

    gross_pnl = (
        entry_notional *
        gross_return
    )

    entry_cost_dollars = (
        entry_notional *
        entry_cost
    )

    exit_notional = (
        abs(quantity) *
        exit_price
    )

    exit_cost_dollars = (
        exit_notional *
        exit_cost
    )

    net_pnl = (
        gross_pnl
        -
        entry_cost_dollars
        -
        exit_cost_dollars
    )

    return {
        "gross_return": gross_return,
        "entry_cost": entry_cost,
        "exit_cost": exit_cost,
        "net_return": net_return,
        "gross_pnl": gross_pnl,
        "entry_cost_dollars": entry_cost_dollars,
        "exit_cost_dollars": exit_cost_dollars,
        "net_pnl": net_pnl,
    }


# =============================================================================
# 12. WALK-FORWARD BACKTEST
# =============================================================================

def run_backtest(
    df,
    symbol
):

    trades = []

    equity = INITIAL_EQUITY

    equity_curve = []

    position = None

    n = len(df)

    i = 0

    # =====================================================================
    # MAIN LOOP
    # =====================================================================

    while i < n:

        row = df.iloc[i]

        current_time = row["open_time"]

        # =================================================================
        # NO POSITION
        # =================================================================

        if position is None:

            # -------------------------------------------------------------
            # Signal is generated from CLOSED candle i.
            #
            # Execution occurs at candle i+1 OPEN.
            # -------------------------------------------------------------

            signal = generate_signal(
                row
            )

            if (
                signal != 0
                and
                i + 1 < n
            ):

                entry_row = df.iloc[
                    i + 1
                ]

                entry_price = float(
                    entry_row["open"]
                )

                atr = float(
                    row["atr_14"]
                )

                if (
                    not np.isfinite(atr)
                    or
                    atr <= 0
                ):

                    i += 1
                    continue

                # ---------------------------------------------------------
                # STOP DISTANCE
                # ---------------------------------------------------------

                stop_distance = (
                    STOP_ATR_MULTIPLIER *
                    atr
                )

                # ---------------------------------------------------------
                # RISK CAPITAL
                # ---------------------------------------------------------

                risk_per_trade = (
                    equity *
                    RISK_PER_TRADE
                )

                # ---------------------------------------------------------
                # POSITION SIZE
                # ---------------------------------------------------------

                quantity = (
                    risk_per_trade /
                    stop_distance
                )

                # ---------------------------------------------------------
                # MAX NOTIONAL CAP
                # ---------------------------------------------------------

                max_notional = (
                    equity *
                    MAX_NOTIONAL_MULTIPLE
                )

                actual_notional = (
                    quantity *
                    entry_price
                )

                if actual_notional > max_notional:

                    quantity = (
                        max_notional /
                        entry_price
                    )

                # ---------------------------------------------------------
                # Do not trade if quantity is invalid
                # ---------------------------------------------------------

                if (
                    not np.isfinite(quantity)
                    or
                    quantity <= 0
                ):

                    i += 1
                    continue

                # ---------------------------------------------------------
                # SL / TP
                # ---------------------------------------------------------

                if signal == 1:

                    stop_price = (
                        entry_price -
                        STOP_ATR_MULTIPLIER * atr
                    )

                    take_profit = (
                        entry_price +
                        TAKE_PROFIT_ATR_MULTIPLIER * atr
                    )

                else:

                    stop_price = (
                        entry_price +
                        STOP_ATR_MULTIPLIER * atr
                    )

                    take_profit = (
                        entry_price -
                        TAKE_PROFIT_ATR_MULTIPLIER * atr
                    )

                # ---------------------------------------------------------
                # Position
                # ---------------------------------------------------------

                position = {
                    "direction": signal,
                    "entry_index": i + 1,
                    "signal_index": i,

                    "entry_time":
                        entry_row["open_time"],

                    "entry_price":
                        entry_price,

                    "quantity":
                        quantity,

                    "atr_at_entry":
                        atr,

                    "stop_price":
                        stop_price,

                    "take_profit":
                        take_profit,

                    "predicted_regime":
                        row["pred_future_regime"],

                    "prediction_confidence":
                        row["prediction_confidence"],

                    "entry_equity":
                        equity,
                }

                # Move to entry candle
                i += 1
                continue

        # =================================================================
        # MANAGE EXISTING POSITION
        # =================================================================

        if position is not None:

            direction = position[
                "direction"
            ]

            entry_price = position[
                "entry_price"
            ]

            quantity = position[
                "quantity"
            ]

            stop_price = position[
                "stop_price"
            ]

            take_profit = position[
                "take_profit"
            ]

            entry_index = position[
                "entry_index"
            ]

            holding_bars = (
                i -
                entry_index +
                1
            )

            current_high = float(
                row["high"]
            )

            current_low = float(
                row["low"]
            )

            exit_reason = None
            exit_price = None
            stopped_out = False

            # =============================================================
            # 1. STOP LOSS
            # =============================================================

            if direction == 1:

                if current_low <= stop_price:

                    exit_reason = "stop_loss"

                    exit_price = stop_price

                    stopped_out = True

            else:

                if current_high >= stop_price:

                    exit_reason = "stop_loss"

                    exit_price = stop_price

                    stopped_out = True

            # =============================================================
            # 2. TAKE PROFIT
            # =============================================================
            #
            # Stop is checked BEFORE TP if both occur in the same hourly
            # candle. This is deliberately conservative because OHLC data
            # cannot tell us which happened first.
            # =============================================================

            if exit_reason is None:

                if direction == 1:

                    if current_high >= take_profit:

                        exit_reason = "take_profit"

                        exit_price = take_profit

                else:

                    if current_low <= take_profit:

                        exit_reason = "take_profit"

                        exit_price = take_profit

            # =============================================================
            # 3. REGIME / DIRECTION FLIP
            # =============================================================
            #
            # Signal is evaluated at the CURRENT candle close.
            #
            # Therefore execution happens at NEXT candle OPEN.
            # =============================================================

            if (
                exit_reason is None
                and
                i + 1 < n
            ):

                current_signal = generate_signal(
                    row
                )

                if (
                    current_signal != 0
                    and
                    current_signal != direction
                ):

                    next_row = df.iloc[
                        i + 1
                    ]

                    exit_reason = (
                        "regime_or_direction_flip"
                    )

                    exit_price = float(
                        next_row["open"]
                    )

            # =============================================================
            # 4. TIME EXIT
            # =============================================================

            if (
                exit_reason is None
                and
                holding_bars >= FORECAST_HORIZON_BARS
            ):

                exit_reason = "time_exit"

                exit_price = float(
                    row["close"]
                )

            # =============================================================
            # CLOSE POSITION
            # =============================================================

            if exit_reason is not None:

                pnl = calculate_trade_pnl(
                    direction=direction,
                    entry_price=entry_price,
                    exit_price=exit_price,
                    quantity=quantity,
                    stopped_out=stopped_out
                )

                equity_before = equity

                equity += pnl[
                    "net_pnl"
                ]

                # ---------------------------------------------------------
                # Trade record
                # ---------------------------------------------------------

                trade = {
                    "symbol":
                        symbol,

                    "signal_time":
                        position["entry_time"],

                    "entry_time":
                        position["entry_time"],

                    "exit_time":
                        row["open_time"],

                    "direction":
                        "LONG"
                        if direction == 1
                        else "SHORT",

                    "direction_code":
                        direction,

                    "predicted_regime":
                        position[
                            "predicted_regime"
                        ],

                    "prediction_confidence":
                        position[
                            "prediction_confidence"
                        ],

                    "entry_price":
                        entry_price,

                    "exit_price":
                        exit_price,

                    "quantity":
                        quantity,

                    "entry_notional":
                        entry_price *
                        quantity,

                    "stop_price":
                        stop_price,

                    "take_profit":
                        take_profit,

                    "atr_at_entry":
                        position[
                            "atr_at_entry"
                        ],

                    "risk_capital":
                        position[
                            "entry_equity"
                        ] *
                        RISK_PER_TRADE,

                    "holding_bars":
                        holding_bars,

                    "exit_reason":
                        exit_reason,

                    "stopped_out":
                        stopped_out,

                    "gross_return":
                        pnl[
                            "gross_return"
                        ],

                    "entry_cost":
                        pnl[
                            "entry_cost"
                        ],

                    "exit_cost":
                        pnl[
                            "exit_cost"
                        ],

                    "net_return":
                        pnl[
                            "net_return"
                        ],

                    "gross_pnl":
                        pnl[
                            "gross_pnl"
                        ],

                    "entry_cost_dollars":
                        pnl[
                            "entry_cost_dollars"
                        ],

                    "exit_cost_dollars":
                        pnl[
                            "exit_cost_dollars"
                        ],

                    "net_pnl":
                        pnl[
                            "net_pnl"
                        ],

                    "equity_before":
                        equity_before,

                    "equity_after":
                        equity,
                }

                trades.append(
                    trade
                )

                position = None

        # =================================================================
        # MARK-TO-MARKET EQUITY
        # =================================================================

        mtm_equity = equity

        if position is not None:

            current_close = float(
                row["close"]
            )

            unrealized_return = (
                position["direction"]
                *
                (
                    current_close /
                    position["entry_price"]
                    - 1
                )
            )

            notional = (
                position["entry_price"]
                *
                position["quantity"]
            )

            unrealized_pnl = (
                notional *
                unrealized_return
            )

            mtm_equity += (
                unrealized_pnl
            )

        equity_curve.append(
            {
                "symbol":
                    symbol,

                "open_time":
                    current_time,

                "equity":
                    mtm_equity,

                "realized_equity":
                    equity,

                "position":
                    0
                    if position is None
                    else position[
                        "direction"
                    ],
            }
        )

        i += 1

    # =====================================================================
    # FORCE CLOSE ANY OPEN POSITION AT FINAL CLOSE
    # =====================================================================

    if position is not None:

        final_row = df.iloc[-1]

        exit_price = float(
            final_row["close"]
        )

        pnl = calculate_trade_pnl(
            direction=position[
                "direction"
            ],
            entry_price=position[
                "entry_price"
            ],
            exit_price=exit_price,
            quantity=position[
                "quantity"
            ],
            stopped_out=False
        )

        equity_before = equity

        equity += pnl[
            "net_pnl"
        ]

        trades.append(
            {
                "symbol":
                    symbol,

                "signal_time":
                    position["entry_time"],

                "entry_time":
                    position["entry_time"],

                "exit_time":
                    final_row["open_time"],

                "direction":
                    "LONG"
                    if position["direction"] == 1
                    else "SHORT",

                "direction_code":
                    position["direction"],

                "predicted_regime":
                    position[
                        "predicted_regime"
                    ],

                "prediction_confidence":
                    position[
                        "prediction_confidence"
                    ],

                "entry_price":
                    position[
                        "entry_price"
                    ],

                "exit_price":
                    exit_price,

                "quantity":
                    position[
                        "quantity"
                    ],

                "entry_notional":
                    position[
                        "entry_price"
                    ] *
                    position[
                        "quantity"
                    ],

                "stop_price":
                    position[
                        "stop_price"
                    ],

                "take_profit":
                    position[
                        "take_profit"
                    ],

                "atr_at_entry":
                    position[
                        "atr_at_entry"
                    ],

                "risk_capital":
                    position[
                        "entry_equity"
                    ] *
                    RISK_PER_TRADE,

                "holding_bars":
                    len(df)
                    -
                    position[
                        "entry_index"
                    ],

                "exit_reason":
                    "end_of_data",

                "stopped_out":
                    False,

                "gross_return":
                    pnl[
                        "gross_return"
                    ],

                "entry_cost":
                    pnl[
                        "entry_cost"
                    ],

                "exit_cost":
                    pnl[
                        "exit_cost"
                    ],

                "net_return":
                    pnl[
                        "net_return"
                    ],

                "gross_pnl":
                    pnl[
                        "gross_pnl"
                    ],

                "entry_cost_dollars":
                    pnl[
                        "entry_cost_dollars"
                    ],

                "exit_cost_dollars":
                    pnl[
                        "exit_cost_dollars"
                    ],

                "net_pnl":
                    pnl[
                        "net_pnl"
                    ],

                "equity_before":
                    equity_before,

                "equity_after":
                    equity,
            }
        )

    trades_df = pd.DataFrame(
        trades
    )

    equity_df = pd.DataFrame(
        equity_curve
    )

    return (
        trades_df,
        equity_df,
        equity
    )


# =============================================================================
# 13. PERFORMANCE METRICS
# =============================================================================

def calculate_metrics(
    trades,
    equity_curve,
    df,
    symbol
):

    if trades.empty:

        return {
            "symbol":
                symbol,

            "bars_tested":
                len(df),

            "total_trades":
                0,

            "win_rate":
                np.nan,

            "profit_factor":
                np.nan,

            "sharpe_ratio":
                np.nan,

            "sortino_ratio":
                np.nan,

            "max_drawdown":
                0.0,

            "gross_return":
                0.0,

            "net_return":
                0.0,

            "total_net_pnl":
                0.0,

            "avg_trade_pnl":
                0.0,

            "buy_hold_return":
                (
                    df["close"].iloc[-1] /
                    df["close"].iloc[0]
                    - 1
                ),

            "long_trades":
                0,

            "short_trades":
                0,
        }

    # ---------------------------------------------------------------------
    # RETURNS
    # ---------------------------------------------------------------------

    total_net_pnl = trades[
        "net_pnl"
    ].sum()

    total_gross_pnl = trades[
        "gross_pnl"
    ].sum()

    net_return = (
        total_net_pnl /
        INITIAL_EQUITY
    )

    gross_return = (
        total_gross_pnl /
        INITIAL_EQUITY
    )

    # ---------------------------------------------------------------------
    # WIN RATE
    # ---------------------------------------------------------------------

    wins = (
        trades["net_pnl"] > 0
    ).sum()

    total_trades = len(
        trades
    )

    win_rate = (
        wins /
        total_trades
    )

    # ---------------------------------------------------------------------
    # PROFIT FACTOR
    # ---------------------------------------------------------------------

    gross_profit = trades.loc[
        trades["net_pnl"] > 0,
        "net_pnl"
    ].sum()

    gross_loss = abs(
        trades.loc[
            trades["net_pnl"] < 0,
            "net_pnl"
        ].sum()
    )

    if gross_loss > 0:

        profit_factor = (
            gross_profit /
            gross_loss
        )

    else:

        profit_factor = np.inf

    # ---------------------------------------------------------------------
    # EQUITY RETURNS
    # ---------------------------------------------------------------------

    eq = equity_curve[
        "equity"
    ].astype(float)

    hourly_returns = (
        eq
        .pct_change()
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
    )

    # ---------------------------------------------------------------------
    # SHARPE
    #
    # Binance perpetuals are 24/7.
    # 24 * 365 = 8760 hourly periods.
    # ---------------------------------------------------------------------

    if (
        len(hourly_returns) > 1
        and
        hourly_returns.std() > 0
    ):

        sharpe = (
            hourly_returns.mean() /
            hourly_returns.std()
            *
            np.sqrt(24 * 365)
        )

    else:

        sharpe = np.nan

    # ---------------------------------------------------------------------
    # SORTINO
    # ---------------------------------------------------------------------

    downside = hourly_returns[
        hourly_returns < 0
    ]

    if (
        len(downside) > 0
        and
        downside.std() > 0
    ):

        sortino = (
            hourly_returns.mean() /
            downside.std()
            *
            np.sqrt(24 * 365)
        )

    else:

        sortino = np.nan

    # ---------------------------------------------------------------------
    # MAX DRAWDOWN
    # ---------------------------------------------------------------------

    running_max = eq.cummax()

    drawdown = (
        eq /
        running_max
        - 1
    )

    max_drawdown = drawdown.min()

    # ---------------------------------------------------------------------
    # BUY AND HOLD
    # ---------------------------------------------------------------------

    buy_hold_return = (
        df["close"].iloc[-1] /
        df["close"].iloc[0]
        - 1
    )

    # ---------------------------------------------------------------------
    # LONG / SHORT
    # ---------------------------------------------------------------------

    long_trades = (
        trades["direction"]
        == "LONG"
    ).sum()

    short_trades = (
        trades["direction"]
        == "SHORT"
    ).sum()

    # ---------------------------------------------------------------------
    # STOP RATE
    # ---------------------------------------------------------------------

    stop_rate = (
        trades["stopped_out"]
        .mean()
    )

    # ---------------------------------------------------------------------
    # AVERAGE TRADE
    # ---------------------------------------------------------------------

    avg_trade_pnl = (
        trades["net_pnl"]
        .mean()
    )

    return {

        "symbol":
            symbol,

        "bars_tested":
            len(df),

        "total_trades":
            total_trades,

        "win_rate":
            win_rate,

        "profit_factor":
            profit_factor,

        "sharpe_ratio":
            sharpe,

        "sortino_ratio":
            sortino,

        "max_drawdown":
            max_drawdown,

        "gross_return":
            gross_return,

        "net_return":
            net_return,

        "total_net_pnl":
            total_net_pnl,

        "avg_trade_pnl":
            avg_trade_pnl,

        "buy_hold_return":
            buy_hold_return,

        "long_trades":
            long_trades,

        "short_trades":
            short_trades,

        "stop_rate":
            stop_rate,
    }


# =============================================================================
# 14. PER-REGIME PERFORMANCE
# =============================================================================

def regime_performance(trades):

    if trades.empty:

        return pd.DataFrame()

    rows = []

    for regime in [
        "Trending",
        "Mean Reversion",
        "High Volatility",
        "Event",
    ]:

        temp = trades[
            trades[
                "predicted_regime"
            ]
            == regime
        ]

        if temp.empty:

            continue

        rows.append(
            {
                "regime":
                    regime,

                "trades":
                    len(temp),

                "win_rate":
                    (
                        temp["net_pnl"] > 0
                    ).mean(),

                "net_pnl":
                    temp["net_pnl"].sum(),

                "avg_pnl":
                    temp["net_pnl"].mean(),

                "gross_return":
                    temp["gross_return"].sum(),

                "net_return":
                    temp["net_return"].sum(),

                "stop_rate":
                    temp["stopped_out"].mean(),
            }
        )

    return pd.DataFrame(rows)


# =============================================================================
# 15. WALK-FORWARD FOLD PERFORMANCE
# =============================================================================

def fold_performance(
    trades,
    stage7
):

    if trades.empty:

        return pd.DataFrame()

    # Map each trade to its Stage 7 fold using signal time
    fold_map = stage7[
        [
            "open_time",
            "fold"
        ]
    ].copy()

    fold_map = (
        fold_map
        .drop_duplicates("open_time")
    )

    trades = trades.copy()

    trades = pd.merge(
        trades,
        fold_map,
        left_on="signal_time",
        right_on="open_time",
        how="left"
    )

    rows = []

    for fold, temp in trades.groupby(
        "fold",
        dropna=False
    ):

        rows.append(
            {
                "fold":
                    fold,

                "trades":
                    len(temp),

                "win_rate":
                    (
                        temp["net_pnl"] > 0
                    ).mean(),

                "net_pnl":
                    temp["net_pnl"].sum(),

                "avg_pnl":
                    temp["net_pnl"].mean(),

                "long_trades":
                    (
                        temp["direction"]
                        == "LONG"
                    ).sum(),

                "short_trades":
                    (
                        temp["direction"]
                        == "SHORT"
                    ).sum(),

                "stop_rate":
                    temp[
                        "stopped_out"
                    ].mean(),
            }
        )

    return pd.DataFrame(
        rows
    ).sort_values(
        "fold"
    )


# =============================================================================
# 16. PROCESS ONE SYMBOL
# =============================================================================

def process_symbol(symbol):

    print("\n\n")
    print("#" * 90)
    print(
        f"STAGE 8 WALK-FORWARD BACKTEST — {symbol}"
    )
    print("#" * 90)

    # ---------------------------------------------------------------------
    # Prepare
    # ---------------------------------------------------------------------

    df = prepare_data(
        symbol
    )

    print(
        f"\nMerged rows: {len(df):,}"
    )

    print(
        f"Date range: "
        f"{df['open_time'].iloc[0]}"
        f" -> "
        f"{df['open_time'].iloc[-1]}"
    )

    # ---------------------------------------------------------------------
    # Remove rows where necessary trading information does not exist
    # ---------------------------------------------------------------------

    strategy_required = [
        "open",
        "high",
        "low",
        "close",
        "atr_14",
        "ema_20",
        "adx_14",
        "momentum_20h",
        "pred_future_regime",
        "prediction_confidence",
    ]

    missing = [
        col
        for col in strategy_required
        if col not in df.columns
    ]

    if missing:

        raise ValueError(
            f"{symbol}: Missing strategy columns:\n"
            f"{missing}"
        )

    # ---------------------------------------------------------------------
    # Numeric conversion
    # ---------------------------------------------------------------------

    numeric_columns = [
        "open",
        "high",
        "low",
        "close",
        "atr_14",
        "ema_20",
        "adx_14",
        "momentum_20h",
        "prediction_confidence",
    ]

    for col in numeric_columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    # ---------------------------------------------------------------------
    # Keep chronological data
    # ---------------------------------------------------------------------

    df = (
        df
        .sort_values("open_time")
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------------------
    # Run strategy
    # ---------------------------------------------------------------------

    trades, equity_curve, final_equity = (
        run_backtest(
            df,
            symbol
        )
    )

    # ---------------------------------------------------------------------
    # Metrics
    # ---------------------------------------------------------------------

    metrics = calculate_metrics(
        trades,
        equity_curve,
        df,
        symbol
    )

    metrics_df = pd.DataFrame(
        [metrics]
    )

    # ---------------------------------------------------------------------
    # Regime performance
    # ---------------------------------------------------------------------

    regime_df = regime_performance(
        trades
    )

    # ---------------------------------------------------------------------
    # Fold performance
    # ---------------------------------------------------------------------

    stage7 = load_stage7(
        symbol
    )

    fold_df = fold_performance(
        trades,
        stage7
    )

    # ---------------------------------------------------------------------
    # Save trades
    # ---------------------------------------------------------------------

    trade_path = (
        OUTPUT_DIR /
        f"{symbol}_08_trades.csv"
    )

    trades.to_csv(
        trade_path,
        index=False
    )

    # ---------------------------------------------------------------------
    # Save equity curve
    # ---------------------------------------------------------------------

    equity_path = (
        OUTPUT_DIR /
        f"{symbol}_08_equity_curve.csv"
    )

    equity_curve.to_csv(
        equity_path,
        index=False
    )

    # ---------------------------------------------------------------------
    # Save summary
    # ---------------------------------------------------------------------

    summary_path = (
        OUTPUT_DIR /
        f"{symbol}_08_summary.csv"
    )

    metrics_df.to_csv(
        summary_path,
        index=False
    )

    # ---------------------------------------------------------------------
    # Save regime attribution
    # ---------------------------------------------------------------------

    regime_path = (
        OUTPUT_DIR /
        f"{symbol}_08_regime_performance.csv"
    )

    regime_df.to_csv(
        regime_path,
        index=False
    )

    # ---------------------------------------------------------------------
    # Save fold attribution
    # ---------------------------------------------------------------------

    fold_path = (
        OUTPUT_DIR /
        f"{symbol}_08_walk_forward_folds.csv"
    )

    fold_df.to_csv(
        fold_path,
        index=False
    )

    # ---------------------------------------------------------------------
    # PRINT RESULTS
    # ---------------------------------------------------------------------

    print("\n" + "-" * 90)
    print(
        f"RESULTS — {symbol}"
    )
    print("-" * 90)

    print(
        f"Bars tested       : "
        f"{metrics['bars_tested']:,}"
    )

    print(
        f"Total trades      : "
        f"{metrics['total_trades']:,}"
    )

    print(
        f"Long trades       : "
        f"{metrics['long_trades']:,}"
    )

    print(
        f"Short trades      : "
        f"{metrics['short_trades']:,}"
    )

    print(
        f"Win rate          : "
        f"{metrics['win_rate']:.2%}"
    )

    print(
        f"Profit factor     : "
        f"{metrics['profit_factor']:.3f}"
    )

    print(
        f"Sharpe ratio      : "
        f"{metrics['sharpe_ratio']:.3f}"
    )

    print(
        f"Sortino ratio     : "
        f"{metrics['sortino_ratio']:.3f}"
    )

    print(
        f"Max drawdown      : "
        f"{metrics['max_drawdown']:.2%}"
    )

    print(
        f"Gross return      : "
        f"{metrics['gross_return']:.2%}"
    )

    print(
        f"Net return        : "
        f"{metrics['net_return']:.2%}"
    )

    print(
        f"Net PnL           : "
        f"{metrics['total_net_pnl']:,.2f}"
    )

    print(
        f"Final equity      : "
        f"{final_equity:,.2f}"
    )

    print(
        f"Buy & hold        : "
        f"{metrics['buy_hold_return']:.2%}"
    )

    print(
        f"Stop rate         : "
        f"{metrics['stop_rate']:.2%}"
    )

    print("\nSaved:")

    print(trade_path)
    print(equity_path)
    print(summary_path)
    print(regime_path)
    print(fold_path)

    return (
        metrics_df,
        trades,
        equity_curve,
        regime_df,
        fold_df
    )


# =============================================================================
# 17. RUN ALL SYMBOLS
# =============================================================================

if __name__ == "__main__":

    all_summaries = []

    for symbol in SYMBOLS:

        try:

            (
                summary,
                trades,
                equity_curve,
                regime_df,
                fold_df
            ) = process_symbol(
                symbol
            )

            all_summaries.append(
                summary
            )

        except Exception as e:

            print("\n" + "!" * 90)

            print(
                f"ERROR PROCESSING {symbol}"
            )

            print(
                str(e)
            )

            print("!" * 90)

    # =====================================================================
    # COMBINED SUMMARY
    # =====================================================================

    if all_summaries:

        combined_summary = pd.concat(
            all_summaries,
            ignore_index=True
        )

        combined_path = (
            OUTPUT_DIR /
            "STAGE_8_ALL_SYMBOLS_SUMMARY.csv"
        )

        combined_summary.to_csv(
            combined_path,
            index=False
        )

        print("\n")
        print("=" * 90)
        print(
            "STAGE 8 — ALL SYMBOLS SUMMARY"
        )
        print("=" * 90)

        print(
            combined_summary.to_string(
                index=False
            )
        )

        print(
            f"\nSaved combined summary:\n"
            f"{combined_path}"
        )

"




##########################################################################################
STAGE 8 WALK-FORWARD BACKTEST — BTCUSDT
##########################################################################################

PREPARING BTCUSDT
Stage 7 rows : 53,556
Feature rows : 58,597

Merged rows: 53,556
Date range: 2020-08-02 04:00:00+00:00 -> 2026-09-11 15:00:00+00:00

------------------------------------------------------------------------------------------
RESULTS — BTCUSDT
------------------------------------------------------------------------------------------
Bars tested       : 53,556
Total trades      : 1,420
Long trades       : 714
Short trades      : 706
Win rate          : 37.18%
Profit factor     : 0.585
Sharpe ratio      : -3.180
Sortino ratio     : -1.354
Max drawdown      : -72.80%
Gross return      : -8.96%
Net return        : -72.51%
Net PnL           : -72,510.94
Final equity      : 27,489.06
Buy & hold        : 594.53%
Stop rate         : 28.10%

Saved:
C:\Users

'\nfor symbol in SYMBOLS:\n    df = prepare_data(symbol)   # your existing Stage 8 prepare_data()\n    # ... same OHLC validation / numeric conversion you already do ...\n    best_config, holdout_metrics, all_results = grid_search_symbol(df, symbol)\n'

In [2]:
"""
===============================================================================
STAGE 8 v2 — META-LABELED, MULTI-FOLD WALK-FORWARD REGIME STRATEGY
===============================================================================
Replaces the single 70/30 grid-search optimizer. Why: your current holdout
results show tune -> holdout decay on every symbol (e.g. BNB win rate
0.446 -> 0.351, XAU 0.200 -> 0.158, all holdout profit factors < 1). That
pattern is the signature of overfitting a grid search to one split, not a
real edge.

WHAT THIS SCRIPT DOES DIFFERENTLY
----------------------------------------------------------------------------
1. META-LABELING
   The rule-based regime signal (Trending momentum/ADX, Mean-Reversion
   EMA-fade) is kept as the "primary model" that proposes WHEN and WHICH
   DIRECTION to trade. A secondary binary classifier (meta-model) is trained
   to predict whether that specific proposed trade will actually win, using
   only information available at signal time (regime probabilities, ADX,
   Hurst, multi-scale momentum agreement, volatility z-score, distance from
   EMA in ATR units). Only trades the meta-model is confident about are
   taken. This is the direct lever for lifting the realized win rate above
   52%: it trades less often, but only on the setups the data says look
   like winners.

2. MULTI-FOLD WALK-FORWARD (not one split)
   The data is split into N rolling folds (train -> embargo -> test). The
   meta-model is refit on each fold's training window only, and every
   config is scored on ALL test folds. A config only "passes" if it clears
   the win-rate bar on a majority of folds, not just one.

3. CONFLUENCE FILTERS PER REGIME
   Trending trades require multi-timeframe momentum agreement (5h/20h/60h
   same sign) AND Hurst > 0.5 (the market is actually persistent). Mean
   reversion trades require Hurst < 0.5 (actually mean-reverting) AND
   volume not exploding (an extension driven by a volume surge is a
   breakout, not something to fade).

4. HONEST OUTPUT
   For each symbol, if no config clears win_rate >= 0.52 on a majority of
   folds with profit_factor > 1, the script says so explicitly instead of
   reporting the least-bad option as if it were a strategy. Backtested
   "best of a grid" numbers that fail out-of-sample validation are not an
   edge, they're a Sharpe-ratio-shaped ghost.

INPUTS (unchanged from your pipeline)
----------------------------------------------------------------------------
    {BASE_DIR}/predictions/{symbol}_07_lightgbm_predictions.csv
    {BASE_DIR}/processed/{symbol}_1h_features.csv

OUTPUTS
----------------------------------------------------------------------------
    {BASE_DIR}/backtest/stage8_v2/{symbol}_fold_results.csv
    {BASE_DIR}/backtest/stage8_v2/{symbol}_best_config.json
    {BASE_DIR}/backtest/stage8_v2/ALL_SYMBOLS_SUMMARY.csv

REQUIRES: pandas, numpy, scikit-learn (LightGBM optional — falls back to
sklearn's GradientBoostingClassifier if lightgbm isn't installed).
===============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import json
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    HAS_LGBM = False


# =============================================================================
# 0. CONFIG — EDIT FOR YOUR MACHINE
# =============================================================================

BASE_DIR = Path(r"C:\Users\dell\Downloads\data")
PREDICTION_DIR = BASE_DIR / "predictions"
FEATURE_DIR = BASE_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "backtest" / "stage8_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SYMBOLS = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XAUUSDT", "XAGUSDT"]

PREDICTIONS_FILENAME_PATTERNS = [
    "{symbol}_07_lightgbm_predictions.csv",
    "{symbol}_07_predictions.csv",
]

INITIAL_EQUITY = 100_000.0
RISK_PER_TRADE = 0.005
FEE_RATE = 0.0004
SLIPPAGE_RATE = 0.0002
MAX_NOTIONAL_MULTIPLE = 1.0
MIN_BARS_BEFORE_FLIP = 2

# --- walk-forward fold geometry (1h bars) ---
N_FOLDS = 6
TRAIN_BARS = 24 * 240     # 240 days train
TEST_BARS = 24 * 45       # 45 days test
EMBARGO_BARS = 8          # >= max regime horizon_bars, prevents label leakage

# --- loose thresholds used ONLY to mine candidate trades for meta-labeling ---
LOOSE_CONFIDENCE = 0.50
LOOSE_ADX = 15.0
LOOSE_MARGIN = 0.0

# --- grid searched on top of the meta-model ---
CONFIDENCE_GRID = (0.55, 0.60, 0.65, 0.70)
ADX_GRID = (20.0, 25.0, 30.0)
MARGIN_GRID = (0.05, 0.10, 0.15)
META_THRESHOLD_GRID = (0.50, 0.55, 0.60, 0.65)

# --- acceptance bar ---
MIN_WIN_RATE = 0.52
MIN_PROFIT_FACTOR = 1.0
MIN_TRADES_PER_FOLD_TOTAL = 30      # summed across test folds
FOLD_PASS_FRACTION = 0.60           # must clear the bar on >=60% of folds

REGIME_PARAMS = {
    "Trending": {"stop_atr_mult": 1.5, "tp_atr_mult": 4.0, "horizon_bars": 8},
    "Mean Reversion": {"stop_atr_mult": 1.2, "tp_atr_mult": 2.0, "horizon_bars": 3},
}

PROB_COLS = ["prob_Trending", "prob_Mean Reversion", "prob_High Volatility", "prob_Event"]

FEATURE_ALIASES = {
    "momentum_5": "momentum_5h", "momentum_20": "momentum_20h", "momentum_60": "momentum_60h",
    "log_realized_vol_20": "realized_vol_20", "log_atr_pct": "atr_pct",
}

META_FEATURES = [
    "prediction_confidence", "prediction_margin", "adx_14", "atr_pct",
    "hurst_100", "momentum_5h", "momentum_20h", "momentum_60h",
    "volume_zscore_20", "realized_vol_20", "ema_distance_atr", "vol_regime_z",
    "direction_code",
]

# minimum viable fold geometry when a symbol's history is too short for the
# defaults above (used by compute_fold_geometry)
MIN_TRAIN_BARS = 24 * 90     # 90 days
MIN_TEST_BARS = 24 * 15      # 15 days
MIN_FOLDS = 3


# =============================================================================
# 1. LOADERS / FEATURE PREP  (same conventions as your existing Stage 8)
# =============================================================================

def find_predictions_path(symbol):
    for pattern in PREDICTIONS_FILENAME_PATTERNS:
        p = PREDICTION_DIR / pattern.format(symbol=symbol)
        if p.exists():
            return p
    raise FileNotFoundError(f"{symbol}: no Stage 7 prediction file found in {PREDICTION_DIR}")


def calculate_atr(df, period=14):
    high, low, close = df["high"], df["low"], df["close"]
    prev_close = close.shift(1)
    tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
    return tr.rolling(period, min_periods=period).mean()


def prediction_margin(row):
    probs = sorted((pd.to_numeric(row.get(c, np.nan), errors="coerce") for c in PROB_COLS), reverse=True)
    probs = [p for p in probs if pd.notna(p)]
    return probs[0] - probs[1] if len(probs) >= 2 else np.nan


def get_first(row, *names, default=np.nan):
    for n in names:
        if n in row and pd.notna(row[n]):
            return row[n]
    return default


def prepare_data(symbol):
    stage7 = pd.read_csv(find_predictions_path(symbol))
    stage7["open_time"] = pd.to_datetime(stage7["open_time"], utc=True)
    stage7 = stage7.sort_values("open_time").drop_duplicates("open_time").reset_index(drop=True)

    feat_path = FEATURE_DIR / f"{symbol}_1h_features.csv"
    features = pd.read_csv(feat_path)
    features["open_time"] = pd.to_datetime(features["open_time"], utc=True)
    features = features.sort_values("open_time").drop_duplicates("open_time").reset_index(drop=True)
    features = features.rename(columns=FEATURE_ALIASES)

    df = pd.merge(stage7, features, on="open_time", how="inner", suffixes=("", "_feature"))
    df = df.sort_values("open_time").reset_index(drop=True)

    df["atr_14"] = calculate_atr(df, 14)
    df["ema_20"] = df["close"].ewm(span=20, adjust=False, min_periods=20).mean()

    code_to_label = {1: "Trending", 2: "Mean Reversion", 3: "High Volatility", 4: "Event"}
    missing_label = df["pred_future_regime"].isna()
    if missing_label.any():
        df.loc[missing_label, "pred_future_regime"] = (
            pd.to_numeric(df.loc[missing_label, "pred_future_regime_code"], errors="coerce").map(code_to_label)
        )

    prob_map = {"Trending": "prob_Trending", "Mean Reversion": "prob_Mean Reversion",
                "High Volatility": "prob_High Volatility", "Event": "prob_Event"}
    df["prediction_confidence"] = np.nan
    for regime, col in prob_map.items():
        mask = df["pred_future_regime"].astype(str).eq(regime)
        df.loc[mask, "prediction_confidence"] = pd.to_numeric(df.loc[mask, col], errors="coerce")

    df["prediction_margin"] = df.apply(prediction_margin, axis=1)

    # normalize alt column names for momentum/vol/atr that may come from either
    # the raw feature file or the HMM-stage feature file
    for canonical, alts in {
        "momentum_5h": ["momentum_5h", "momentum_5"],
        "momentum_20h": ["momentum_20h", "momentum_20"],
        "momentum_60h": ["momentum_60h", "momentum_60"],
        "atr_pct": ["atr_pct", "log_atr_pct"],
        "realized_vol_20": ["realized_vol_20", "log_realized_vol_20"],
    }.items():
        if canonical not in df.columns:
            for alt in alts:
                if alt in df.columns:
                    df[canonical] = df[alt]
                    break

    df["ema_distance_atr"] = (df["close"] - df["ema_20"]) / df["atr_14"].replace(0, np.nan)

    # volatility regime z-score of realized_vol_20 itself (rolling, causal)
    rv = df["realized_vol_20"]
    df["vol_regime_z"] = (rv - rv.rolling(500, min_periods=100).mean()) / rv.rolling(500, min_periods=100).std()

    numeric_cols = ["open", "high", "low", "close", "atr_14", "ema_20", "adx_14",
                     "momentum_5h", "momentum_20h", "momentum_60h", "hurst_100",
                     "volume_zscore_20", "realized_vol_20", "prediction_confidence"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.sort_values("open_time").reset_index(drop=True)

    # Fail loud, not silent: a missing META_FEATURES column means every mined
    # candidate trade silently gets dropped in build_meta_feature_row and
    # you see 'not enough candidate trades' with no clue why. Check now.
    required_meta_cols = [f for f in META_FEATURES if f != "direction_code"]
    missing = [c for c in required_meta_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"{symbol}: missing columns needed for meta-labeling features: {missing}. "
            f"Available columns: {sorted(df.columns.tolist())}"
        )
    non_null_frac = {c: float(df[c].notna().mean()) for c in required_meta_cols}
    sparse = {c: f for c, f in non_null_frac.items() if f < 0.5}
    if sparse:
        print(f"  WARNING [{symbol}]: these meta-feature columns are >50% NaN, "
              f"which will silently starve candidate mining: {sparse}")

    return df


# =============================================================================
# 2. PRIMARY SIGNAL (rule-based, with confluence filters)
# =============================================================================

def generate_primary_signal(row, confidence_threshold, adx_threshold, margin_threshold):
    """Returns (direction, regime) or (0, None). direction in {-1, 0, 1}.

    Deliberately BROAD: this is the primary rule the meta-model will later
    filter. Hurst / momentum-agreement / volume-z / vol-regime-z are NOT
    gated here — they're passed to the meta-model as features instead, so
    it can learn (from data) which combinations actually predict a win,
    rather than us hard-coding a guess and starving the mining stage of
    examples. Gating here as well as in the meta-model is double-jeopardy:
    it's what caused 'not enough mined candidate trades' in the first run.
    """
    regime = row["pred_future_regime"]
    confidence = row["prediction_confidence"]
    margin = row.get("prediction_margin", np.nan)

    if pd.isna(confidence) or confidence < confidence_threshold:
        return 0, None
    if pd.isna(margin) or margin < margin_threshold:
        return 0, None

    m20 = row.get("momentum_20h", np.nan)
    adx = row.get("adx_14", np.nan)
    close, ema20, atr = row.get("close", np.nan), row.get("ema_20", np.nan), row.get("atr_14", np.nan)

    if any(pd.isna(x) for x in [m20, adx, close, ema20, atr]) or atr <= 0:
        return 0, None

    if regime == "Trending":
        if m20 > 0 and adx >= adx_threshold and close > ema20:
            return 1, regime
        if m20 < 0 and adx >= adx_threshold and close < ema20:
            return -1, regime
        return 0, None

    if regime == "Mean Reversion":
        if close > (ema20 + atr):
            return -1, regime
        if close < (ema20 - atr):
            return 1, regime
        return 0, None

    return 0, None  # High Volatility / Event: no entries


# =============================================================================
# 3. BACKTEST ENGINE (unchanged mechanics: next-bar fill, ATR stop/TP,
#    regime-flip exit, time exit)
# =============================================================================

def run_backtest(df, symbol, confidence_threshold, adx_threshold, margin_threshold,
                  meta_model=None, meta_scaler=None, meta_threshold=None):
    trades, equity_curve = [], []
    equity = INITIAL_EQUITY
    position = None
    n = len(df)
    i = 0

    def sig(row):
        direction, regime = generate_primary_signal(row, confidence_threshold, adx_threshold, margin_threshold)
        if direction == 0 or meta_model is None:
            return direction, regime
        feat = build_meta_feature_row(row, direction)
        if feat is None:
            return 0, None
        x = meta_scaler.transform([feat])
        p_win = meta_model.predict_proba(x)[0, 1]
        if p_win < meta_threshold:
            return 0, None
        return direction, regime

    while i < n:
        row = df.iloc[i]
        current_time = row["open_time"]

        if position is None:
            direction, regime = sig(row)
            if direction != 0 and i + 1 < n:
                entry_row = df.iloc[i + 1]
                entry_price = float(entry_row["open"])
                atr = float(row["atr_14"])
                if not np.isfinite(atr) or atr <= 0:
                    i += 1
                    continue

                params = REGIME_PARAMS.get(regime, REGIME_PARAMS["Trending"])
                stop_distance = params["stop_atr_mult"] * atr
                risk_capital = equity * RISK_PER_TRADE
                quantity = risk_capital / stop_distance
                max_notional = equity * MAX_NOTIONAL_MULTIPLE
                if quantity * entry_price > max_notional:
                    quantity = max_notional / entry_price
                if not np.isfinite(quantity) or quantity <= 0:
                    i += 1
                    continue

                if direction == 1:
                    stop_price = entry_price - params["stop_atr_mult"] * atr
                    take_profit = entry_price + params["tp_atr_mult"] * atr
                else:
                    stop_price = entry_price + params["stop_atr_mult"] * atr
                    take_profit = entry_price - params["tp_atr_mult"] * atr

                position = {
                    "direction": direction, "entry_index": i + 1, "entry_time": entry_row["open_time"],
                    "entry_price": entry_price, "quantity": quantity, "stop_price": stop_price,
                    "take_profit": take_profit, "horizon_bars": params["horizon_bars"], "predicted_regime": regime,
                }
                i += 1
                continue

        if position is not None:
            direction = position["direction"]
            entry_price = position["entry_price"]
            quantity = position["quantity"]
            stop_price = position["stop_price"]
            take_profit = position["take_profit"]
            holding_bars = i - position["entry_index"] + 1

            current_high, current_low = float(row["high"]), float(row["low"])
            exit_reason, exit_price, stopped_out = None, None, False

            if direction == 1 and current_low <= stop_price:
                exit_reason, exit_price, stopped_out = "stop_loss", stop_price, True
            elif direction == -1 and current_high >= stop_price:
                exit_reason, exit_price, stopped_out = "stop_loss", stop_price, True

            if exit_reason is None:
                if direction == 1 and current_high >= take_profit:
                    exit_reason, exit_price = "take_profit", take_profit
                elif direction == -1 and current_low <= take_profit:
                    exit_reason, exit_price = "take_profit", take_profit

            if exit_reason is None and i + 1 < n and holding_bars >= MIN_BARS_BEFORE_FLIP:
                current_signal, _ = sig(row)
                if current_signal != 0 and current_signal != direction:
                    next_row = df.iloc[i + 1]
                    exit_reason, exit_price = "regime_or_direction_flip", float(next_row["open"])

            if exit_reason is None and holding_bars >= position["horizon_bars"]:
                exit_reason, exit_price = "time_exit", float(row["close"])

            if exit_reason is not None:
                gross_return = direction * (exit_price / entry_price - 1.0)
                cost = 2 * (FEE_RATE + SLIPPAGE_RATE)
                net_return = gross_return - cost
                net_pnl = quantity * entry_price * net_return
                equity += net_pnl
                trades.append({
                    "symbol": symbol, "entry_time": position["entry_time"], "exit_time": row["open_time"],
                    "direction": "LONG" if direction == 1 else "SHORT",
                    "predicted_regime": position["predicted_regime"], "holding_bars": holding_bars,
                    "exit_reason": exit_reason, "stopped_out": stopped_out,
                    "net_return": net_return, "net_pnl": net_pnl,
                })
                position = None

        mtm_equity = equity
        if position is not None:
            current_close = float(row["close"])
            unrealized = position["direction"] * (current_close / position["entry_price"] - 1)
            mtm_equity += position["entry_price"] * position["quantity"] * unrealized
        equity_curve.append({"open_time": current_time, "equity": mtm_equity})
        i += 1

    return pd.DataFrame(trades), pd.DataFrame(equity_curve), equity


def quick_metrics(trades_df, equity_df):
    if trades_df.empty:
        return {"total_trades": 0, "win_rate": np.nan, "profit_factor": np.nan,
                "max_drawdown": 0.0, "net_return": 0.0}
    wins = (trades_df["net_pnl"] > 0).sum()
    total_trades = len(trades_df)
    win_rate = wins / total_trades
    gross_profit = trades_df.loc[trades_df["net_pnl"] > 0, "net_pnl"].sum()
    gross_loss = abs(trades_df.loc[trades_df["net_pnl"] < 0, "net_pnl"].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf
    eq = equity_df["equity"].astype(float)
    drawdown = (eq - eq.cummax()) / eq.cummax()
    max_drawdown = drawdown.min()
    net_return = trades_df["net_pnl"].sum() / INITIAL_EQUITY
    return {"total_trades": total_trades, "win_rate": win_rate, "profit_factor": profit_factor,
            "max_drawdown": max_drawdown, "net_return": net_return}


# =============================================================================
# 4. META-LABELING
# =============================================================================

def build_meta_feature_row(row, direction):
    vals = []
    for f in META_FEATURES:
        if f == "direction_code":
            vals.append(direction)
        else:
            v = row.get(f, np.nan)
            if pd.isna(v):
                return None
            vals.append(v)
    return vals


def mine_candidate_trades(df, symbol):
    """Runs the primary signal at LOOSE thresholds (no meta-filter) purely to
    harvest a labeled dataset of (features_at_signal_time, won_or_lost)."""
    trades, _, _ = run_backtest(df, symbol, LOOSE_CONFIDENCE, LOOSE_ADX, LOOSE_MARGIN)
    return trades


def build_meta_training_set(df, symbol):
    """Re-walks the data to attach signal-time features to each mined trade."""
    records = []
    n = len(df)
    for i in range(n - 1):
        row = df.iloc[i]
        direction, regime = generate_primary_signal(row, LOOSE_CONFIDENCE, LOOSE_ADX, LOOSE_MARGIN)
        if direction == 0:
            continue
        feat = build_meta_feature_row(row, direction)
        if feat is None:
            continue
        records.append((i, feat))
    if not records:
        return None, None

    trades_df = mine_candidate_trades(df, symbol)
    if trades_df.empty:
        return None, None

    # Align mined trades to signal bars by order (run_backtest processes signals
    # sequentially and skips overlapping ones while in a position, so trades_df
    # rows correspond, in order, to a subset of `records`).
    feats_ordered = [r[1] for r in records]
    k = min(len(feats_ordered), len(trades_df))
    X = np.array(feats_ordered[:k])
    y = (trades_df["net_pnl"].values[:k] > 0).astype(int)
    return X, y


def train_meta_model(X, y):
    if len(np.unique(y)) < 2 or len(y) < 40:
        return None, None
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    if HAS_LGBM:
        model = LGBMClassifier(n_estimators=150, max_depth=4, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=-1)
    else:
        model = GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42)
    model.fit(Xs, y)
    return model, scaler


# =============================================================================
# 5. WALK-FORWARD FOLDS
# =============================================================================

def compute_fold_geometry(n_rows):
    """Fits the walk-forward geometry to however much history this symbol
    actually has, instead of failing outright on shorter series (e.g.
    XAUUSDT/XAGUSDT perpetuals, which have far less Binance history than
    BTCUSDT). Tries the default geometry first, then shrinks train/test
    windows and fold count together until something fits, down to
    MIN_TRAIN_BARS/MIN_TEST_BARS/MIN_FOLDS. Returns (train_bars, test_bars,
    n_folds) or (None, None, 0) if even the minimum doesn't fit.
    """
    for scale in (1.0, 0.75, 0.6, 0.5, 0.4, 0.3, 0.2, 0.15, 0.1):
        train_bars = max(MIN_TRAIN_BARS, int(TRAIN_BARS * scale))
        test_bars = max(MIN_TEST_BARS, int(TEST_BARS * scale))
        for n_folds in range(N_FOLDS, MIN_FOLDS - 1, -1):
            required = train_bars + EMBARGO_BARS + test_bars * n_folds
            if required <= n_rows:
                return train_bars, test_bars, n_folds
    return None, None, 0


def make_folds(n_rows, train_bars, test_bars, n_folds):
    folds = []
    start = 0
    while True:
        train_end = start + train_bars
        test_start = train_end + EMBARGO_BARS
        test_end = test_start + test_bars
        if test_end > n_rows:
            break
        folds.append((start, train_end, test_start, test_end))
        start += test_bars
        if len(folds) >= n_folds:
            break
    return folds


# =============================================================================
# 6. PER-SYMBOL WALK-FORWARD OPTIMIZATION
# =============================================================================

def optimize_symbol(symbol):
    print(f"\n{'='*90}\n{symbol}\n{'='*90}")
    df = prepare_data(symbol)
    print(f"  Total usable bars (merged predictions+features): {len(df):,} "
          f"({df['open_time'].min()} -> {df['open_time'].max()})")

    train_bars, test_bars, n_folds = compute_fold_geometry(len(df))
    if n_folds == 0:
        print(f"  Not enough history even at minimum geometry "
              f"({MIN_TRAIN_BARS}+{EMBARGO_BARS}+{MIN_TEST_BARS}*{MIN_FOLDS} bars needed) — skipping.")
        return None
    if (train_bars, test_bars, n_folds) != (TRAIN_BARS, TEST_BARS, N_FOLDS):
        print(f"  History shorter than default geometry — using "
              f"train={train_bars} test={test_bars} n_folds={n_folds} bars instead.")

    folds = make_folds(len(df), train_bars, test_bars, n_folds)
    if not folds:
        print("  Fold construction failed unexpectedly — skipping.")
        return None

    config_grid = list(product(CONFIDENCE_GRID, ADX_GRID, MARGIN_GRID, META_THRESHOLD_GRID))
    per_config_fold_metrics = {cfg: [] for cfg in config_grid}
    fold_rows = []

    for fold_i, (tr_s, tr_e, te_s, te_e) in enumerate(folds):
        train_df = df.iloc[tr_s:tr_e].reset_index(drop=True)
        test_df = df.iloc[te_s:te_e].reset_index(drop=True)

        X, y = build_meta_training_set(train_df, symbol)
        n_candidates = 0 if X is None else len(y)
        n_wins = 0 if X is None else int(y.sum())
        meta_model, meta_scaler = (train_meta_model(X, y) if X is not None else (None, None))
        if meta_model is None:
            print(f"  Fold {fold_i}: only {n_candidates} mined candidate trades "
                  f"({n_wins} wins) — need >=40 with both classes present, skipping fold.")
            continue
        print(f"  Fold {fold_i}: trained meta-model on {n_candidates} candidate trades "
              f"({n_wins} wins, {n_candidates - n_wins} losses).")

        for cfg in config_grid:
            conf_t, adx_t, margin_t, meta_t = cfg
            trades_df, equity_df, _ = run_backtest(
                test_df, symbol, conf_t, adx_t, margin_t,
                meta_model=meta_model, meta_scaler=meta_scaler, meta_threshold=meta_t,
            )
            m = quick_metrics(trades_df, equity_df)
            m["fold"] = fold_i
            per_config_fold_metrics[cfg].append(m)
            fold_rows.append({**m, "confidence_threshold": conf_t, "adx_threshold": adx_t,
                               "margin_threshold": margin_t, "meta_threshold": meta_t})

    if not fold_rows:
        print("  No folds produced results (meta-model training failed everywhere).")
        return None

    fold_results_df = pd.DataFrame(fold_rows)
    fold_results_df.to_csv(OUTPUT_DIR / f"{symbol}_fold_results.csv", index=False)

    # aggregate per config across folds
    agg_rows = []
    for cfg, fold_list in per_config_fold_metrics.items():
        if not fold_list:
            continue
        wr = [m["win_rate"] for m in fold_list if m["total_trades"] > 0]
        pf = [m["profit_factor"] for m in fold_list if m["total_trades"] > 0 and np.isfinite(m["profit_factor"])]
        dd = [m["max_drawdown"] for m in fold_list]
        total_trades = sum(m["total_trades"] for m in fold_list)
        net_return = sum(m["net_return"] for m in fold_list)
        n_folds_with_trades = len(wr)
        n_folds_passing = sum(1 for w, p in zip(wr, pf) if w >= MIN_WIN_RATE and p >= MIN_PROFIT_FACTOR) if pf else 0

        agg_rows.append({
            "confidence_threshold": cfg[0], "adx_threshold": cfg[1],
            "margin_threshold": cfg[2], "meta_threshold": cfg[3],
            "total_trades": total_trades,
            "median_win_rate": np.median(wr) if wr else np.nan,
            "median_profit_factor": np.median(pf) if pf else np.nan,
            "median_max_drawdown": np.median(dd) if dd else np.nan,
            "net_return_sum": net_return,
            "n_folds_with_trades": n_folds_with_trades,
            "n_folds_passing": n_folds_passing,
            "fold_pass_fraction": (n_folds_passing / n_folds_with_trades) if n_folds_with_trades else 0.0,
        })

    agg_df = pd.DataFrame(agg_rows)

    robust = agg_df[
        (agg_df["total_trades"] >= MIN_TRADES_PER_FOLD_TOTAL)
        & (agg_df["fold_pass_fraction"] >= FOLD_PASS_FRACTION)
        & (agg_df["median_win_rate"] >= MIN_WIN_RATE)
        & (agg_df["median_profit_factor"] >= MIN_PROFIT_FACTOR)
    ].copy()

    if robust.empty:
        best_overall = agg_df.sort_values(
            ["fold_pass_fraction", "median_profit_factor"], ascending=False
        ).head(1)
        print(f"  NO ROBUST EDGE FOUND for {symbol}: no config cleared "
              f"win_rate>={MIN_WIN_RATE:.0%} AND profit_factor>={MIN_PROFIT_FACTOR} "
              f"on >={FOLD_PASS_FRACTION:.0%} of test folds.")
        if not best_overall.empty:
            r = best_overall.iloc[0]
            print(f"  Closest attempt: conf={r['confidence_threshold']}, adx={r['adx_threshold']}, "
                  f"margin={r['margin_threshold']}, meta={r['meta_threshold']} -> "
                  f"median_win_rate={r['median_win_rate']:.2%}, "
                  f"median_pf={r['median_profit_factor']:.2f}, "
                  f"passed {r['n_folds_passing']}/{r['n_folds_with_trades']} folds.")
        status = "NO_EDGE"
        chosen = best_overall.iloc[0].to_dict() if not best_overall.empty else None
    else:
        # composite score: reward profit factor and net return, penalize drawdown
        robust["score"] = (
            robust["median_profit_factor"]
            - 2.0 * robust["median_max_drawdown"].abs()
            + robust["net_return_sum"]
        )
        robust = robust.sort_values("score", ascending=False)
        chosen = robust.iloc[0].to_dict()
        status = "EDGE_FOUND"
        print(f"  EDGE FOUND: conf={chosen['confidence_threshold']}, adx={chosen['adx_threshold']}, "
              f"margin={chosen['margin_threshold']}, meta={chosen['meta_threshold']}")
        print(f"  median_win_rate={chosen['median_win_rate']:.2%}  "
              f"median_profit_factor={chosen['median_profit_factor']:.2f}  "
              f"median_max_dd={chosen['median_max_drawdown']:.2%}  "
              f"passed {chosen['n_folds_passing']}/{chosen['n_folds_with_trades']} folds  "
              f"total_trades={chosen['total_trades']:.0f}")

    agg_df.to_csv(OUTPUT_DIR / f"{symbol}_config_aggregate.csv", index=False)
    with open(OUTPUT_DIR / f"{symbol}_best_config.json", "w") as f:
        json.dump({"symbol": symbol, "status": status,
                    "config": {k: (None if pd.isna(v) else v) for k, v in (chosen or {}).items()}},
                   f, indent=2, default=str)

    return {"symbol": symbol, "status": status, **({} if chosen is None else chosen)}


# =============================================================================
# 7. RUN ALL SYMBOLS
# =============================================================================

if __name__ == "__main__":
    summary = []
    for symbol in SYMBOLS:
        try:
            result = optimize_symbol(symbol)
            if result:
                summary.append(result)
        except Exception as e:
            print(f"\n{'!'*90}\nERROR PROCESSING {symbol}: {e}\n{'!'*90}")

    if summary:
        summary_df = pd.DataFrame(summary)
        summary_df.to_csv(OUTPUT_DIR / "ALL_SYMBOLS_SUMMARY.csv", index=False)
        print(f"\n\n{'='*90}\nSUMMARY\n{'='*90}")
        print(summary_df.to_string(index=False))


BTCUSDT
  Total usable bars (merged predictions+features): 53,556 (2020-08-02 04:00:00+00:00 -> 2026-09-11 15:00:00+00:00)
  Fold 0: trained meta-model on 297 candidate trades (115 wins, 182 losses).
  Fold 1: trained meta-model on 211 candidate trades (75 wins, 136 losses).
  Fold 2: trained meta-model on 80 candidate trades (33 wins, 47 losses).
  Fold 3: trained meta-model on 76 candidate trades (34 wins, 42 losses).
  Fold 4: trained meta-model on 115 candidate trades (48 wins, 67 losses).
  Fold 5: trained meta-model on 159 candidate trades (66 wins, 93 losses).
  NO ROBUST EDGE FOUND for BTCUSDT: no config cleared win_rate>=52% AND profit_factor>=1.0 on >=60% of test folds.
  Closest attempt: conf=0.55, adx=20.0, margin=0.05, meta=0.55 -> median_win_rate=33.33%, median_pf=1.06, passed 2.0/5.0 folds.

ETHUSDT
  Total usable bars (merged predictions+features): 53,557 (2020-08-02 04:00:00+00:00 -> 2026-09-11 16:00:00+00:00)
  Fold 0: trained meta-model on 349 candidate trades (143 

In [10]:
"""
===============================================================================
ETHUSDT WALK-FORWARD BACKTEST  +  INDIVIDUAL $5,000 PnL ACCOUNT (2020-2025)
===============================================================================
Self-contained version of your Stage 9 (meta-labeled walk-forward optimizer)
+ Stage 10 (compounding PnL account), narrowed to ETHUSDT and pointed at the
files you actually have on disk right now:

    C:\\Users\\dell\\Downloads\\data\\regimes\\ETHUSDT_07_predictions.csv
    C:\\Users\\dell\\Downloads\\data\\processed\\ETHUSDT_1h_features.csv   <- ASSUMED PATH,
                                                                             see NOTE below

 ON THE FEATURES FILE
----------------------------------------------------------------------------
Your Stage 7 output lives in `regimes/`, but the backtest also needs the raw
OHLCV + engineered columns (open/high/low/close, adx_14, momentum_*, hurst_100,
volume_zscore_20, realized_vol_20) that come from your Stage 1/2 pipeline and
are normally written to `processed/{symbol}_1h_features.csv`. If that file
isn't there anymore, point FEATURE_DIR at wherever it now lives -- this script
will raise a clear error naming the missing columns rather than fail silently.

WHAT THIS SCRIPT DOES
----------------------------------------------------------------------------
1. Loads Stage 5/7 regime predictions + Stage 1/2 features for ETHUSDT only,
   restricted to 2020-01-01 -> 2025-12-31.
2. Builds N rolling walk-forward folds (train -> embargo -> test).
3. Per fold: mines candidate trades at loose thresholds, trains a meta-model
   (win/lose classifier) on the training window only, then grid-searches
   (confidence, ADX, margin, meta) thresholds on the held-out test window.
4. Picks the config that passes the acceptance bar on the most test folds
   (or reports NO_EDGE honestly if none do).
5. Re-runs the OOS folds in order as ONE continuous, isolated $5,000 account
   (STAGE 10 step) -- compounding, all fees + slippage included -- and saves
   the trade blotter, equity curve, and summary metrics.

STRATEGY DEFINITION (see printed summary at the end for the numbers actually
used on your data)
----------------------------------------------------------------------------
- Direction:      regime classifier (Trending / Mean Reversion) + confidence
                  + margin-of-confidence gate + ADX/EMA confluence, then a
                  meta-model veto that only allows the trade if it predicts
                  a win with probability >= meta_threshold.
- Stop-loss:      1.5x ATR(14) in Trending, 1.2x ATR(14) in Mean Reversion.
- Take-profit:    4.0x ATR(14) in Trending, 2.0x ATR(14) in Mean Reversion.
- Time exit:      close after 8 bars (Trending) / 3 bars (Mean Reversion)
                  if neither SL nor TP was hit.
- Position size:  risk_per_trade% of CURRENT account equity / stop-distance,
                  capped so notional <= 1x equity (no leverage).
- Frictions:      taker fee 0.04% + slippage 0.02%, charged on entry AND exit.

WHY THIS DESIGN TARGETS "MAX PROFIT, MIN DRAWDOWN"
----------------------------------------------------------------------------
- ATR-based stops scale with realized volatility, so the stop is neither too
  tight in a volatile stretch (which would just get stopped out by noise)
  nor too loose in a calm stretch (which would give back more than needed).
- Fixed-fractional sizing means no single trade can damage the account by
  more than a fixed % of ITS OWN equity -- this is what bounds max drawdown
  regardless of how large a losing streak gets, and it's why the position
  size shrinks automatically after a drawdown (protecting capital) and grows
  again after a run of wins (compounding profit).
- The meta-model doesn't add profit by predicting the market better in some
  abstract sense -- it adds profit by refusing the subset of primary-rule
  trades that historically lost, which mechanically raises win rate and
  profit factor for the SAME rule.
- Walk-forward + embargo means the chosen thresholds had to survive being
  tested on data the model never trained on, across multiple folds -- this
  is what stops a config that looks great on one lucky slice of history
  from being deployed on your real $5,000.

===============================================================================
"""

import json
import warnings
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression  # noqa: F401 (kept for parity)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    HAS_LGBM = False


# =============================================================================
# 0. CONFIG -- EDIT FOR YOUR MACHINE
# =============================================================================

SYMBOL = "ETHUSDT"

BASE_DIR = Path(r"C:\Users\dell\Downloads\data")
PREDICTION_DIR = BASE_DIR / "regimes"      # <- where your ETHUSDT_07_predictions.csv lives
FEATURE_DIR = BASE_DIR / "processed"       # <- where ETHUSDT_1h_features.csv lives (check this!)
OUTPUT_DIR = BASE_DIR / "backtest" / "eth_walkforward"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_FILENAME_PATTERNS = [
    "{symbol}_07_predictions.csv",
    "{symbol}_07_lightgbm_predictions.csv",
]

BACKTEST_START = pd.Timestamp("2019-01-01", tz="UTC")
BACKTEST_END = pd.Timestamp("2026-09-11", tz="UTC")

PER_SYMBOL_STARTING_CAPITAL = 5_000.0     # isolated account size, as requested
INITIAL_EQUITY = PER_SYMBOL_STARTING_CAPITAL   # mutated during compounding, see Stage 10 loop

RISK_PER_TRADE = 0.005          # 0.5% of CURRENT equity risked per trade
FEE_RATE = 0.0004               # 0.04% taker fee
SLIPPAGE_RATE = 0.0002          # 0.02% slippage
MAX_NOTIONAL_MULTIPLE = 1.0     # no leverage: notional <= 1x equity
MIN_BARS_BEFORE_FLIP = 2

# --- walk-forward fold geometry (1h bars) ---
N_FOLDS = 6
TRAIN_BARS = 24 * 240        # 240 days train
TEST_BARS = 24 * 45          # 45 days test
EMBARGO_BARS = 8             # >= max horizon_bars, prevents label leakage
MIN_TRAIN_BARS = 24 * 90
MIN_TEST_BARS = 24 * 15
MIN_FOLDS = 3

# --- loose thresholds used ONLY to mine candidate trades for meta-labeling ---
LOOSE_CONFIDENCE = 0.50
LOOSE_ADX = 15.0
LOOSE_MARGIN = 0.0

# --- grid searched on top of the meta-model ---
CONFIDENCE_GRID = (0.55, 0.60, 0.65, 0.70)
ADX_GRID = (20.0, 25.0, 30.0)
MARGIN_GRID = (0.05, 0.10, 0.15)
META_THRESHOLD_GRID = (0.50, 0.55, 0.60, 0.65)

# --- acceptance bar ---
MIN_WIN_RATE = 0.52
MIN_PROFIT_FACTOR = 1.0
MIN_TRADES_PER_FOLD_TOTAL = 30
FOLD_PASS_FRACTION = 0.60

REGIME_PARAMS = {
    "Trending": {"stop_atr_mult": 1.5, "tp_atr_mult": 4.0, "horizon_bars": 8},
    "Mean Reversion": {"stop_atr_mult": 1.2, "tp_atr_mult": 2.0, "horizon_bars": 3},
}

PROB_COLS = ["prob_Trending", "prob_Mean Reversion", "prob_High Volatility", "prob_Event"]

FEATURE_ALIASES = {
    "momentum_5": "momentum_5h", "momentum_20": "momentum_20h", "momentum_60": "momentum_60h",
    "log_realized_vol_20": "realized_vol_20", "log_atr_pct": "atr_pct",
}

META_FEATURES = [
    "prediction_confidence", "prediction_margin", "adx_14", "atr_pct",
    "hurst_100", "momentum_5h", "momentum_20h", "momentum_60h",
    "volume_zscore_20", "realized_vol_20", "ema_distance_atr", "vol_regime_z",
    "direction_code",
]


# =============================================================================
# 1. LOADERS / FEATURE PREP
# =============================================================================

def find_predictions_path(symbol):
    for pattern in PREDICTIONS_FILENAME_PATTERNS:
        p = PREDICTION_DIR / pattern.format(symbol=symbol)
        if p.exists():
            return p
    raise FileNotFoundError(f"{symbol}: no Stage 7 prediction file found in {PREDICTION_DIR}")


def calculate_atr(df, period=14):
    high, low, close = df["high"], df["low"], df["close"]
    prev_close = close.shift(1)
    tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
    return tr.rolling(period, min_periods=period).mean()


def prediction_margin(row):
    probs = sorted((pd.to_numeric(row.get(c, np.nan), errors="coerce") for c in PROB_COLS), reverse=True)
    probs = [p for p in probs if pd.notna(p)]
    return probs[0] - probs[1] if len(probs) >= 2 else np.nan


def prepare_data(symbol):
    stage7 = pd.read_csv(find_predictions_path(symbol))
    stage7["open_time"] = pd.to_datetime(stage7["open_time"], utc=True)
    stage7 = stage7.sort_values("open_time").drop_duplicates("open_time").reset_index(drop=True)

    feat_path = FEATURE_DIR / f"{symbol}_1h_features.csv"
    if not feat_path.exists():
        raise FileNotFoundError(
            f"Features file not found: {feat_path}\n"
            f"Update FEATURE_DIR at the top of this script to wherever "
            f"{symbol}_1h_features.csv actually lives."
        )
    features = pd.read_csv(feat_path)
    features["open_time"] = pd.to_datetime(features["open_time"], utc=True)
    features = features.sort_values("open_time").drop_duplicates("open_time").reset_index(drop=True)
    features = features.rename(columns=FEATURE_ALIASES)

    df = pd.merge(stage7, features, on="open_time", how="inner", suffixes=("", "_feature"))
    df = df.sort_values("open_time").reset_index(drop=True)

    # restrict to the requested 2020-2025 walk-forward window
    df = df[(df["open_time"] >= BACKTEST_START) & (df["open_time"] <= BACKTEST_END)].reset_index(drop=True)

    df["atr_14"] = calculate_atr(df, 14)
    df["ema_20"] = df["close"].ewm(span=20, adjust=False, min_periods=20).mean()

    code_to_label = {1: "Trending", 2: "Mean Reversion", 3: "High Volatility", 4: "Event"}
    missing_label = df["pred_future_regime"].isna()
    if missing_label.any():
        df.loc[missing_label, "pred_future_regime"] = (
            pd.to_numeric(df.loc[missing_label, "pred_future_regime_code"], errors="coerce").map(code_to_label)
        )

    prob_map = {"Trending": "prob_Trending", "Mean Reversion": "prob_Mean Reversion",
                "High Volatility": "prob_High Volatility", "Event": "prob_Event"}
    df["prediction_confidence"] = np.nan
    for regime, col in prob_map.items():
        mask = df["pred_future_regime"].astype(str).eq(regime)
        df.loc[mask, "prediction_confidence"] = pd.to_numeric(df.loc[mask, col], errors="coerce")

    df["prediction_margin"] = df.apply(prediction_margin, axis=1)

    for canonical, alts in {
        "momentum_5h": ["momentum_5h", "momentum_5"],
        "momentum_20h": ["momentum_20h", "momentum_20"],
        "momentum_60h": ["momentum_60h", "momentum_60"],
        "atr_pct": ["atr_pct", "log_atr_pct"],
        "realized_vol_20": ["realized_vol_20", "log_realized_vol_20"],
    }.items():
        if canonical not in df.columns:
            for alt in alts:
                if alt in df.columns:
                    df[canonical] = df[alt]
                    break

    df["ema_distance_atr"] = (df["close"] - df["ema_20"]) / df["atr_14"].replace(0, np.nan)

    rv = df["realized_vol_20"]
    df["vol_regime_z"] = (rv - rv.rolling(500, min_periods=100).mean()) / rv.rolling(500, min_periods=100).std()

    numeric_cols = ["open", "high", "low", "close", "atr_14", "ema_20", "adx_14",
                     "momentum_5h", "momentum_20h", "momentum_60h", "hurst_100",
                     "volume_zscore_20", "realized_vol_20", "prediction_confidence"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.sort_values("open_time").reset_index(drop=True)

    required_meta_cols = [f for f in META_FEATURES if f != "direction_code"]
    missing = [c for c in required_meta_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"{symbol}: missing columns needed for meta-labeling features: {missing}. "
            f"Available columns: {sorted(df.columns.tolist())}"
        )
    return df


# =============================================================================
# 2. PRIMARY SIGNAL
# =============================================================================

def generate_primary_signal(row, confidence_threshold, adx_threshold, margin_threshold):
    regime = row["pred_future_regime"]
    confidence = row["prediction_confidence"]
    margin = row.get("prediction_margin", np.nan)

    if pd.isna(confidence) or confidence < confidence_threshold:
        return 0, None
    if pd.isna(margin) or margin < margin_threshold:
        return 0, None

    m20 = row.get("momentum_20h", np.nan)
    adx = row.get("adx_14", np.nan)
    close, ema20, atr = row.get("close", np.nan), row.get("ema_20", np.nan), row.get("atr_14", np.nan)

    if any(pd.isna(x) for x in [m20, adx, close, ema20, atr]) or atr <= 0:
        return 0, None

    if regime == "Trending":
        if m20 > 0 and adx >= adx_threshold and close > ema20:
            return 1, regime
        if m20 < 0 and adx >= adx_threshold and close < ema20:
            return -1, regime
        return 0, None

    if regime == "Mean Reversion":
        if close > (ema20 + atr):
            return -1, regime
        if close < (ema20 - atr):
            return 1, regime
        return 0, None

    return 0, None  # High Volatility / Event: no entries


# =============================================================================
# 3. BACKTEST ENGINE (next-bar fill, ATR stop/TP, regime-flip exit, time exit)
# =============================================================================

def run_backtest(df, symbol, confidence_threshold, adx_threshold, margin_threshold,
                  meta_model=None, meta_scaler=None, meta_threshold=None,
                  starting_equity=None):
    trades, equity_curve = [], []
    equity = starting_equity if starting_equity is not None else INITIAL_EQUITY
    position = None
    n = len(df)
    i = 0

    def sig(row):
        direction, regime = generate_primary_signal(row, confidence_threshold, adx_threshold, margin_threshold)
        if direction == 0 or meta_model is None:
            return direction, regime
        feat = build_meta_feature_row(row, direction)
        if feat is None:
            return 0, None
        x = meta_scaler.transform([feat])
        p_win = meta_model.predict_proba(x)[0, 1]
        if p_win < meta_threshold:
            return 0, None
        return direction, regime

    while i < n:
        row = df.iloc[i]
        current_time = row["open_time"]

        if position is None:
            direction, regime = sig(row)
            if direction != 0 and i + 1 < n:
                entry_row = df.iloc[i + 1]
                entry_price = float(entry_row["open"])
                atr = float(row["atr_14"])
                if not np.isfinite(atr) or atr <= 0:
                    i += 1
                    continue

                params = REGIME_PARAMS.get(regime, REGIME_PARAMS["Trending"])
                stop_distance = params["stop_atr_mult"] * atr
                risk_capital = equity * RISK_PER_TRADE
                quantity = risk_capital / stop_distance
                max_notional = equity * MAX_NOTIONAL_MULTIPLE
                if quantity * entry_price > max_notional:
                    quantity = max_notional / entry_price
                if not np.isfinite(quantity) or quantity <= 0:
                    i += 1
                    continue

                if direction == 1:
                    stop_price = entry_price - params["stop_atr_mult"] * atr
                    take_profit = entry_price + params["tp_atr_mult"] * atr
                else:
                    stop_price = entry_price + params["stop_atr_mult"] * atr
                    take_profit = entry_price - params["tp_atr_mult"] * atr

                position = {
                    "direction": direction, "entry_index": i + 1, "entry_time": entry_row["open_time"],
                    "entry_price": entry_price, "quantity": quantity, "stop_price": stop_price,
                    "take_profit": take_profit, "horizon_bars": params["horizon_bars"], "predicted_regime": regime,
                }
                i += 1
                continue

        if position is not None:
            direction = position["direction"]
            entry_price = position["entry_price"]
            quantity = position["quantity"]
            stop_price = position["stop_price"]
            take_profit = position["take_profit"]
            holding_bars = i - position["entry_index"] + 1

            current_high, current_low = float(row["high"]), float(row["low"])
            exit_reason, exit_price, stopped_out = None, None, False

            if direction == 1 and current_low <= stop_price:
                exit_reason, exit_price, stopped_out = "stop_loss", stop_price, True
            elif direction == -1 and current_high >= stop_price:
                exit_reason, exit_price, stopped_out = "stop_loss", stop_price, True

            if exit_reason is None:
                if direction == 1 and current_high >= take_profit:
                    exit_reason, exit_price = "take_profit", take_profit
                elif direction == -1 and current_low <= take_profit:
                    exit_reason, exit_price = "take_profit", take_profit

            if exit_reason is None and i + 1 < n and holding_bars >= MIN_BARS_BEFORE_FLIP:
                current_signal, _ = sig(row)
                if current_signal != 0 and current_signal != direction:
                    next_row = df.iloc[i + 1]
                    exit_reason, exit_price = "regime_or_direction_flip", float(next_row["open"])

            if exit_reason is None and holding_bars >= position["horizon_bars"]:
                exit_reason, exit_price = "time_exit", float(row["close"])

            if exit_reason is not None:
                gross_return = direction * (exit_price / entry_price - 1.0)
                cost = 2 * (FEE_RATE + SLIPPAGE_RATE)
                net_return = gross_return - cost
                net_pnl = quantity * entry_price * net_return
                equity += net_pnl
                trades.append({
                    "symbol": symbol, "entry_time": position["entry_time"], "exit_time": row["open_time"],
                    "direction": "LONG" if direction == 1 else "SHORT",
                    "predicted_regime": position["predicted_regime"], "holding_bars": holding_bars,
                    "exit_reason": exit_reason, "stopped_out": stopped_out,
                    "net_return": net_return, "net_pnl": net_pnl,
                })
                position = None

        mtm_equity = equity
        if position is not None:
            current_close = float(row["close"])
            unrealized = position["direction"] * (current_close / position["entry_price"] - 1)
            mtm_equity += position["entry_price"] * position["quantity"] * unrealized
        equity_curve.append({"open_time": current_time, "equity": mtm_equity})
        i += 1

    return pd.DataFrame(trades), pd.DataFrame(equity_curve), equity


def quick_metrics(trades_df, equity_df, base_equity):
    if trades_df.empty:
        return {"total_trades": 0, "win_rate": np.nan, "profit_factor": np.nan,
                "max_drawdown": 0.0, "net_return": 0.0}
    wins = (trades_df["net_pnl"] > 0).sum()
    total_trades = len(trades_df)
    win_rate = wins / total_trades
    gross_profit = trades_df.loc[trades_df["net_pnl"] > 0, "net_pnl"].sum()
    gross_loss = abs(trades_df.loc[trades_df["net_pnl"] < 0, "net_pnl"].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf
    eq = equity_df["equity"].astype(float)
    drawdown = (eq - eq.cummax()) / eq.cummax()
    max_drawdown = drawdown.min()
    net_return = trades_df["net_pnl"].sum() / base_equity
    return {"total_trades": total_trades, "win_rate": win_rate, "profit_factor": profit_factor,
            "max_drawdown": max_drawdown, "net_return": net_return}


# =============================================================================
# 4. META-LABELING
# =============================================================================

def build_meta_feature_row(row, direction):
    vals = []
    for f in META_FEATURES:
        if f == "direction_code":
            vals.append(direction)
        else:
            v = row.get(f, np.nan)
            if pd.isna(v):
                return None
            vals.append(v)
    return vals


def mine_candidate_trades(df, symbol):
    trades, _, _ = run_backtest(df, symbol, LOOSE_CONFIDENCE, LOOSE_ADX, LOOSE_MARGIN,
                                 starting_equity=PER_SYMBOL_STARTING_CAPITAL)
    return trades


def build_meta_training_set(df, symbol):
    records = []
    n = len(df)
    for i in range(n - 1):
        row = df.iloc[i]
        direction, regime = generate_primary_signal(row, LOOSE_CONFIDENCE, LOOSE_ADX, LOOSE_MARGIN)
        if direction == 0:
            continue
        feat = build_meta_feature_row(row, direction)
        if feat is None:
            continue
        records.append((i, feat))
    if not records:
        return None, None

    trades_df = mine_candidate_trades(df, symbol)
    if trades_df.empty:
        return None, None

    feats_ordered = [r[1] for r in records]
    k = min(len(feats_ordered), len(trades_df))
    X = np.array(feats_ordered[:k])
    y = (trades_df["net_pnl"].values[:k] > 0).astype(int)
    return X, y


def train_meta_model(X, y):
    if len(np.unique(y)) < 2 or len(y) < 40:
        return None, None
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    if HAS_LGBM:
        model = LGBMClassifier(n_estimators=150, max_depth=4, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=-1)
    else:
        model = GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42)
    model.fit(Xs, y)
    return model, scaler


# =============================================================================
# 5. WALK-FORWARD FOLDS
# =============================================================================

def compute_fold_geometry(n_rows):
    for scale in (1.0, 0.75, 0.6, 0.5, 0.4, 0.3, 0.2, 0.15, 0.1):
        train_bars = max(MIN_TRAIN_BARS, int(TRAIN_BARS * scale))
        test_bars = max(MIN_TEST_BARS, int(TEST_BARS * scale))
        for n_folds in range(N_FOLDS, MIN_FOLDS - 1, -1):
            required = train_bars + EMBARGO_BARS + test_bars * n_folds
            if required <= n_rows:
                return train_bars, test_bars, n_folds
    return None, None, 0


def make_folds(n_rows, train_bars, test_bars, n_folds):
    folds = []
    start = 0
    while True:
        train_end = start + train_bars
        test_start = train_end + EMBARGO_BARS
        test_end = test_start + test_bars
        if test_end > n_rows:
            break
        folds.append((start, train_end, test_start, test_end))
        start += test_bars
        if len(folds) >= n_folds:
            break
    return folds


# =============================================================================
# 6. STAGE A: WALK-FORWARD THRESHOLD SELECTION
# =============================================================================

def optimize_eth():
    print(f"\n{'='*90}\n{SYMBOL} -- WALK-FORWARD THRESHOLD SELECTION (2020-2025)\n{'='*90}")
    df = prepare_data(SYMBOL)
    print(f"  Usable bars: {len(df):,} ({df['open_time'].min()} -> {df['open_time'].max()})")

    train_bars, test_bars, n_folds = compute_fold_geometry(len(df))
    if n_folds == 0:
        print("  Not enough history even at minimum geometry -- aborting.")
        return None
    if (train_bars, test_bars, n_folds) != (TRAIN_BARS, TEST_BARS, N_FOLDS):
        print(f"  History shorter than default geometry -- using "
              f"train={train_bars} test={test_bars} n_folds={n_folds} bars instead.")

    folds = make_folds(len(df), train_bars, test_bars, n_folds)
    if not folds:
        print("  Fold construction failed -- aborting.")
        return None

    config_grid = list(product(CONFIDENCE_GRID, ADX_GRID, MARGIN_GRID, META_THRESHOLD_GRID))
    per_config_fold_metrics = {cfg: [] for cfg in config_grid}
    fold_rows = []

    for fold_i, (tr_s, tr_e, te_s, te_e) in enumerate(folds):
        train_df = df.iloc[tr_s:tr_e].reset_index(drop=True)
        test_df = df.iloc[te_s:te_e].reset_index(drop=True)

        X, y = build_meta_training_set(train_df, SYMBOL)
        n_candidates = 0 if X is None else len(y)
        n_wins = 0 if X is None else int(y.sum())
        meta_model, meta_scaler = (train_meta_model(X, y) if X is not None else (None, None))
        if meta_model is None:
            print(f"  Fold {fold_i}: only {n_candidates} mined candidate trades "
                  f"({n_wins} wins) -- need >=40 with both classes present, skipping fold.")
            continue
        print(f"  Fold {fold_i}: trained meta-model on {n_candidates} candidate trades "
              f"({n_wins} wins, {n_candidates - n_wins} losses).")

        for cfg in config_grid:
            conf_t, adx_t, margin_t, meta_t = cfg
            trades_df, equity_df, _ = run_backtest(
                test_df, SYMBOL, conf_t, adx_t, margin_t,
                meta_model=meta_model, meta_scaler=meta_scaler, meta_threshold=meta_t,
                starting_equity=PER_SYMBOL_STARTING_CAPITAL,
            )
            m = quick_metrics(trades_df, equity_df, PER_SYMBOL_STARTING_CAPITAL)
            m["fold"] = fold_i
            per_config_fold_metrics[cfg].append(m)
            fold_rows.append({**m, "confidence_threshold": conf_t, "adx_threshold": adx_t,
                               "margin_threshold": margin_t, "meta_threshold": meta_t})

    if not fold_rows:
        print("  No folds produced results (meta-model training failed everywhere).")
        return None

    fold_results_df = pd.DataFrame(fold_rows)
    fold_results_df.to_csv(OUTPUT_DIR / f"{SYMBOL}_fold_results.csv", index=False)

    agg_rows = []
    for cfg, fold_list in per_config_fold_metrics.items():
        if not fold_list:
            continue
        wr = [m["win_rate"] for m in fold_list if m["total_trades"] > 0]
        pf = [m["profit_factor"] for m in fold_list if m["total_trades"] > 0 and np.isfinite(m["profit_factor"])]
        dd = [m["max_drawdown"] for m in fold_list]
        total_trades = sum(m["total_trades"] for m in fold_list)
        net_return = sum(m["net_return"] for m in fold_list)
        n_folds_with_trades = len(wr)
        n_folds_passing = sum(1 for w, p in zip(wr, pf) if w >= MIN_WIN_RATE and p >= MIN_PROFIT_FACTOR) if pf else 0

        agg_rows.append({
            "confidence_threshold": cfg[0], "adx_threshold": cfg[1],
            "margin_threshold": cfg[2], "meta_threshold": cfg[3],
            "total_trades": total_trades,
            "median_win_rate": np.median(wr) if wr else np.nan,
            "median_profit_factor": np.median(pf) if pf else np.nan,
            "median_max_drawdown": np.median(dd) if dd else np.nan,
            "net_return_sum": net_return,
            "n_folds_with_trades": n_folds_with_trades,
            "n_folds_passing": n_folds_passing,
            "fold_pass_fraction": (n_folds_passing / n_folds_with_trades) if n_folds_with_trades else 0.0,
        })

    agg_df = pd.DataFrame(agg_rows)
    robust = agg_df[
        (agg_df["total_trades"] >= MIN_TRADES_PER_FOLD_TOTAL)
        & (agg_df["fold_pass_fraction"] >= FOLD_PASS_FRACTION)
        & (agg_df["median_win_rate"] >= MIN_WIN_RATE)
        & (agg_df["median_profit_factor"] >= MIN_PROFIT_FACTOR)
    ].copy()

    if robust.empty:
        best_overall = agg_df.sort_values(["fold_pass_fraction", "median_profit_factor"], ascending=False).head(1)
        print(f"  NO ROBUST EDGE FOUND for {SYMBOL}.")
        chosen = best_overall.iloc[0].to_dict() if not best_overall.empty else None
        status = "NO_EDGE"
    else:
        robust["score"] = (
            robust["median_profit_factor"] - 2.0 * robust["median_max_drawdown"].abs() + robust["net_return_sum"]
        )
        robust = robust.sort_values("score", ascending=False)
        chosen = robust.iloc[0].to_dict()
        status = "EDGE_FOUND"
        print(f"  EDGE FOUND: conf={chosen['confidence_threshold']}, adx={chosen['adx_threshold']}, "
              f"margin={chosen['margin_threshold']}, meta={chosen['meta_threshold']}")
        print(f"  median_win_rate={chosen['median_win_rate']:.2%}  "
              f"median_profit_factor={chosen['median_profit_factor']:.2f}  "
              f"median_max_dd={chosen['median_max_drawdown']:.2%}  "
              f"passed {chosen['n_folds_passing']}/{chosen['n_folds_with_trades']} folds")

    agg_df.to_csv(OUTPUT_DIR / f"{SYMBOL}_config_aggregate.csv", index=False)
    with open(OUTPUT_DIR / f"{SYMBOL}_best_config.json", "w") as f:
        json.dump({"symbol": SYMBOL, "status": status,
                    "config": {k: (None if pd.isna(v) else v) for k, v in (chosen or {}).items()}},
                   f, indent=2, default=str)

    return {"df": df, "folds": folds, "status": status, "config": chosen}


# =============================================================================
# 7. STAGE B: INDIVIDUAL $5,000 PnL ACCOUNT (compounded across OOS folds)
# =============================================================================

def build_pnl_account(stage_a_result):
    if stage_a_result is None or stage_a_result["status"] != "EDGE_FOUND":
        print(f"\n{SYMBOL}: no validated edge -- refusing to build a live-style "
              f"$5,000 PnL account on an unvalidated signal.")
        return None

    df, folds, cfg = stage_a_result["df"], stage_a_result["folds"], stage_a_result["config"]
    conf_t = float(cfg["confidence_threshold"])
    adx_t = float(cfg["adx_threshold"])
    margin_t = float(cfg["margin_threshold"])
    meta_t = float(cfg["meta_threshold"])

    equity = PER_SYMBOL_STARTING_CAPITAL
    all_trades, all_equity = [], []

    for fold_i, (tr_s, tr_e, te_s, te_e) in enumerate(folds):
        train_df = df.iloc[tr_s:tr_e].reset_index(drop=True)
        test_df = df.iloc[te_s:te_e].reset_index(drop=True)

        X, y = build_meta_training_set(train_df, SYMBOL)
        if X is None:
            continue
        meta_model, meta_scaler = train_meta_model(X, y)
        if meta_model is None:
            continue

        trades_df, equity_df, equity = run_backtest(
            test_df, SYMBOL, conf_t, adx_t, margin_t,
            meta_model=meta_model, meta_scaler=meta_scaler, meta_threshold=meta_t,
            starting_equity=equity,   # <-- compounds fold to fold, like a live account
        )
        if not trades_df.empty:
            trades_df = trades_df.copy()
            trades_df["fold"] = fold_i
            all_trades.append(trades_df)
        if not equity_df.empty:
            equity_df = equity_df.copy()
            equity_df["fold"] = fold_i
            all_equity.append(equity_df)

    if not all_trades:
        print(f"  {SYMBOL}: no trades generated across any OOS fold.")
        return None

    trades_df = pd.concat(all_trades, ignore_index=True)
    equity_df = pd.concat(all_equity, ignore_index=True)
    trades_df.to_csv(OUTPUT_DIR / f"{SYMBOL}_FINAL_PNL_TRADES.csv", index=False)
    equity_df.to_csv(OUTPUT_DIR / f"{SYMBOL}_FINAL_EQUITY_CURVE.csv", index=False)

    eq = equity_df["equity"].astype(float)
    running_max = eq.cummax()
    drawdown_pct = (eq - running_max) / running_max
    final_equity = float(eq.iloc[-1])
    net_pnl_usd = final_equity - PER_SYMBOL_STARTING_CAPITAL

    wins = trades_df[trades_df["net_pnl"] > 0]
    losses = trades_df[trades_df["net_pnl"] <= 0]
    gross_profit = wins["net_pnl"].sum()
    gross_loss = -losses["net_pnl"].sum()
    profit_factor = (gross_profit / gross_loss) if gross_loss > 0 else np.nan
    win_rate = len(wins) / len(trades_df) if len(trades_df) else np.nan

    summary = {
        "symbol": SYMBOL,
        "starting_capital_usd": PER_SYMBOL_STARTING_CAPITAL,
        "final_equity_usd": round(final_equity, 2),
        "net_pnl_usd": round(net_pnl_usd, 2),
        "net_return_pct": round(net_pnl_usd / PER_SYMBOL_STARTING_CAPITAL * 100, 2),
        "max_drawdown_pct": round(float(drawdown_pct.min()) * 100, 2),
        "total_trades": int(len(trades_df)),
        "win_rate": round(float(win_rate), 4) if pd.notna(win_rate) else None,
        "profit_factor": round(float(profit_factor), 3) if pd.notna(profit_factor) else None,
    }
    print(f"\n{'='*90}\n{SYMBOL} FINAL $5,000 PnL ACCOUNT (2020-2025, walk-forward, all folds compounded)\n{'='*90}")
    for k, v in summary.items():
        print(f"  {k}: {v}")
    return summary


if __name__ == "__main__":
    # Normalize Stage 7 predictions to the schema expected by prepare_data().
    # Some files only export pred_future_regime_code instead of pred_future_regime.
    pred_path = find_predictions_path(SYMBOL)
    pred_df = pd.read_csv(pred_path)

    needs_backfill = (
        "pred_future_regime" not in pred_df.columns
        or pred_df["pred_future_regime"].isna().all()
    )

    if needs_backfill:
        if "pred_future_regime_code" in pred_df.columns:
            code_to_label = {
                1: "Trending",
                2: "Mean Reversion",
                3: "High Volatility",
                4: "Event",
            }
            pred_df["pred_future_regime"] = (
                pd.to_numeric(pred_df["pred_future_regime_code"], errors="coerce")
                .map(code_to_label)
            )
            pred_df.to_csv(pred_path, index=False)
            print(f"Backfilled missing/empty pred_future_regime in {pred_path.name}")
        else:
            # Some Stage-7 exports use a different schema than the legacy one the rest of the
            # notebook expects (e.g. `predicted_regime`, `prob_trending`, etc.). Normalize the
            # file to the names used below before continuing.
            pred_df = pred_df.copy()

            # --- normalize probability columns ---
            prob_aliases = {
                "prob_trending": "prob_Trending",
                "prob_mean_reversion": "prob_Mean Reversion",
                "prob_high_volatility": "prob_High Volatility",
                "prob_event": "prob_Event",
            }
            for src, dst in prob_aliases.items():
                if src in pred_df.columns and dst not in pred_df.columns:
                    pred_df[dst] = pd.to_numeric(pred_df[src], errors="coerce")

            # --- normalize regime label/code columns ---
            code_to_label = {
                1: "Trending",
                2: "Mean Reversion",
                3: "High Volatility",
                4: "Event",
            }

            if "pred_future_regime" not in pred_df.columns:
                if "pred_future_regime_code" in pred_df.columns:
                    pred_df["pred_future_regime"] = (
                        pd.to_numeric(pred_df["pred_future_regime_code"], errors="coerce")
                        .map(code_to_label)
                    )
                elif "predicted_regime" in pred_df.columns:
                    pred_df["pred_future_regime_code"] = pd.to_numeric(pred_df["predicted_regime"], errors="coerce")
                    pred_df["pred_future_regime"] = pred_df["pred_future_regime_code"].map(code_to_label)
                elif "predicted_regime_label" in pred_df.columns:
                    pred_df["pred_future_regime"] = pred_df["predicted_regime_label"].astype(str).str.strip()
                else:
                    raise KeyError(
                        f"{SYMBOL}: prediction file is missing both 'pred_future_regime' and "
                        "'pred_future_regime_code'. Available columns: {list(pred_df.columns)}"
                    )

            # Backfill code if it exists as a string label but not numeric code
            if "pred_future_regime_code" not in pred_df.columns and "pred_future_regime" in pred_df.columns:
                pred_df["pred_future_regime_code"] = (
                    pred_df["pred_future_regime"].map({v: k for k, v in code_to_label.items()})
                )

            # Safety: if the file already has a confidence column with the expected name, keep it.
            # If it doesn't, try to infer it from the model output if present.
            if "prediction_confidence" not in pred_df.columns and "predicted_regime_confidence" in pred_df.columns:
                pred_df["prediction_confidence"] = pd.to_numeric(pred_df["predicted_regime_confidence"], errors="coerce")

            # Save the normalized schema so future notebook runs don't keep failing.
            pred_df.to_csv(pred_path, index=False)
            print(f"Normalized prediction schema in {pred_path.name}")

    stage_a = optimize_eth()
    if stage_a is not None and stage_a.get("status") == "EDGE_FOUND":
        build_pnl_account(stage_a)
    build_pnl_account(stage_a)


ETHUSDT -- WALK-FORWARD THRESHOLD SELECTION (2020-2025)
  Usable bars: 49,641 (2020-12-31 04:00:00+00:00 -> 2026-09-11 00:00:00+00:00)
  Fold 0: trained meta-model on 378 candidate trades (161 wins, 217 losses).
  Fold 1: trained meta-model on 474 candidate trades (211 wins, 263 losses).
  Fold 2: trained meta-model on 532 candidate trades (227 wins, 305 losses).
  Fold 3: trained meta-model on 549 candidate trades (242 wins, 307 losses).
  Fold 4: trained meta-model on 549 candidate trades (250 wins, 299 losses).
  Fold 5: trained meta-model on 530 candidate trades (235 wins, 295 losses).
  NO ROBUST EDGE FOUND for ETHUSDT.

ETHUSDT: no validated edge -- refusing to build a live-style $5,000 PnL account on an unvalidated signal.


In [11]:
"""
===============================================================================
ETHUSDT WALK-FORWARD BACKTEST  +  INDIVIDUAL $5,000 PnL ACCOUNT (2020-2025)
===============================================================================
Self-contained version of your Stage 9 (meta-labeled walk-forward optimizer)
+ Stage 10 (compounding PnL account), narrowed to ETHUSDT and pointed at the
files you actually have on disk right now:

    C:\\Users\\dell\\Downloads\\data\\regimes\\ETHUSDT_07_predictions.csv
    C:\\Users\\dell\\Downloads\\data\\processed\\ETHUSDT_1h_features.csv   <- ASSUMED PATH,
                                                                             see NOTE below

 ON THE FEATURES FILE
----------------------------------------------------------------------------
Your Stage 7 output lives in `regimes/`, but the backtest also needs the raw
OHLCV + engineered columns (open/high/low/close, adx_14, momentum_*, hurst_100,
volume_zscore_20, realized_vol_20) that come from your Stage 1/2 pipeline and
are normally written to `processed/{symbol}_1h_features.csv`. If that file
isn't there anymore, point FEATURE_DIR at wherever it now lives -- this script
will raise a clear error naming the missing columns rather than fail silently.

WHAT THIS SCRIPT DOES
----------------------------------------------------------------------------
1. Loads Stage 5/7 regime predictions + Stage 1/2 features for ETHUSDT only,
   restricted to 2020-01-01 -> 2025-12-31.
2. Builds N rolling walk-forward folds (train -> embargo -> test).
3. Per fold: mines candidate trades at loose thresholds, trains a meta-model
   (win/lose classifier) on the training window only, then grid-searches
   (confidence, ADX, margin, meta) thresholds on the held-out test window.
4. Picks the config that passes the acceptance bar on the most test folds
   (or reports NO_EDGE honestly if none do).
5. Re-runs the OOS folds in order as ONE continuous, isolated $5,000 account
   (STAGE 10 step) -- compounding, all fees + slippage included -- and saves
   the trade blotter, equity curve, and summary metrics.

STRATEGY DEFINITION (see printed summary at the end for the numbers actually
used on your data)
----------------------------------------------------------------------------
- Direction:      regime classifier (Trending / Mean Reversion) + confidence
                  + margin-of-confidence gate + ADX/EMA confluence, then a
                  meta-model veto that only allows the trade if it predicts
                  a win with probability >= meta_threshold.
- Stop-loss:      1.5x ATR(14) in Trending, 1.2x ATR(14) in Mean Reversion.
- Take-profit:    4.0x ATR(14) in Trending, 2.0x ATR(14) in Mean Reversion.
- Time exit:      close after 8 bars (Trending) / 3 bars (Mean Reversion)
                  if neither SL nor TP was hit.
- Position size:  risk_per_trade% of CURRENT account equity / stop-distance,
                  capped so notional <= 1x equity (no leverage).
- Frictions:      taker fee 0.04% + slippage 0.02%, charged on entry AND exit.

WHY THIS DESIGN TARGETS "MAX PROFIT, MIN DRAWDOWN"
----------------------------------------------------------------------------
- ATR-based stops scale with realized volatility, so the stop is neither too
  tight in a volatile stretch (which would just get stopped out by noise)
  nor too loose in a calm stretch (which would give back more than needed).
- Fixed-fractional sizing means no single trade can damage the account by
  more than a fixed % of ITS OWN equity -- this is what bounds max drawdown
  regardless of how large a losing streak gets, and it's why the position
  size shrinks automatically after a drawdown (protecting capital) and grows
  again after a run of wins (compounding profit).
- The meta-model doesn't add profit by predicting the market better in some
  abstract sense -- it adds profit by refusing the subset of primary-rule
  trades that historically lost, which mechanically raises win rate and
  profit factor for the SAME rule.
- Walk-forward + embargo means the chosen thresholds had to survive being
  tested on data the model never trained on, across multiple folds -- this
  is what stops a config that looks great on one lucky slice of history
  from being deployed on your real $5,000.

===============================================================================
"""

import json
import warnings
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression  # noqa: F401 (kept for parity)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    HAS_LGBM = False


# =============================================================================
# 0. CONFIG -- EDIT FOR YOUR MACHINE
# =============================================================================

SYMBOL = "BTCUSDT"

BASE_DIR = Path(r"C:\Users\dell\Downloads\data")
PREDICTION_DIR = BASE_DIR / "regimes"      # <- where your ETHUSDT_07_predictions.csv lives
FEATURE_DIR = BASE_DIR / "processed"       # <- where ETHUSDT_1h_features.csv lives (check this!)
OUTPUT_DIR = BASE_DIR / "backtest" / "eth_walkforward"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_FILENAME_PATTERNS = [
    "{symbol}_07_predictions.csv",
    "{symbol}_07_lightgbm_predictions.csv",
]

BACKTEST_START = pd.Timestamp("2019-01-01", tz="UTC")
BACKTEST_END = pd.Timestamp("2026-09-11", tz="UTC")

PER_SYMBOL_STARTING_CAPITAL = 5_000.0     # isolated account size, as requested
INITIAL_EQUITY = PER_SYMBOL_STARTING_CAPITAL   # mutated during compounding, see Stage 10 loop

RISK_PER_TRADE = 0.005          # 0.5% of CURRENT equity risked per trade
FEE_RATE = 0.0004               # 0.04% taker fee
SLIPPAGE_RATE = 0.0002          # 0.02% slippage
MAX_NOTIONAL_MULTIPLE = 1.0     # no leverage: notional <= 1x equity
MIN_BARS_BEFORE_FLIP = 2

# --- walk-forward fold geometry (1h bars) ---
N_FOLDS = 6
TRAIN_BARS = 24 * 240        # 240 days train
TEST_BARS = 24 * 45          # 45 days test
EMBARGO_BARS = 8             # >= max horizon_bars, prevents label leakage
MIN_TRAIN_BARS = 24 * 90
MIN_TEST_BARS = 24 * 15
MIN_FOLDS = 3

# --- loose thresholds used ONLY to mine candidate trades for meta-labeling ---
LOOSE_CONFIDENCE = 0.50
LOOSE_ADX = 15.0
LOOSE_MARGIN = 0.0

# --- grid searched on top of the meta-model ---
CONFIDENCE_GRID = (0.55, 0.60, 0.65, 0.70)
ADX_GRID = (20.0, 25.0, 30.0)
MARGIN_GRID = (0.05, 0.10, 0.15)
META_THRESHOLD_GRID = (0.50, 0.55, 0.60, 0.65)

# --- acceptance bar ---
MIN_WIN_RATE = 0.52
MIN_PROFIT_FACTOR = 1.0
MIN_TRADES_PER_FOLD_TOTAL = 30
FOLD_PASS_FRACTION = 0.60

REGIME_PARAMS = {
    "Trending": {"stop_atr_mult": 1.5, "tp_atr_mult": 4.0, "horizon_bars": 8},
    "Mean Reversion": {"stop_atr_mult": 1.2, "tp_atr_mult": 2.0, "horizon_bars": 3},
}

PROB_COLS = ["prob_Trending", "prob_Mean Reversion", "prob_High Volatility", "prob_Event"]

FEATURE_ALIASES = {
    "momentum_5": "momentum_5h", "momentum_20": "momentum_20h", "momentum_60": "momentum_60h",
    "log_realized_vol_20": "realized_vol_20", "log_atr_pct": "atr_pct",
}

META_FEATURES = [
    "prediction_confidence", "prediction_margin", "adx_14", "atr_pct",
    "hurst_100", "momentum_5h", "momentum_20h", "momentum_60h",
    "volume_zscore_20", "realized_vol_20", "ema_distance_atr", "vol_regime_z",
    "direction_code",
]


# =============================================================================
# 1. LOADERS / FEATURE PREP
# =============================================================================

def find_predictions_path(symbol):
    for pattern in PREDICTIONS_FILENAME_PATTERNS:
        p = PREDICTION_DIR / pattern.format(symbol=symbol)
        if p.exists():
            return p
    raise FileNotFoundError(f"{symbol}: no Stage 7 prediction file found in {PREDICTION_DIR}")


def calculate_atr(df, period=14):
    high, low, close = df["high"], df["low"], df["close"]
    prev_close = close.shift(1)
    tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
    return tr.rolling(period, min_periods=period).mean()


def prediction_margin(row):
    probs = sorted((pd.to_numeric(row.get(c, np.nan), errors="coerce") for c in PROB_COLS), reverse=True)
    probs = [p for p in probs if pd.notna(p)]
    return probs[0] - probs[1] if len(probs) >= 2 else np.nan


def prepare_data(symbol):
    stage7 = pd.read_csv(find_predictions_path(symbol))
    stage7["open_time"] = pd.to_datetime(stage7["open_time"], utc=True)
    stage7 = stage7.sort_values("open_time").drop_duplicates("open_time").reset_index(drop=True)

    feat_path = FEATURE_DIR / f"{symbol}_1h_features.csv"
    if not feat_path.exists():
        raise FileNotFoundError(
            f"Features file not found: {feat_path}\n"
            f"Update FEATURE_DIR at the top of this script to wherever "
            f"{symbol}_1h_features.csv actually lives."
        )
    features = pd.read_csv(feat_path)
    features["open_time"] = pd.to_datetime(features["open_time"], utc=True)
    features = features.sort_values("open_time").drop_duplicates("open_time").reset_index(drop=True)
    features = features.rename(columns=FEATURE_ALIASES)

    df = pd.merge(stage7, features, on="open_time", how="inner", suffixes=("", "_feature"))
    df = df.sort_values("open_time").reset_index(drop=True)

    # restrict to the requested 2020-2025 walk-forward window
    df = df[(df["open_time"] >= BACKTEST_START) & (df["open_time"] <= BACKTEST_END)].reset_index(drop=True)

    df["atr_14"] = calculate_atr(df, 14)
    df["ema_20"] = df["close"].ewm(span=20, adjust=False, min_periods=20).mean()

    code_to_label = {1: "Trending", 2: "Mean Reversion", 3: "High Volatility", 4: "Event"}
    missing_label = df["pred_future_regime"].isna()
    if missing_label.any():
        df.loc[missing_label, "pred_future_regime"] = (
            pd.to_numeric(df.loc[missing_label, "pred_future_regime_code"], errors="coerce").map(code_to_label)
        )

    prob_map = {"Trending": "prob_Trending", "Mean Reversion": "prob_Mean Reversion",
                "High Volatility": "prob_High Volatility", "Event": "prob_Event"}
    df["prediction_confidence"] = np.nan
    for regime, col in prob_map.items():
        mask = df["pred_future_regime"].astype(str).eq(regime)
        df.loc[mask, "prediction_confidence"] = pd.to_numeric(df.loc[mask, col], errors="coerce")

    df["prediction_margin"] = df.apply(prediction_margin, axis=1)

    for canonical, alts in {
        "momentum_5h": ["momentum_5h", "momentum_5"],
        "momentum_20h": ["momentum_20h", "momentum_20"],
        "momentum_60h": ["momentum_60h", "momentum_60"],
        "atr_pct": ["atr_pct", "log_atr_pct"],
        "realized_vol_20": ["realized_vol_20", "log_realized_vol_20"],
    }.items():
        if canonical not in df.columns:
            for alt in alts:
                if alt in df.columns:
                    df[canonical] = df[alt]
                    break

    df["ema_distance_atr"] = (df["close"] - df["ema_20"]) / df["atr_14"].replace(0, np.nan)

    rv = df["realized_vol_20"]
    df["vol_regime_z"] = (rv - rv.rolling(500, min_periods=100).mean()) / rv.rolling(500, min_periods=100).std()

    numeric_cols = ["open", "high", "low", "close", "atr_14", "ema_20", "adx_14",
                     "momentum_5h", "momentum_20h", "momentum_60h", "hurst_100",
                     "volume_zscore_20", "realized_vol_20", "prediction_confidence"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.sort_values("open_time").reset_index(drop=True)

    required_meta_cols = [f for f in META_FEATURES if f != "direction_code"]
    missing = [c for c in required_meta_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"{symbol}: missing columns needed for meta-labeling features: {missing}. "
            f"Available columns: {sorted(df.columns.tolist())}"
        )
    return df


# =============================================================================
# 2. PRIMARY SIGNAL
# =============================================================================

def generate_primary_signal(row, confidence_threshold, adx_threshold, margin_threshold):
    regime = row["pred_future_regime"]
    confidence = row["prediction_confidence"]
    margin = row.get("prediction_margin", np.nan)

    if pd.isna(confidence) or confidence < confidence_threshold:
        return 0, None
    if pd.isna(margin) or margin < margin_threshold:
        return 0, None

    m20 = row.get("momentum_20h", np.nan)
    adx = row.get("adx_14", np.nan)
    close, ema20, atr = row.get("close", np.nan), row.get("ema_20", np.nan), row.get("atr_14", np.nan)

    if any(pd.isna(x) for x in [m20, adx, close, ema20, atr]) or atr <= 0:
        return 0, None

    if regime == "Trending":
        if m20 > 0 and adx >= adx_threshold and close > ema20:
            return 1, regime
        if m20 < 0 and adx >= adx_threshold and close < ema20:
            return -1, regime
        return 0, None

    if regime == "Mean Reversion":
        if close > (ema20 + atr):
            return -1, regime
        if close < (ema20 - atr):
            return 1, regime
        return 0, None

    return 0, None  # High Volatility / Event: no entries


# =============================================================================
# 3. BACKTEST ENGINE (next-bar fill, ATR stop/TP, regime-flip exit, time exit)
# =============================================================================

def run_backtest(df, symbol, confidence_threshold, adx_threshold, margin_threshold,
                  meta_model=None, meta_scaler=None, meta_threshold=None,
                  starting_equity=None):
    trades, equity_curve = [], []
    equity = starting_equity if starting_equity is not None else INITIAL_EQUITY
    position = None
    n = len(df)
    i = 0

    def sig(row):
        direction, regime = generate_primary_signal(row, confidence_threshold, adx_threshold, margin_threshold)
        if direction == 0 or meta_model is None:
            return direction, regime
        feat = build_meta_feature_row(row, direction)
        if feat is None:
            return 0, None
        x = meta_scaler.transform([feat])
        p_win = meta_model.predict_proba(x)[0, 1]
        if p_win < meta_threshold:
            return 0, None
        return direction, regime

    while i < n:
        row = df.iloc[i]
        current_time = row["open_time"]

        if position is None:
            direction, regime = sig(row)
            if direction != 0 and i + 1 < n:
                entry_row = df.iloc[i + 1]
                entry_price = float(entry_row["open"])
                atr = float(row["atr_14"])
                if not np.isfinite(atr) or atr <= 0:
                    i += 1
                    continue

                params = REGIME_PARAMS.get(regime, REGIME_PARAMS["Trending"])
                stop_distance = params["stop_atr_mult"] * atr
                risk_capital = equity * RISK_PER_TRADE
                quantity = risk_capital / stop_distance
                max_notional = equity * MAX_NOTIONAL_MULTIPLE
                if quantity * entry_price > max_notional:
                    quantity = max_notional / entry_price
                if not np.isfinite(quantity) or quantity <= 0:
                    i += 1
                    continue

                if direction == 1:
                    stop_price = entry_price - params["stop_atr_mult"] * atr
                    take_profit = entry_price + params["tp_atr_mult"] * atr
                else:
                    stop_price = entry_price + params["stop_atr_mult"] * atr
                    take_profit = entry_price - params["tp_atr_mult"] * atr

                position = {
                    "direction": direction, "entry_index": i + 1, "entry_time": entry_row["open_time"],
                    "entry_price": entry_price, "quantity": quantity, "stop_price": stop_price,
                    "take_profit": take_profit, "horizon_bars": params["horizon_bars"], "predicted_regime": regime,
                }
                i += 1
                continue

        if position is not None:
            direction = position["direction"]
            entry_price = position["entry_price"]
            quantity = position["quantity"]
            stop_price = position["stop_price"]
            take_profit = position["take_profit"]
            holding_bars = i - position["entry_index"] + 1

            current_high, current_low = float(row["high"]), float(row["low"])
            exit_reason, exit_price, stopped_out = None, None, False

            if direction == 1 and current_low <= stop_price:
                exit_reason, exit_price, stopped_out = "stop_loss", stop_price, True
            elif direction == -1 and current_high >= stop_price:
                exit_reason, exit_price, stopped_out = "stop_loss", stop_price, True

            if exit_reason is None:
                if direction == 1 and current_high >= take_profit:
                    exit_reason, exit_price = "take_profit", take_profit
                elif direction == -1 and current_low <= take_profit:
                    exit_reason, exit_price = "take_profit", take_profit

            if exit_reason is None and i + 1 < n and holding_bars >= MIN_BARS_BEFORE_FLIP:
                current_signal, _ = sig(row)
                if current_signal != 0 and current_signal != direction:
                    next_row = df.iloc[i + 1]
                    exit_reason, exit_price = "regime_or_direction_flip", float(next_row["open"])

            if exit_reason is None and holding_bars >= position["horizon_bars"]:
                exit_reason, exit_price = "time_exit", float(row["close"])

            if exit_reason is not None:
                gross_return = direction * (exit_price / entry_price - 1.0)
                cost = 2 * (FEE_RATE + SLIPPAGE_RATE)
                net_return = gross_return - cost
                net_pnl = quantity * entry_price * net_return
                equity += net_pnl
                trades.append({
                    "symbol": symbol, "entry_time": position["entry_time"], "exit_time": row["open_time"],
                    "direction": "LONG" if direction == 1 else "SHORT",
                    "predicted_regime": position["predicted_regime"], "holding_bars": holding_bars,
                    "exit_reason": exit_reason, "stopped_out": stopped_out,
                    "net_return": net_return, "net_pnl": net_pnl,
                })
                position = None

        mtm_equity = equity
        if position is not None:
            current_close = float(row["close"])
            unrealized = position["direction"] * (current_close / position["entry_price"] - 1)
            mtm_equity += position["entry_price"] * position["quantity"] * unrealized
        equity_curve.append({"open_time": current_time, "equity": mtm_equity})
        i += 1

    return pd.DataFrame(trades), pd.DataFrame(equity_curve), equity


def quick_metrics(trades_df, equity_df, base_equity):
    if trades_df.empty:
        return {"total_trades": 0, "win_rate": np.nan, "profit_factor": np.nan,
                "max_drawdown": 0.0, "net_return": 0.0}
    wins = (trades_df["net_pnl"] > 0).sum()
    total_trades = len(trades_df)
    win_rate = wins / total_trades
    gross_profit = trades_df.loc[trades_df["net_pnl"] > 0, "net_pnl"].sum()
    gross_loss = abs(trades_df.loc[trades_df["net_pnl"] < 0, "net_pnl"].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf
    eq = equity_df["equity"].astype(float)
    drawdown = (eq - eq.cummax()) / eq.cummax()
    max_drawdown = drawdown.min()
    net_return = trades_df["net_pnl"].sum() / base_equity
    return {"total_trades": total_trades, "win_rate": win_rate, "profit_factor": profit_factor,
            "max_drawdown": max_drawdown, "net_return": net_return}


# =============================================================================
# 4. META-LABELING
# =============================================================================

def build_meta_feature_row(row, direction):
    vals = []
    for f in META_FEATURES:
        if f == "direction_code":
            vals.append(direction)
        else:
            v = row.get(f, np.nan)
            if pd.isna(v):
                return None
            vals.append(v)
    return vals


def mine_candidate_trades(df, symbol):
    trades, _, _ = run_backtest(df, symbol, LOOSE_CONFIDENCE, LOOSE_ADX, LOOSE_MARGIN,
                                 starting_equity=PER_SYMBOL_STARTING_CAPITAL)
    return trades


def build_meta_training_set(df, symbol):
    records = []
    n = len(df)
    for i in range(n - 1):
        row = df.iloc[i]
        direction, regime = generate_primary_signal(row, LOOSE_CONFIDENCE, LOOSE_ADX, LOOSE_MARGIN)
        if direction == 0:
            continue
        feat = build_meta_feature_row(row, direction)
        if feat is None:
            continue
        records.append((i, feat))
    if not records:
        return None, None

    trades_df = mine_candidate_trades(df, symbol)
    if trades_df.empty:
        return None, None

    feats_ordered = [r[1] for r in records]
    k = min(len(feats_ordered), len(trades_df))
    X = np.array(feats_ordered[:k])
    y = (trades_df["net_pnl"].values[:k] > 0).astype(int)
    return X, y


def train_meta_model(X, y):
    if len(np.unique(y)) < 2 or len(y) < 40:
        return None, None
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    if HAS_LGBM:
        model = LGBMClassifier(n_estimators=150, max_depth=4, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=-1)
    else:
        model = GradientBoostingClassifier(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42)
    model.fit(Xs, y)
    return model, scaler


# =============================================================================
# 5. WALK-FORWARD FOLDS
# =============================================================================

def compute_fold_geometry(n_rows):
    for scale in (1.0, 0.75, 0.6, 0.5, 0.4, 0.3, 0.2, 0.15, 0.1):
        train_bars = max(MIN_TRAIN_BARS, int(TRAIN_BARS * scale))
        test_bars = max(MIN_TEST_BARS, int(TEST_BARS * scale))
        for n_folds in range(N_FOLDS, MIN_FOLDS - 1, -1):
            required = train_bars + EMBARGO_BARS + test_bars * n_folds
            if required <= n_rows:
                return train_bars, test_bars, n_folds
    return None, None, 0


def make_folds(n_rows, train_bars, test_bars, n_folds):
    folds = []
    start = 0
    while True:
        train_end = start + train_bars
        test_start = train_end + EMBARGO_BARS
        test_end = test_start + test_bars
        if test_end > n_rows:
            break
        folds.append((start, train_end, test_start, test_end))
        start += test_bars
        if len(folds) >= n_folds:
            break
    return folds


# =============================================================================
# 6. STAGE A: WALK-FORWARD THRESHOLD SELECTION
# =============================================================================

def optimize_eth():
    print(f"\n{'='*90}\n{SYMBOL} -- WALK-FORWARD THRESHOLD SELECTION (2020-2025)\n{'='*90}")
    df = prepare_data(SYMBOL)
    print(f"  Usable bars: {len(df):,} ({df['open_time'].min()} -> {df['open_time'].max()})")

    train_bars, test_bars, n_folds = compute_fold_geometry(len(df))
    if n_folds == 0:
        print("  Not enough history even at minimum geometry -- aborting.")
        return None
    if (train_bars, test_bars, n_folds) != (TRAIN_BARS, TEST_BARS, N_FOLDS):
        print(f"  History shorter than default geometry -- using "
              f"train={train_bars} test={test_bars} n_folds={n_folds} bars instead.")

    folds = make_folds(len(df), train_bars, test_bars, n_folds)
    if not folds:
        print("  Fold construction failed -- aborting.")
        return None

    config_grid = list(product(CONFIDENCE_GRID, ADX_GRID, MARGIN_GRID, META_THRESHOLD_GRID))
    per_config_fold_metrics = {cfg: [] for cfg in config_grid}
    fold_rows = []

    for fold_i, (tr_s, tr_e, te_s, te_e) in enumerate(folds):
        train_df = df.iloc[tr_s:tr_e].reset_index(drop=True)
        test_df = df.iloc[te_s:te_e].reset_index(drop=True)

        X, y = build_meta_training_set(train_df, SYMBOL)
        n_candidates = 0 if X is None else len(y)
        n_wins = 0 if X is None else int(y.sum())
        meta_model, meta_scaler = (train_meta_model(X, y) if X is not None else (None, None))
        if meta_model is None:
            print(f"  Fold {fold_i}: only {n_candidates} mined candidate trades "
                  f"({n_wins} wins) -- need >=40 with both classes present, skipping fold.")
            continue
        print(f"  Fold {fold_i}: trained meta-model on {n_candidates} candidate trades "
              f"({n_wins} wins, {n_candidates - n_wins} losses).")

        for cfg in config_grid:
            conf_t, adx_t, margin_t, meta_t = cfg
            trades_df, equity_df, _ = run_backtest(
                test_df, SYMBOL, conf_t, adx_t, margin_t,
                meta_model=meta_model, meta_scaler=meta_scaler, meta_threshold=meta_t,
                starting_equity=PER_SYMBOL_STARTING_CAPITAL,
            )
            m = quick_metrics(trades_df, equity_df, PER_SYMBOL_STARTING_CAPITAL)
            m["fold"] = fold_i
            per_config_fold_metrics[cfg].append(m)
            fold_rows.append({**m, "confidence_threshold": conf_t, "adx_threshold": adx_t,
                               "margin_threshold": margin_t, "meta_threshold": meta_t})

    if not fold_rows:
        print("  No folds produced results (meta-model training failed everywhere).")
        return None

    fold_results_df = pd.DataFrame(fold_rows)
    fold_results_df.to_csv(OUTPUT_DIR / f"{SYMBOL}_fold_results.csv", index=False)

    agg_rows = []
    for cfg, fold_list in per_config_fold_metrics.items():
        if not fold_list:
            continue
        wr = [m["win_rate"] for m in fold_list if m["total_trades"] > 0]
        pf = [m["profit_factor"] for m in fold_list if m["total_trades"] > 0 and np.isfinite(m["profit_factor"])]
        dd = [m["max_drawdown"] for m in fold_list]
        total_trades = sum(m["total_trades"] for m in fold_list)
        net_return = sum(m["net_return"] for m in fold_list)
        n_folds_with_trades = len(wr)
        n_folds_passing = sum(1 for w, p in zip(wr, pf) if w >= MIN_WIN_RATE and p >= MIN_PROFIT_FACTOR) if pf else 0

        agg_rows.append({
            "confidence_threshold": cfg[0], "adx_threshold": cfg[1],
            "margin_threshold": cfg[2], "meta_threshold": cfg[3],
            "total_trades": total_trades,
            "median_win_rate": np.median(wr) if wr else np.nan,
            "median_profit_factor": np.median(pf) if pf else np.nan,
            "median_max_drawdown": np.median(dd) if dd else np.nan,
            "net_return_sum": net_return,
            "n_folds_with_trades": n_folds_with_trades,
            "n_folds_passing": n_folds_passing,
            "fold_pass_fraction": (n_folds_passing / n_folds_with_trades) if n_folds_with_trades else 0.0,
        })

    agg_df = pd.DataFrame(agg_rows)
    robust = agg_df[
        (agg_df["total_trades"] >= MIN_TRADES_PER_FOLD_TOTAL)
        & (agg_df["fold_pass_fraction"] >= FOLD_PASS_FRACTION)
        & (agg_df["median_win_rate"] >= MIN_WIN_RATE)
        & (agg_df["median_profit_factor"] >= MIN_PROFIT_FACTOR)
    ].copy()

    if robust.empty:
        best_overall = agg_df.sort_values(["fold_pass_fraction", "median_profit_factor"], ascending=False).head(1)
        print(f"  NO ROBUST EDGE FOUND for {SYMBOL}.")
        chosen = best_overall.iloc[0].to_dict() if not best_overall.empty else None
        status = "NO_EDGE"
    else:
        robust["score"] = (
            robust["median_profit_factor"] - 2.0 * robust["median_max_drawdown"].abs() + robust["net_return_sum"]
        )
        robust = robust.sort_values("score", ascending=False)
        chosen = robust.iloc[0].to_dict()
        status = "EDGE_FOUND"
        print(f"  EDGE FOUND: conf={chosen['confidence_threshold']}, adx={chosen['adx_threshold']}, "
              f"margin={chosen['margin_threshold']}, meta={chosen['meta_threshold']}")
        print(f"  median_win_rate={chosen['median_win_rate']:.2%}  "
              f"median_profit_factor={chosen['median_profit_factor']:.2f}  "
              f"median_max_dd={chosen['median_max_drawdown']:.2%}  "
              f"passed {chosen['n_folds_passing']}/{chosen['n_folds_with_trades']} folds")

    agg_df.to_csv(OUTPUT_DIR / f"{SYMBOL}_config_aggregate.csv", index=False)
    with open(OUTPUT_DIR / f"{SYMBOL}_best_config.json", "w") as f:
        json.dump({"symbol": SYMBOL, "status": status,
                    "config": {k: (None if pd.isna(v) else v) for k, v in (chosen or {}).items()}},
                   f, indent=2, default=str)

    return {"df": df, "folds": folds, "status": status, "config": chosen}


# =============================================================================
# 7. STAGE B: INDIVIDUAL $5,000 PnL ACCOUNT (compounded across OOS folds)
# =============================================================================

def build_pnl_account(stage_a_result):
    if stage_a_result is None or stage_a_result["status"] != "EDGE_FOUND":
        print(f"\n{SYMBOL}: no validated edge -- refusing to build a live-style "
              f"$5,000 PnL account on an unvalidated signal.")
        return None

    df, folds, cfg = stage_a_result["df"], stage_a_result["folds"], stage_a_result["config"]
    conf_t = float(cfg["confidence_threshold"])
    adx_t = float(cfg["adx_threshold"])
    margin_t = float(cfg["margin_threshold"])
    meta_t = float(cfg["meta_threshold"])

    equity = PER_SYMBOL_STARTING_CAPITAL
    all_trades, all_equity = [], []

    for fold_i, (tr_s, tr_e, te_s, te_e) in enumerate(folds):
        train_df = df.iloc[tr_s:tr_e].reset_index(drop=True)
        test_df = df.iloc[te_s:te_e].reset_index(drop=True)

        X, y = build_meta_training_set(train_df, SYMBOL)
        if X is None:
            continue
        meta_model, meta_scaler = train_meta_model(X, y)
        if meta_model is None:
            continue

        trades_df, equity_df, equity = run_backtest(
            test_df, SYMBOL, conf_t, adx_t, margin_t,
            meta_model=meta_model, meta_scaler=meta_scaler, meta_threshold=meta_t,
            starting_equity=equity,   # <-- compounds fold to fold, like a live account
        )
        if not trades_df.empty:
            trades_df = trades_df.copy()
            trades_df["fold"] = fold_i
            all_trades.append(trades_df)
        if not equity_df.empty:
            equity_df = equity_df.copy()
            equity_df["fold"] = fold_i
            all_equity.append(equity_df)

    if not all_trades:
        print(f"  {SYMBOL}: no trades generated across any OOS fold.")
        return None

    trades_df = pd.concat(all_trades, ignore_index=True)
    equity_df = pd.concat(all_equity, ignore_index=True)
    trades_df.to_csv(OUTPUT_DIR / f"{SYMBOL}_FINAL_PNL_TRADES.csv", index=False)
    equity_df.to_csv(OUTPUT_DIR / f"{SYMBOL}_FINAL_EQUITY_CURVE.csv", index=False)

    eq = equity_df["equity"].astype(float)
    running_max = eq.cummax()
    drawdown_pct = (eq - running_max) / running_max
    final_equity = float(eq.iloc[-1])
    net_pnl_usd = final_equity - PER_SYMBOL_STARTING_CAPITAL

    wins = trades_df[trades_df["net_pnl"] > 0]
    losses = trades_df[trades_df["net_pnl"] <= 0]
    gross_profit = wins["net_pnl"].sum()
    gross_loss = -losses["net_pnl"].sum()
    profit_factor = (gross_profit / gross_loss) if gross_loss > 0 else np.nan
    win_rate = len(wins) / len(trades_df) if len(trades_df) else np.nan

    summary = {
        "symbol": SYMBOL,
        "starting_capital_usd": PER_SYMBOL_STARTING_CAPITAL,
        "final_equity_usd": round(final_equity, 2),
        "net_pnl_usd": round(net_pnl_usd, 2),
        "net_return_pct": round(net_pnl_usd / PER_SYMBOL_STARTING_CAPITAL * 100, 2),
        "max_drawdown_pct": round(float(drawdown_pct.min()) * 100, 2),
        "total_trades": int(len(trades_df)),
        "win_rate": round(float(win_rate), 4) if pd.notna(win_rate) else None,
        "profit_factor": round(float(profit_factor), 3) if pd.notna(profit_factor) else None,
    }
    print(f"\n{'='*90}\n{SYMBOL} FINAL $5,000 PnL ACCOUNT (2020-2025, walk-forward, all folds compounded)\n{'='*90}")
    for k, v in summary.items():
        print(f"  {k}: {v}")
    return summary


if __name__ == "__main__":
    # Normalize Stage 7 predictions to the schema expected by prepare_data().
    # Some files only export pred_future_regime_code instead of pred_future_regime.
    pred_path = find_predictions_path(SYMBOL)
    pred_df = pd.read_csv(pred_path)

    needs_backfill = (
        "pred_future_regime" not in pred_df.columns
        or pred_df["pred_future_regime"].isna().all()
    )

    if needs_backfill:
        if "pred_future_regime_code" in pred_df.columns:
            code_to_label = {
                1: "Trending",
                2: "Mean Reversion",
                3: "High Volatility",
                4: "Event",
            }
            pred_df["pred_future_regime"] = (
                pd.to_numeric(pred_df["pred_future_regime_code"], errors="coerce")
                .map(code_to_label)
            )
            pred_df.to_csv(pred_path, index=False)
            print(f"Backfilled missing/empty pred_future_regime in {pred_path.name}")
        else:
            # Some Stage-7 exports use a different schema than the legacy one the rest of the
            # notebook expects (e.g. `predicted_regime`, `prob_trending`, etc.). Normalize the
            # file to the names used below before continuing.
            pred_df = pred_df.copy()

            # --- normalize probability columns ---
            prob_aliases = {
                "prob_trending": "prob_Trending",
                "prob_mean_reversion": "prob_Mean Reversion",
                "prob_high_volatility": "prob_High Volatility",
                "prob_event": "prob_Event",
            }
            for src, dst in prob_aliases.items():
                if src in pred_df.columns and dst not in pred_df.columns:
                    pred_df[dst] = pd.to_numeric(pred_df[src], errors="coerce")

            # --- normalize regime label/code columns ---
            code_to_label = {
                1: "Trending",
                2: "Mean Reversion",
                3: "High Volatility",
                4: "Event",
            }

            if "pred_future_regime" not in pred_df.columns:
                if "pred_future_regime_code" in pred_df.columns:
                    pred_df["pred_future_regime"] = (
                        pd.to_numeric(pred_df["pred_future_regime_code"], errors="coerce")
                        .map(code_to_label)
                    )
                elif "predicted_regime" in pred_df.columns:
                    pred_df["pred_future_regime_code"] = pd.to_numeric(pred_df["predicted_regime"], errors="coerce")
                    pred_df["pred_future_regime"] = pred_df["pred_future_regime_code"].map(code_to_label)
                elif "predicted_regime_label" in pred_df.columns:
                    pred_df["pred_future_regime"] = pred_df["predicted_regime_label"].astype(str).str.strip()
                else:
                    raise KeyError(
                        f"{SYMBOL}: prediction file is missing both 'pred_future_regime' and "
                        "'pred_future_regime_code'. Available columns: {list(pred_df.columns)}"
                    )

            # Backfill code if it exists as a string label but not numeric code
            if "pred_future_regime_code" not in pred_df.columns and "pred_future_regime" in pred_df.columns:
                pred_df["pred_future_regime_code"] = (
                    pred_df["pred_future_regime"].map({v: k for k, v in code_to_label.items()})
                )

            # Safety: if the file already has a confidence column with the expected name, keep it.
            # If it doesn't, try to infer it from the model output if present.
            if "prediction_confidence" not in pred_df.columns and "predicted_regime_confidence" in pred_df.columns:
                pred_df["prediction_confidence"] = pd.to_numeric(pred_df["predicted_regime_confidence"], errors="coerce")

            # Save the normalized schema so future notebook runs don't keep failing.
            pred_df.to_csv(pred_path, index=False)
            print(f"Normalized prediction schema in {pred_path.name}")

    stage_a = optimize_eth()
    if stage_a is not None and stage_a.get("status") == "EDGE_FOUND":
        build_pnl_account(stage_a)
    build_pnl_account(stage_a)

Normalized prediction schema in BTCUSDT_07_predictions.csv

BTCUSDT -- WALK-FORWARD THRESHOLD SELECTION (2020-2025)
  Usable bars: 49,641 (2020-12-31 04:00:00+00:00 -> 2026-09-11 00:00:00+00:00)
  Fold 0: trained meta-model on 357 candidate trades (152 wins, 205 losses).
  Fold 1: trained meta-model on 471 candidate trades (200 wins, 271 losses).
  Fold 2: trained meta-model on 592 candidate trades (249 wins, 343 losses).
  Fold 3: trained meta-model on 573 candidate trades (226 wins, 347 losses).
  Fold 4: trained meta-model on 614 candidate trades (241 wins, 373 losses).
  Fold 5: trained meta-model on 551 candidate trades (207 wins, 344 losses).
  NO ROBUST EDGE FOUND for BTCUSDT.

BTCUSDT: no validated edge -- refusing to build a live-style $5,000 PnL account on an unvalidated signal.
